# sources_test

Evaluate the **current ranking implementation** against the labeled dataset produced by `eval_dataset_builder.ipynb`.

This notebook:
- Loads a labeled dataset (by default the most recent under `eval_dataset/datasets/*/labeled_dataset.csv`; override via `DATASET_PATH`; falls back to `eval_dataset/labeled_dataset.csv`)
- Computes IR metrics per chapter (each chapter = one query)
- Logs results persistently to `eval_dataset/experiments/`

Important: labels are LLM-generated, so treat this as a **directional** benchmark. We’ll still evaluate across **3 different chapters** to reduce chapter-specific overfitting.


In [1]:
from __future__ import annotations

import os
import json
import hashlib
import subprocess
import textwrap
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Dataset selection
# - If `DATASET_PATH` env var is set, we use it.
# - Else we prefer the most-recent dataset-versioned build under `eval_dataset/datasets/*/labeled_dataset.csv`.
# - Else we fall back to the legacy path `eval_dataset/labeled_dataset.csv`.
DATASET_PATH_ENV = os.getenv("DATASET_PATH", "").strip()
DATASET_PATH = Path(DATASET_PATH_ENV) if DATASET_PATH_ENV else None
if DATASET_PATH is not None and not DATASET_PATH.exists():
    print(f"Warning: DATASET_PATH env var is set but missing: {DATASET_PATH}. Ignoring it.")
    DATASET_PATH = None
if DATASET_PATH is None:
    legacy = Path("eval_dataset/labeled_dataset.csv")
    datasets_root = Path("eval_dataset/datasets")
    candidates = []
    if datasets_root.exists():
        candidates = sorted(
            datasets_root.glob("*/labeled_dataset.csv"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
    DATASET_PATH = candidates[0] if candidates else legacy

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing {DATASET_PATH}. Run eval_dataset_builder.ipynb first.")
print("Using DATASET_PATH:", DATASET_PATH)

EXP_DIR = Path("eval_dataset/experiments")
EXP_DIR.mkdir(parents=True, exist_ok=True)

KS = (10, 20, 50)
CONFIDENCE_MIN: Optional[int] = None  # e.g. 70 to filter low-confidence labels

EXPECTED_CHAPTERS = {
    "platform_theory",
    "platform_methodology",
    "platform_empirical_case",
}

# -----------------------------
# Run mode
# -----------------------------
# Choose what this notebook should do when you click "Run all".
# - "stageB_ab": evaluate BOTH datasets (baseline vs coverage_v1) and compare.
# - "single": evaluate DATASET_PATH once (classic mode).
# - "sweeps": run parameter sweeps (Stage C tuning) on the pinned dataset.
# - "rerank": run Stage C.3 LLM rerank tests (calls OpenAI).
# - "rerank_topn": run Stage C.3 shortlist model tests (calls OpenAI, ~top-50 only).
# - "rerank_analysis": offline analysis of rerank outputs (no API calls).
# - "label_adjudication": label-quality audit + adjudication (calls OpenAI; small).
# - "stageB_diag": run blueprint diagnostics (can call OpenAI).
RUN_MODE = "label_adjudication"

# -----------------------------
# Label adjudication settings (used when RUN_MODE=="label_adjudication")
# -----------------------------
# Tip: set MAX_ITEMS=40 for a quick audit; set to 220 to relabel the full chapter.
LABEL_ADJUDICATION_MODEL = "gpt-5-mini"
LABEL_ADJUDICATION_CHAPTER_ID = "platform_empirical_case"
LABEL_ADJUDICATION_MAX_ITEMS = 40
LABEL_ADJUDICATION_CONCURRENCY = 10
LABEL_ADJUDICATION_FORCE = 0
LABEL_ADJUDICATION_APPLY_CONFIDENCE_MIN = 0
LABEL_ADJUDICATION_APPLY = 0  # set to 1 to create a new dataset tag + label-synced scored CSV

os.environ["LABEL_ADJUDICATION_MODEL"] = str(LABEL_ADJUDICATION_MODEL)
os.environ["LABEL_ADJUDICATION_CHAPTER_ID"] = str(LABEL_ADJUDICATION_CHAPTER_ID)
os.environ["LABEL_ADJUDICATION_MAX_ITEMS"] = str(LABEL_ADJUDICATION_MAX_ITEMS)
os.environ["LABEL_ADJUDICATION_CONCURRENCY"] = str(LABEL_ADJUDICATION_CONCURRENCY)
os.environ["LABEL_ADJUDICATION_FORCE"] = str(int(bool(LABEL_ADJUDICATION_FORCE)))
os.environ["LABEL_ADJUDICATION_APPLY_CONFIDENCE_MIN"] = str(int(LABEL_ADJUDICATION_APPLY_CONFIDENCE_MIN))
os.environ["LABEL_ADJUDICATION_APPLY"] = str(int(bool(LABEL_ADJUDICATION_APPLY)))

# Stage B A/B settings (end-to-end)
STAGEB_AB_DATASET_TAGS = ["stageB_baseline_v1", "stageB_coverage_v1_v1"]
STAGEB_AB_EXPERIMENT_TAG = "stageB_ab_eval"
STAGEB_AB_SCORE_COL = "score_hybrid_pool"

if RUN_MODE == "stageB_ab":
    datasets_root = Path("eval_dataset/datasets")
    for tag in STAGEB_AB_DATASET_TAGS:
        p = datasets_root / tag / "labeled_dataset.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing dataset for tag '{tag}': {p}")

# Pinned dataset for Stage C tests (keeps results reproducible)
PINNED_DATASET_TAG = os.getenv("PINNED_DATASET_TAG", "latest").strip() or "latest"
if PINNED_DATASET_TAG.lower() == "latest":
    datasets_root = Path("eval_dataset/datasets")
    cands = sorted(
        datasets_root.glob("*/labeled_dataset.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not cands:
        raise FileNotFoundError("No datasets found under eval_dataset/datasets/*/labeled_dataset.csv")
    PINNED_DATASET_TAG = cands[0].parent.name
if RUN_MODE in ("sweeps", "rerank", "rerank_topn", "rerank_analysis", "label_adjudication"):
    datasets_root = Path("eval_dataset/datasets")
    p = datasets_root / PINNED_DATASET_TAG / "labeled_dataset.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing pinned dataset for tag '{PINNED_DATASET_TAG}': {p}")
    DATASET_PATH = p
    print("Pinned DATASET_PATH:", DATASET_PATH)

    def _get_git_head() -> str:
        try:
            return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        except Exception:
            return "unknown"

    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    experiment_tag = (
        "stageC_sweeps"
        if RUN_MODE == "sweeps"
        else (
            "stageC3_rerank_v1"
            if RUN_MODE == "rerank"
            else ("stageC3_rerank_topn_v2" if RUN_MODE == "rerank_topn" else ("label_adjudication_v2" if RUN_MODE == "label_adjudication" else "stageC3_rerank_analysis_v1"))
        )
    )
    meta = {
        "run_id": run_id,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": _get_git_head(),
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": experiment_tag,
        "experiment_notes": f"pinned_dataset_tag={PINNED_DATASET_TAG}",
    }
    print("Pinned run_id:", run_id)

EXPERIMENT_TAG = (
    STAGEB_AB_EXPERIMENT_TAG
    if RUN_MODE == "stageB_ab"
    else (
        "stageC_sweeps"
        if RUN_MODE == "sweeps"
        else (
            "stageC3_rerank_v1"
            if RUN_MODE == "rerank"
            else (
                "stageC3_rerank_topn_v2"
                if RUN_MODE == "rerank_topn"
                else ("stageC3_rerank_analysis_v1" if RUN_MODE == "rerank_analysis" else "baseline")
            )
        )
    )
)
EXPERIMENT_NOTES = ""  # set automatically in stageB_ab mode

BASELINE_SCORE_COLS = [
    "score_hybrid_pool",
    "score_relevance_hybrid",
    "score_embed_combo",
    "score_tfidf",
    "score_stageC1",
    "score_cite_norm",
]

print("Config OK")
print("RUN_MODE:", RUN_MODE)
if RUN_MODE == "stageB_ab":
    print("Stage B A/B dataset tags:", STAGEB_AB_DATASET_TAGS)
if RUN_MODE in ("sweeps", "rerank", "rerank_topn", "rerank_analysis", "label_adjudication"):
    print("Pinned dataset tag:", PINNED_DATASET_TAG)


Using DATASET_PATH: eval_dataset\datasets\stageB_coverage_v1_v1__platform_methodology_labels_v2_b62a29d1b69b\labeled_dataset.csv
Pinned DATASET_PATH: eval_dataset\datasets\stageB_coverage_v1_v1__platform_methodology_labels_v2_b62a29d1b69b\labeled_dataset.csv
Pinned run_id: 20260202_103812_b62a29d1b69b
Config OK
RUN_MODE: label_adjudication
Pinned dataset tag: stageB_coverage_v1_v1__platform_methodology_labels_v2_b62a29d1b69b


In [2]:
if RUN_MODE not in ("single", "sweeps", "rerank", "rerank_topn", "rerank_analysis", "label_adjudication"):
    print(f"Skipping st_load (RUN_MODE={RUN_MODE})")
else:
    df = pd.read_csv(DATASET_PATH)
    print("Loaded:", DATASET_PATH, "shape=", df.shape)
    
    required = ["chapter_id", "merge_key", "final_label", "final_confidence"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Dataset missing required columns: {missing}")
    
    if CONFIDENCE_MIN is not None:
        df = df[df["final_confidence"].fillna(0) >= CONFIDENCE_MIN].copy()
        print(f"Filtered by final_confidence >= {CONFIDENCE_MIN}: shape=", df.shape)
    
    score_cols = [c for c in BASELINE_SCORE_COLS if c in df.columns]
    if not score_cols:
        raise RuntimeError("No score columns found to evaluate.")
    print("Score columns:", score_cols)
    
    print("\nChapters:")
    display(df.groupby("chapter_id").size().rename("n_docs").to_frame())
    
    present = set(df["chapter_id"].dropna().astype(str).unique().tolist())
    missing = sorted(EXPECTED_CHAPTERS - present)
    if missing:
        raise RuntimeError(f"Dataset is missing expected chapters: {missing}. Present: {sorted(present)}")
    
    print("\nLabel distribution:")
    display(df.groupby(["chapter_id", "final_label"]).size().unstack(fill_value=0))


Loaded: eval_dataset\datasets\stageB_coverage_v1_v1__platform_methodology_labels_v2_b62a29d1b69b\labeled_dataset.csv shape= (660, 35)
Score columns: ['score_hybrid_pool', 'score_relevance_hybrid', 'score_embed_combo', 'score_tfidf', 'score_stageC1', 'score_cite_norm']

Chapters:


,n_docs
chapter_id,
platform_empirical_case,220
platform_methodology,220
platform_theory,220



Label distribution:


final_label,exclude,include,maybe
chapter_id,,,
platform_empirical_case,163,2,55
platform_methodology,66,23,131
platform_theory,100,19,101


In [3]:
LABEL_TO_BIN_INCLUDE = {"include": 1, "maybe": 0, "exclude": 0}
LABEL_TO_BIN_INCLUDE_OR_MAYBE = {"include": 1, "maybe": 1, "exclude": 0}
LABEL_TO_GRADE = {"include": 2, "maybe": 1, "exclude": 0}

def _to_numeric_score(s: pd.Series) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").fillna(-1e9).to_numpy(dtype=float)

def dcg(rels: np.ndarray) -> float:
    rels = np.asarray(rels, dtype=float)
    if rels.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rels.size + 2))
    gains = (2.0 ** rels - 1.0)
    return float(np.sum(gains * discounts))

def ndcg_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = np.asarray(rels[:k], dtype=float)
    ideal = np.sort(rels)[::-1][:k]
    denom = dcg(ideal)
    if denom <= 0:
        return float("nan")
    return dcg(rels_k) / denom

def precision_at_k(rel_bin: np.ndarray, k: int) -> float:
    rel_bin = np.asarray(rel_bin[:k], dtype=float)
    return float(rel_bin.sum() / k) if k > 0 else float("nan")

def recall_at_k(rel_bin: np.ndarray, k: int) -> float:
    rel_bin = np.asarray(rel_bin, dtype=float)
    total = float(rel_bin.sum())
    if total <= 0:
        return float("nan")
    return float(rel_bin[:k].sum() / total)

def average_precision(rel_bin: np.ndarray) -> float:
    rel_bin = np.asarray(rel_bin, dtype=float)
    total = float(rel_bin.sum())
    if total <= 0:
        return float("nan")
    cumsum = np.cumsum(rel_bin)
    precision_at_i = cumsum / (np.arange(len(rel_bin)) + 1)
    return float((precision_at_i * rel_bin).sum() / total)

def mrr(rel_bin: np.ndarray) -> float:
    rel_bin = np.asarray(rel_bin, dtype=float)
    hits = np.where(rel_bin > 0)[0]
    if hits.size == 0:
        return 0.0
    return float(1.0 / (hits[0] + 1))

@dataclass(frozen=True)
class EvalResult:
    chapter_id: str
    score_col: str
    n_docs: int
    n_include: int
    n_maybe: int
    auc_include: float
    mrr_include: float
    p_at: Dict[int, float]
    r_at: Dict[int, float]
    ndcg_at: Dict[int, float]

def evaluate_chapter(g: pd.DataFrame, score_col: str, ks: Iterable[int]) -> EvalResult:
    scores = _to_numeric_score(g[score_col])
    order = np.argsort(-scores, kind="mergesort")

    labels = g["final_label"].fillna("exclude").astype(str).to_numpy()[order]
    rel_bin = np.array([LABEL_TO_BIN_INCLUDE.get(x, 0) for x in labels], dtype=float)
    rel_grade = np.array([LABEL_TO_GRADE.get(x, 0) for x in labels], dtype=float)

    # AUC for include vs non-include (only if both classes exist)
    try:
        auc = float(roc_auc_score(rel_bin, scores[order])) if len(set(rel_bin.tolist())) > 1 else float("nan")
    except Exception:
        auc = float("nan")

    p_at = {k: precision_at_k(rel_bin, k) for k in ks}
    r_at = {k: recall_at_k(rel_bin, k) for k in ks}
    ndcg = {k: ndcg_at_k(rel_grade, k) for k in ks}

    return EvalResult(
        chapter_id=str(g["chapter_id"].iloc[0]),
        score_col=score_col,
        n_docs=int(len(g)),
        n_include=int((g["final_label"] == "include").sum()),
        n_maybe=int((g["final_label"] == "maybe").sum()),
        auc_include=auc,
        mrr_include=mrr(rel_bin),
        p_at=p_at,
        r_at=r_at,
        ndcg_at=ndcg,
    )

def evaluate_all(df_in: pd.DataFrame, score_cols: List[str], ks: Iterable[int]) -> pd.DataFrame:
    chapters = sorted(df_in["chapter_id"].dropna().astype(str).unique().tolist())
    rows = []
    for score_col in score_cols:
        per_ch = []
        for cid in chapters:
            g = df_in[df_in["chapter_id"].astype(str) == cid].copy()
            r = evaluate_chapter(g, score_col=score_col, ks=ks)
            per_ch.append(r)

            row = {
                "chapter_id": r.chapter_id,
                "score_col": r.score_col,
                "n_docs": r.n_docs,
                "n_include": r.n_include,
                "n_maybe": r.n_maybe,
                "auc_include": r.auc_include,
                "mrr_include": r.mrr_include,
            }
            for k in ks:
                row[f"p@{k}"] = r.p_at[k]
                row[f"r@{k}"] = r.r_at[k]
                row[f"ndcg@{k}"] = r.ndcg_at[k]
            rows.append(row)

        # Macro average across chapters (each chapter = one query)
        macro = {
            "chapter_id": "__ALL__",
            "score_col": score_col,
            "n_docs": int(sum(x.n_docs for x in per_ch)),
            "n_include": int(sum(x.n_include for x in per_ch)),
            "n_maybe": int(sum(x.n_maybe for x in per_ch)),
            "auc_include": float(np.nanmean([x.auc_include for x in per_ch])),
            "mrr_include": float(np.nanmean([x.mrr_include for x in per_ch])),
        }
        for k in ks:
            macro[f"p@{k}"] = float(np.nanmean([x.p_at[k] for x in per_ch]))
            macro[f"r@{k}"] = float(np.nanmean([x.r_at[k] for x in per_ch]))
            macro[f"ndcg@{k}"] = float(np.nanmean([x.ndcg_at[k] for x in per_ch]))
        rows.append(macro)

    out = pd.DataFrame(rows)
    return out

print("Metrics utilities ready")


Metrics utilities ready


In [4]:
# Stage B end-to-end A/B runner (evaluates BOTH datasets and persists results)
#
# Safe to "Run all":
# - In RUN_MODE="stageB_ab" this is the main entrypoint.
# - It appends macro rows to `eval_dataset/experiments/runs.csv` for later comparison.

if RUN_MODE != "stageB_ab":
    print(f"Skipping st_stageB_ab_run (RUN_MODE={RUN_MODE})")
else:
    datasets_root = Path("eval_dataset/datasets")

    dataset_pairs: List[Tuple[str, Path]] = []
    for tag in STAGEB_AB_DATASET_TAGS:
        p = datasets_root / tag / "labeled_dataset.csv"
        if not p.exists():
            raise FileNotFoundError(f"Missing dataset for tag '{tag}': {p}")
        dataset_pairs.append((tag, p))

    def _get_git_head() -> str:
        try:
            return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        except Exception:
            return "unknown"

    def _persist(results_df: pd.DataFrame, dataset_path: Path, experiment_tag: str, experiment_notes: str) -> None:
        dataset_sha = hashlib.sha1(dataset_path.read_bytes()).hexdigest()[:12]
        run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"

        meta = {
            "run_id": run_id,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "dataset_path": str(dataset_path),
            "dataset_sha1_12": dataset_sha,
            "git_head": _get_git_head(),
            "confidence_min": CONFIDENCE_MIN,
            "experiment_tag": experiment_tag,
            "experiment_notes": experiment_notes,
        }

        safe_tag = "".join([c if (c.isalnum() or c in "-_" ) else "_" for c in str(experiment_tag or "").strip()])
        safe_tag = safe_tag or "run"
        report_path = EXP_DIR / f"{run_id}_{safe_tag}_report.csv"
        results_df.assign(**meta).to_csv(report_path, index=False)
        print("Saved report:", report_path)

        runs_csv = EXP_DIR / "runs.csv"
        macro_rows = results_df[results_df["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
        macro_rows = macro_rows.assign(**meta, experiment=safe_tag)
        if runs_csv.exists():
            macro_rows.to_csv(runs_csv, mode="a", header=False, index=False)
        else:
            macro_rows.to_csv(runs_csv, index=False)
        print("Updated:", runs_csv)

    for tag, dataset_path in dataset_pairs:
        print(f"\n=== Evaluating dataset: {tag} ===")
        df_ab = pd.read_csv(dataset_path)

        required = ["chapter_id", "merge_key", "final_label", "final_confidence"]
        missing = [c for c in required if c not in df_ab.columns]
        if missing:
            raise RuntimeError(f"Dataset missing required columns {missing}: {dataset_path}")

        if CONFIDENCE_MIN is not None:
            df_ab = df_ab[df_ab["final_confidence"].fillna(0) >= CONFIDENCE_MIN].copy()
            print(f"Filtered by final_confidence >= {CONFIDENCE_MIN}: shape=", df_ab.shape)

        present = set(df_ab["chapter_id"].dropna().astype(str).unique().tolist())
        missing_ch = sorted(EXPECTED_CHAPTERS - present)
        if missing_ch:
            raise RuntimeError(f"Dataset is missing expected chapters: {missing_ch}. Present: {sorted(present)}")

        if STAGEB_AB_SCORE_COL not in df_ab.columns:
            raise RuntimeError(f"Missing required score column '{STAGEB_AB_SCORE_COL}' in {dataset_path}")

        res_ab = evaluate_all(df_ab, score_cols=[STAGEB_AB_SCORE_COL], ks=KS)
        print("Macro (__ALL__) summary:")
        display(res_ab[res_ab["chapter_id"] == "__ALL__"].reset_index(drop=True))

        _persist(
            res_ab,
            dataset_path=dataset_path,
            experiment_tag=STAGEB_AB_EXPERIMENT_TAG,
            experiment_notes=f"dataset_tag={tag}; score_col={STAGEB_AB_SCORE_COL}",
        )

    print("\nStage B A/B evaluation complete.")


Skipping st_stageB_ab_run (RUN_MODE=label_adjudication)


In [5]:
if RUN_MODE != "single":
    print(f"Skipping st_baseline_eval (RUN_MODE={RUN_MODE})")
else:
    results = evaluate_all(df, score_cols=score_cols, ks=KS)
    
    print("Macro (__ALL__) summary:")
    display(
        results[results["chapter_id"] == "__ALL__"]
        .sort_values(by=f"ndcg@{KS[1]}", ascending=False)
        .reset_index(drop=True)
    )
    
    print("Per-chapter detail:")
    display(
        results[results["chapter_id"] != "__ALL__"]
        .sort_values(["score_col", "chapter_id"], ascending=True)
        .reset_index(drop=True)
    )


Skipping st_baseline_eval (RUN_MODE=label_adjudication)


In [6]:
if RUN_MODE != "single":
    print(f"Skipping st_error_analysis (RUN_MODE={RUN_MODE})")
else:
    # Quick qualitative check: show high-ranked EXCLUDEs and low-ranked INCLUDEs
    
    def inspect_errors(df_in: pd.DataFrame, chapter_id: str, score_col: str, top_n: int = 8):
        g = df_in[df_in["chapter_id"] == chapter_id].copy()
        g["_score"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
        g = g.sort_values("_score", ascending=False).reset_index(drop=True)
    
        fp = g[g["final_label"] == "exclude"].head(top_n)
        fn = g[g["final_label"] == "include"].tail(top_n)
    
        cols = ["final_label", "final_confidence", score_col, "title", "venue", "year", "p1_short_reason", "p2_short_reason"]
        cols = [c for c in cols if c in g.columns]
    
        print(f"\n=== {chapter_id} | {score_col} ===")
        print("Top false positives (exclude but ranked high):")
        display(fp[cols])
        print("Bottom false negatives (include but ranked low):")
        display(fn[cols])
    
    best_score = (
        results[results["chapter_id"] == "__ALL__"]
        .sort_values(by=f"ndcg@{KS[1]}", ascending=False)
        .iloc[0]["score_col"]
    )
    
    for cid in sorted(df["chapter_id"].unique().tolist()):
        inspect_errors(df, chapter_id=cid, score_col=best_score, top_n=6)


Skipping st_error_analysis (RUN_MODE=label_adjudication)


In [7]:
if RUN_MODE != "single":
    print(f"Skipping st_persist (RUN_MODE={RUN_MODE})")
else:
    def get_git_head() -> str:
        try:
            return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
        except Exception:
            return "unknown"
    
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    git_head = get_git_head()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    
    meta = {
        "run_id": run_id,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": git_head,
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": EXPERIMENT_TAG,
        "experiment_notes": EXPERIMENT_NOTES,
    }
    
    safe_tag = "".join([c if (c.isalnum() or c in "-_" ) else "_" for c in str(EXPERIMENT_TAG or "").strip()])
    safe_tag = safe_tag or "run"
    report_path = EXP_DIR / f"{run_id}_{safe_tag}_report.csv"
    results.assign(**meta).to_csv(report_path, index=False)
    print("Saved report:", report_path)
    
    # Append macro rows to a persistent runs.csv for quick tracking
    runs_csv = EXP_DIR / "runs.csv"
    macro_rows = results[results["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
    macro_rows = macro_rows.assign(**meta, experiment=safe_tag)
    
    if runs_csv.exists():
        macro_rows.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        macro_rows.to_csv(runs_csv, index=False)
    
    print("Updated:", runs_csv)
    display(pd.read_csv(runs_csv).tail(25))


Skipping st_persist (RUN_MODE=label_adjudication)


In [8]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_cite_weight_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: citation-weight sweep (offline; no API calls)
# Goal: decide whether CITE_WEIGHT should be reduced/removed.

CITE_WEIGHTS = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12, 0.20]
BASE_COL = "score_relevance_hybrid"  # embed+tfidf hybrid (no citations)
CITE_COL = "score_cite_norm"
OUT_COL = "score_cite_sweep"

assert BASE_COL in df.columns, f"Missing {BASE_COL}"
assert CITE_COL in df.columns, f"Missing {CITE_COL}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

df_base = df.copy()
df_base["_base_n"] = minmax_by_group(df_base, BASE_COL)
df_base["_cite_n"] = minmax_by_group(df_base, CITE_COL)

rows = []
for w in CITE_WEIGHTS:
    d = df_base.copy()
    d[OUT_COL] = (1.0 - float(w)) * d["_base_n"] + float(w) * d["_cite_n"]
    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "cite_weight": float(w),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_w = float(sweep_df.iloc[0]["cite_weight"]) if not sweep_df.empty else 0.0
print(f"Best cite_weight by ndcg@20: {best_w:.3f}")

# Leave-one-chapter-out (LOCO) sanity check (with only 3 chapters)
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w in CITE_WEIGHTS:
        t = train.copy()
        t[OUT_COL] = (1.0 - float(w)) * t["_base_n"] + float(w) * t["_cite_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w)

    u = test.copy()
    u[OUT_COL] = (1.0 - float(best_train_w)) * u["_base_n"] + float(best_train_w) * u["_cite_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_cite_weight": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts (uses run_id/meta if available; otherwise creates local ids)
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "cite_weight_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_cite_weight_sweep.csv"
sweep_df.assign(**meta_use, experiment="cite_weight_sweep", base_col=BASE_COL, cite_col=CITE_COL).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_cite_weight_loco.csv"
loco_df.assign(**meta_use, experiment="cite_weight_loco", base_col=BASE_COL, cite_col=CITE_COL).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv (keeps history easy to scan)
d_best = df_base.copy()
d_best[OUT_COL] = (1.0 - best_w) * d_best["_base_n"] + best_w * d_best["_cite_n"]
best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "cite_weight_sweep"
meta_best["experiment_notes"] = f"base={BASE_COL}; per-chapter minmax; best_w={best_w:.3f}"
best_macro = best_macro.assign(**meta_best, experiment="cite_weight_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


Skipping st_cite_weight_sweep (RUN_MODE=label_adjudication)


In [9]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_weight_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Example: offline sweep of a simple hybrid weight (no API calls)
# This is a safe, scientific starting point: change ONE thing, measure, log.

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

if ("score_embed_combo" in df.columns) and ("score_tfidf" in df.columns):
    df_sweep = df.copy()
    df_sweep["_emb_n"] = minmax_by_group(df_sweep, "score_embed_combo")
    df_sweep["_tf_n"] = minmax_by_group(df_sweep, "score_tfidf")
    cite = df_sweep["score_cite_norm"] if "score_cite_norm" in df_sweep.columns else 0.0

    rows = []
    for w in np.linspace(0, 1, 11):
        base = w * df_sweep["_emb_n"] + (1 - w) * df_sweep["_tf_n"]
        df_sweep["score_sweep"] = (1 - 0.08) * base + 0.08 * cite

        res_w = evaluate_all(df_sweep, score_cols=["score_sweep"], ks=KS)
        macro = res_w[res_w["chapter_id"] == "__ALL__"].iloc[0].to_dict()
        rows.append({
            "w_embed": float(w),
            "ndcg@20": float(macro.get("ndcg@20")),
            "p@20": float(macro.get("p@20")),
            "mrr_include": float(macro.get("mrr_include")),
        })

    sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
    display(sweep_df)

    best = sweep_df.iloc[0].to_dict()
    print("Best by ndcg@20:", best)

    # Save artifacts (ensure run_id/meta exist in sweeps mode)
    run_id_use = globals().get("run_id")
    meta_use = globals().get("meta")
    if run_id_use is None or meta_use is None:
        dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
        run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
        meta_use = {
            "run_id": run_id_use,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "dataset_path": str(DATASET_PATH),
            "dataset_sha1_12": dataset_sha,
            "git_head": "unknown",
            "confidence_min": CONFIDENCE_MIN,
            "experiment_tag": "weight_sweep",
            "experiment_notes": "auto-meta (run_id/meta missing)",
        }

    sweep_path = EXP_DIR / f"{run_id_use}_weight_sweep.csv"
    sweep_df.assign(**meta_use, experiment="weight_sweep").to_csv(sweep_path, index=False)
    print("Saved sweep:", sweep_path)
else:
    print("Skipping sweep: missing score_embed_combo or score_tfidf columns")
    '''
    exec(_CODE, globals())


Skipping st_weight_sweep (RUN_MODE=label_adjudication)


In [10]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_embed_combo_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: embedding combo sweep (max vs breadth)
# Goal: tune how we combine score_embed_max vs score_embed_mean_top3.
# This stays chapter-agnostic and uses only already-computed columns (no API calls).

W_EMBED = 0.60  # from weight sweep best
W_TFIDF = 1.0 - W_EMBED

EMBED_MAX_WEIGHTS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
OUT_COL = "score_embed_combo_sweep"

for c in ["score_embed_max", "score_embed_mean_top3", "score_tfidf"]:
    assert c in df.columns, f"Missing {c}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

df_base = df.copy()
df_base["_tf_n"] = minmax_by_group(df_base, "score_tfidf")

rows = []
for w_max in EMBED_MAX_WEIGHTS:
    d = df_base.copy()
    emb_max = pd.to_numeric(d["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(d["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    d["_emb_raw"] = float(w_max) * emb_max + (1.0 - float(w_max)) * emb_b
    d["_emb_n"] = minmax_by_group(d, "_emb_raw")

    d[OUT_COL] = W_EMBED * d["_emb_n"] + W_TFIDF * d["_tf_n"]

    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "w_embed_max": float(w_max),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_wmax = float(sweep_df.iloc[0]["w_embed_max"]) if not sweep_df.empty else 1.0
print(f"Best w_embed_max by ndcg@20: {best_wmax:.3f}")

# Leave-one-chapter-out (LOCO) sanity check
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w_max in EMBED_MAX_WEIGHTS:
        t = train.copy()
        emb_max = pd.to_numeric(t["score_embed_max"], errors="coerce").fillna(0.0)
        emb_b = pd.to_numeric(t["score_embed_mean_top3"], errors="coerce").fillna(0.0)
        t["_emb_raw"] = float(w_max) * emb_max + (1.0 - float(w_max)) * emb_b
        t["_emb_n"] = minmax_by_group(t, "_emb_raw")
        t[OUT_COL] = W_EMBED * t["_emb_n"] + W_TFIDF * t["_tf_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w_max)

    u = test.copy()
    emb_max = pd.to_numeric(u["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(u["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    u["_emb_raw"] = float(best_train_w) * emb_max + (1.0 - float(best_train_w)) * emb_b
    u["_emb_n"] = minmax_by_group(u, "_emb_raw")
    u[OUT_COL] = W_EMBED * u["_emb_n"] + W_TFIDF * u["_tf_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_w_embed_max": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "embed_combo_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_embed_combo_sweep.csv"
sweep_df.assign(**meta_use, experiment="embed_combo_sweep", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_embed_combo_loco.csv"
loco_df.assign(**meta_use, experiment="embed_combo_loco", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
d_best = df_base.copy()
emb_max = pd.to_numeric(d_best["score_embed_max"], errors="coerce").fillna(0.0)
emb_b = pd.to_numeric(d_best["score_embed_mean_top3"], errors="coerce").fillna(0.0)
d_best["_emb_raw"] = float(best_wmax) * emb_max + (1.0 - float(best_wmax)) * emb_b
d_best["_emb_n"] = minmax_by_group(d_best, "_emb_raw")
d_best[OUT_COL] = W_EMBED * d_best["_emb_n"] + W_TFIDF * d_best["_tf_n"]

best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "embed_combo_sweep"
meta_best["experiment_notes"] = f"W_EMBED={W_EMBED:.2f}; w_embed_max={best_wmax:.3f}; per-chapter minmax(tfidf, emb_raw)"
best_macro = best_macro.assign(**meta_best, experiment="embed_combo_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


Skipping st_embed_combo_sweep (RUN_MODE=label_adjudication)


In [11]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_w_embed_breadth_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: re-sweep W_EMBED after we learned "breadth" > "max"
# We fix w_embed_max=0.0 (only score_embed_mean_top3) and sweep W_EMBED vs TF-IDF.
# No API calls.

EMBED_MAX_WEIGHT_FIXED = 0.0
W_EMBED_GRID = [round(float(x), 2) for x in np.linspace(0, 1, 11)]
OUT_COL = "score_w_embed_breadth_sweep"

for c in ["score_embed_mean_top3", "score_tfidf"]:
    assert c in df.columns, f"Missing {c}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

df_base = df.copy()
df_base["_tf_n"] = minmax_by_group(df_base, "score_tfidf")
df_base["_emb_raw"] = pd.to_numeric(df_base["score_embed_mean_top3"], errors="coerce").fillna(0.0)
df_base["_emb_n"] = minmax_by_group(df_base, "_emb_raw")

rows = []
for w_embed in W_EMBED_GRID:
    d = df_base.copy()
    d[OUT_COL] = float(w_embed) * d["_emb_n"] + (1.0 - float(w_embed)) * d["_tf_n"]

    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "w_embed": float(w_embed),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_w = float(sweep_df.iloc[0]["w_embed"]) if not sweep_df.empty else 0.6
print(f"Best w_embed (breadth) by ndcg@20: {best_w:.3f}")

# LOCO sanity check
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w_embed in W_EMBED_GRID:
        t = train.copy()
        t[OUT_COL] = float(w_embed) * t["_emb_n"] + (1.0 - float(w_embed)) * t["_tf_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w_embed)

    u = test.copy()
    u[OUT_COL] = float(best_train_w) * u["_emb_n"] + (1.0 - float(best_train_w)) * u["_tf_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_w_embed": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "w_embed_breadth_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_w_embed_breadth_sweep.csv"
sweep_df.assign(**meta_use, experiment="w_embed_breadth_sweep", w_embed_max_fixed=EMBED_MAX_WEIGHT_FIXED).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_w_embed_breadth_loco.csv"
loco_df.assign(**meta_use, experiment="w_embed_breadth_loco", w_embed_max_fixed=EMBED_MAX_WEIGHT_FIXED).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
d_best = df_base.copy()
d_best[OUT_COL] = float(best_w) * d_best["_emb_n"] + (1.0 - float(best_w)) * d_best["_tf_n"]
best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "w_embed_breadth_sweep"
meta_best["experiment_notes"] = f"w_embed_max_fixed={EMBED_MAX_WEIGHT_FIXED:.1f}; best_w_embed={best_w:.3f}; per-chapter minmax(tfidf, emb_mean_top3)"
best_macro = best_macro.assign(**meta_best, experiment="w_embed_breadth_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


Skipping st_w_embed_breadth_sweep (RUN_MODE=label_adjudication)


In [12]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_cite_weight_breadth_sweep (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Next test: citation weight on top of the improved "breadth" base score
# We fix the base to: W_EMBED * emb_mean_top3 + W_TFIDF * tfidf (per-chapter minmax),
# then sweep cite_weight.

W_EMBED = 0.60
W_TFIDF = 1.0 - W_EMBED

CITE_WEIGHTS = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12, 0.20, 0.30]
OUT_COL = "score_breadth_cite_sweep"

for c in ["score_embed_mean_top3", "score_tfidf", "score_cite_norm"]:
    assert c in df.columns, f"Missing {c}"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for cid, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

# Build the fixed base score (breadth embedding + tfidf)
df_base = df.copy()
df_base["_tf_n"] = minmax_by_group(df_base, "score_tfidf")
df_base["_emb_raw"] = pd.to_numeric(df_base["score_embed_mean_top3"], errors="coerce").fillna(0.0)
df_base["_emb_n"] = minmax_by_group(df_base, "_emb_raw")
df_base["_base"] = float(W_EMBED) * df_base["_emb_n"] + float(W_TFIDF) * df_base["_tf_n"]
df_base["_cite_n"] = minmax_by_group(df_base, "score_cite_norm")

rows = []
for w in CITE_WEIGHTS:
    d = df_base.copy()
    d[OUT_COL] = (1.0 - float(w)) * d["_base"] + float(w) * d["_cite_n"]
    res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "cite_weight": float(w),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df)

best_w = float(sweep_df.iloc[0]["cite_weight"]) if not sweep_df.empty else 0.0
print(f"Best cite_weight (breadth base) by ndcg@20: {best_w:.3f}")

# LOCO sanity check
chapters = sorted(df_base["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df_base[df_base["chapter_id"] != holdout].copy()
    test = df_base[df_base["chapter_id"] == holdout].copy()

    best_train_w = None
    best_train_ndcg20 = -1e9
    for w in CITE_WEIGHTS:
        t = train.copy()
        t[OUT_COL] = (1.0 - float(w)) * t["_base"] + float(w) * t["_cite_n"]
        r = evaluate_all(t, score_cols=[OUT_COL], ks=KS)
        train_ndcg20 = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
        if train_ndcg20 > best_train_ndcg20:
            best_train_ndcg20 = train_ndcg20
            best_train_w = float(w)

    u = test.copy()
    u[OUT_COL] = (1.0 - float(best_train_w)) * u["_base"] + float(best_train_w) * u["_cite_n"]
    r2 = evaluate_all(u, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_cite_weight": float(best_train_w),
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Save artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "cite_weight_breadth_sweep",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_cite_weight_breadth_sweep.csv"
sweep_df.assign(**meta_use, experiment="cite_weight_breadth_sweep", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_cite_weight_breadth_loco.csv"
loco_df.assign(**meta_use, experiment="cite_weight_breadth_loco", w_embed=W_EMBED, w_tfidf=W_TFIDF).to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
d_best = df_base.copy()
d_best[OUT_COL] = (1.0 - float(best_w)) * d_best["_base"] + float(best_w) * d_best["_cite_n"]
best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
best_macro["score_col"] = OUT_COL

meta_best = dict(meta_use)
meta_best["experiment_tag"] = "cite_weight_breadth_sweep"
meta_best["experiment_notes"] = f"W_EMBED={W_EMBED:.2f}; breadth emb (mean_top3); per-chapter minmax; best_cite_weight={best_w:.3f}"
best_macro = best_macro.assign(**meta_best, experiment="cite_weight_breadth_sweep_best")

runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in best_macro.columns:
            best_macro[c] = None
    best_macro = best_macro[header]
    best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    best_macro.to_csv(runs_csv, index=False)

print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


Skipping st_cite_weight_breadth_sweep (RUN_MODE=label_adjudication)


In [13]:
if RUN_MODE != "sweeps":
    print(f"Skipping st_stageC_grid_search_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C: joint grid search (offline; no API calls)
#
# Why:
# Sequential sweeps can miss interactions (w_embed_max ↔ w_embed ↔ cite_weight).
# This grid search selects a single, globally best scoring recipe on the *winning Stage B dataset*.
#
# Score recipe:
# 1) emb_raw = w_embed_max * score_embed_max + (1-w_embed_max) * score_embed_mean_top3
# 2) per-chapter minmax normalize: emb_n, tfidf_n, cite_n
# 3) base = w_embed * emb_n + (1-w_embed) * tfidf_n
# 4) score = (1-cite_weight) * base + cite_weight * cite_n

REQUIRED = ["score_embed_max", "score_embed_mean_top3", "score_tfidf", "score_cite_norm"]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required columns for Stage C grid search: {missing}")

# Grids (keep small; can expand later)
W_EMBED_MAX_GRID = [0.0, 0.2, 0.5, 0.8, 1.0]
W_EMBED_GRID = [round(float(x), 2) for x in np.linspace(0, 1, 11)]
CITE_WEIGHT_GRID = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12]

OUT_COL = "score_stageC_grid_v1"

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

def score_df(df_in: pd.DataFrame, w_embed_max: float, w_embed: float, cite_weight: float) -> pd.DataFrame:
    d = df_in.copy()
    emb_max = pd.to_numeric(d["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(d["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    d["_emb_raw"] = float(w_embed_max) * emb_max + (1.0 - float(w_embed_max)) * emb_b
    d["_emb_n"] = minmax_by_group(d, "_emb_raw")
    d["_tf_n"] = minmax_by_group(d, "score_tfidf")
    d["_cite_n"] = minmax_by_group(d, "score_cite_norm")

    base = float(w_embed) * d["_emb_n"] + (1.0 - float(w_embed)) * d["_tf_n"]
    d[OUT_COL] = (1.0 - float(cite_weight)) * base + float(cite_weight) * d["_cite_n"]
    return d

rows = []
for wmax in W_EMBED_MAX_GRID:
    for w in W_EMBED_GRID:
        for cw in CITE_WEIGHT_GRID:
            d = score_df(df, w_embed_max=wmax, w_embed=w, cite_weight=cw)
            res = evaluate_all(d, score_cols=[OUT_COL], ks=KS)
            macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
            rows.append({
                "w_embed_max": float(wmax),
                "w_embed": float(w),
                "cite_weight": float(cw),
                "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
                "p@20": float(macro.get("p@20", float("nan"))),
                "mrr_include": float(macro.get("mrr_include", float("nan"))),
                "auc_include": float(macro.get("auc_include", float("nan"))),
            })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(sweep_df.head(25))

best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best Stage C grid (by ndcg@20):", best)

# LOCO sanity check (with only 3 chapters)
chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df[df["chapter_id"] != holdout].copy()
    test = df[df["chapter_id"] == holdout].copy()

    best_train = None
    best_train_ndcg20 = -1e9
    for wmax in W_EMBED_MAX_GRID:
        for w in W_EMBED_GRID:
            for cw in CITE_WEIGHT_GRID:
                dtr = score_df(train, w_embed_max=wmax, w_embed=w, cite_weight=cw)
                r = evaluate_all(dtr, score_cols=[OUT_COL], ks=KS)
                nd = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
                if nd > best_train_ndcg20:
                    best_train_ndcg20 = nd
                    best_train = {"w_embed_max": float(wmax), "w_embed": float(w), "cite_weight": float(cw)}

    dte = score_df(test, **best_train)
    r2 = evaluate_all(dte, score_cols=[OUT_COL], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()
    loco_rows.append({
        "holdout_chapter": holdout,
        **best_train,
        "train_ndcg@20": float(best_train_ndcg20),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_p@20": float(hold.get("p@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
    })

loco_df = pd.DataFrame(loco_rows)
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))

# Persist artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC_grid_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_stageC_grid_v1_sweep.csv"
sweep_df.assign(**meta_use, experiment="stageC_grid_v1_sweep").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC_grid_v1_loco.csv"
loco_df.assign(**meta_use, experiment="stageC_grid_v1_loco").to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
if best:
    d_best = score_df(df, w_embed_max=float(best["w_embed_max"]), w_embed=float(best["w_embed"]), cite_weight=float(best["cite_weight"]))
    best_res = evaluate_all(d_best, score_cols=[OUT_COL], ks=KS)
    best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy()
    best_macro["score_col"] = OUT_COL
    meta_best = dict(meta_use)
    meta_best["experiment_tag"] = "stageC_grid_v1"
    meta_best["experiment_notes"] = f"best_by=ndcg@20; w_embed_max={best['w_embed_max']}; w_embed={best['w_embed']}; cite_weight={best['cite_weight']}"
    best_macro = best_macro.assign(**meta_best, experiment="stageC_grid_v1_best")

    runs_csv = EXP_DIR / "runs.csv"
    if runs_csv.exists():
        header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
        for c in header:
            if c not in best_macro.columns:
                best_macro[c] = None
        best_macro = best_macro[header]
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)

    print("Appended best macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageC_grid_search_v1 (RUN_MODE=label_adjudication)


In [14]:
if RUN_MODE != "rerank":
    print(f"Skipping st_stageC3_llm_rerank_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3: LLM rerank test (API calls; cached to disk)
#
# Scientific intent:
# - Keep Stage B fixed (coverage_v1 dataset).
# - Keep labels fixed (eval_rubrics).
# - Measure whether an LLM-based rerank score improves ranking quality.

import os
import json
import time
import random
import asyncio
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

from openai import OpenAI
from agents import Agent, Runner, ModelSettings


# -----------------------------
# Settings
# -----------------------------
RERANK_MODEL = os.getenv("RERANK_MODEL", "gpt-5-nano").strip() or "gpt-5-nano"
PROMPT_VERSION = "v1"
FORCE_RERANK = False  # set True to ignore cache

CONCURRENCY = 8
MAX_RETRIES = 8
BACKOFF_INITIAL = 1.0
BACKOFF_MAX = 30.0
ABSTRACT_MAX_CHARS = 2000

# Pricing for cost estimation (USD per 1M tokens)
MODEL_PRICES_USD_PER_1M = {
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
}


def _price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})


def cost_from_usage(usage, model: str) -> dict:
    prices = _price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)

        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }


def _truncate(s: str, max_chars: int) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + "…"


RUBRIC_DIR = Path("eval_dataset/eval_rubrics")
if not RUBRIC_DIR.exists():
    raise FileNotFoundError(f"Missing rubric dir: {RUBRIC_DIR}")

rubrics: Dict[str, dict] = {}
rubric_sigs: Dict[str, str] = {}
for cid in sorted(EXPECTED_CHAPTERS):
    rp = RUBRIC_DIR / f"{cid}.json"
    if not rp.exists():
        raise FileNotFoundError(f"Missing rubric for chapter '{cid}': {rp}")
    r = json.loads(rp.read_text(encoding="utf-8"))
    rubrics[cid] = r
    payload = {
        "scope_statement": r.get("scope_statement"),
        "must_cover": r.get("must_cover", []),
        "must_avoid": r.get("must_avoid", []),
        "scoring_guidance": r.get("scoring_guidance"),
    }
    rubric_sigs[cid] = hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:8]


class RerankOut(BaseModel):
    score: int = Field(..., ge=0, le=100)
    notes: str = Field("", max_length=140)


rerank_agent = Agent(
    name=f"StageC3 Rerank ({PROMPT_VERSION})",
    model=RERANK_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=(
        "You score academic papers for inclusion in a specific thesis chapter. "
        "Use the rubric strictly. Return ONLY the structured output."
    ),
    output_type=RerankOut,
)


def build_prompt(rubric: dict, title: str, abstract: str) -> str:
    scope = rubric.get("scope_statement", "")
    must_cover = rubric.get("must_cover", [])
    must_avoid = rubric.get("must_avoid", [])
    guidance = rubric.get("scoring_guidance", "")

    lines = []
    lines.append("Score this paper for inclusion in the chapter.")
    lines.append("Return only the schema fields.")
    lines.append("")
    lines.append("RUBRIC")
    lines.append(f"SCOPE: {scope}")
    if guidance:
        lines.append(f"GUIDANCE: {guidance}")
    if must_cover:
        lines.append("MUST_COVER:")
        for b in must_cover:
            lines.append(f"- {b}")
    if must_avoid:
        lines.append("MUST_AVOID:")
        for b in must_avoid:
            lines.append(f"- {b}")

    lines.append("")
    lines.append("PAPER")
    lines.append(f"TITLE: {title}")
    if abstract:
        lines.append(f"ABSTRACT: {abstract}")
    else:
        lines.append("ABSTRACT: (missing)")

    lines.append("")
    lines.append("Scoring:")
    lines.append("- 100 = must cite for this chapter")
    lines.append("- 50 = partially relevant / might cite")
    lines.append("- 0 = irrelevant or off-scope")
    return "\n".join(lines)


CACHE_DIR = EXP_DIR / "llm_rerank_cache_v1" / PINNED_DATASET_TAG
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_path(chapter_id: str, merge_key: str) -> Path:
    mk = str(merge_key)
    mk_h = hashlib.sha1(mk.encode("utf-8")).hexdigest()[:16]
    sig = rubric_sigs.get(str(chapter_id), "nosig")
    safe_model = RERANK_MODEL.replace("/", "_")
    d = CACHE_DIR / str(chapter_id)
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{mk_h}_{sig}_{PROMPT_VERSION}_{safe_model}.json"


def atomic_write_json(path: Path, obj: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


async def score_one(row: pd.Series) -> dict:
    cid = str(row["chapter_id"])
    mk = str(row["merge_key"])
    title = str(row.get("title", "") or "").strip()
    abstract = _truncate(str(row.get("abstract", "") or ""), ABSTRACT_MAX_CHARS)

    out_path = cache_path(cid, mk)
    if (not FORCE_RERANK) and out_path.exists():
        cached = json.loads(out_path.read_text(encoding="utf-8"))
        cached["_cached"] = True
        return cached

    rubric = rubrics[cid]
    prompt = build_prompt(rubric, title=title, abstract=abstract)

    last_err: Optional[str] = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = await Runner.run(rerank_agent, prompt)
            out = res.final_output.model_dump()
            usage = res.context_wrapper.usage
            cost = cost_from_usage(usage, model=RERANK_MODEL)
            payload = {
                "chapter_id": cid,
                "merge_key": mk,
                **out,
                "_meta": {
                    "model": RERANK_MODEL,
                    "prompt_version": PROMPT_VERSION,
                    "rubric_sig": rubric_sigs.get(cid),
                    **cost,
                },
                "_cached": False,
            }
            atomic_write_json(out_path, payload)
            return payload
        except Exception as e:
            last_err = repr(e)
            backoff = min(BACKOFF_MAX, BACKOFF_INITIAL * (2 ** (attempt - 1)))
            backoff *= (1.0 + random.uniform(-0.15, 0.15))
            await asyncio.sleep(max(0.1, backoff))

    raise RuntimeError(f"Rerank failed after {MAX_RETRIES} retries for {cid} {mk}: {last_err}")


async def run_all() -> List[dict]:
    sem = asyncio.Semaphore(CONCURRENCY)
    rows = df.copy()

    async def _bound(row: pd.Series) -> dict:
        async with sem:
            return await score_one(row)

    tasks = [_bound(r) for _, r in rows.iterrows()]
    out: List[dict] = []
    for fut in asyncio.as_completed(tasks):
        out.append(await fut)
        if len(out) % 50 == 0:
            print(f"Scored {len(out)}/{len(tasks)}")
    return out


t0 = time.time()
client = OpenAI()  # ensure API client init
print("Starting rerank… model=", RERANK_MODEL, "docs=", len(df))
results = await run_all()
seconds = time.time() - t0

# Build score columns
score_rows = []
for r in results:
    meta_r = (r.get("_meta") or {})
    cached = bool(r.get("_cached", False))
    score_rows.append(
        {
            "chapter_id": r.get("chapter_id"),
            "merge_key": r.get("merge_key"),
            "llm_score_topn_v2": float(r.get("score", 0.0)),
            "llm_notes_topn_v2": r.get("notes", ""),
            "llm_cached": cached,
            # Incremental cost: cached entries should not count as new spend
            "cost_usd": 0.0 if cached else float(meta_r.get("cost_usd", 0.0)),
            "input_tokens": 0 if cached else int(meta_r.get("input_tokens", 0)),
            "cached_input_tokens": 0 if cached else int(meta_r.get("cached_input_tokens", 0)),
            "output_tokens": 0 if cached else int(meta_r.get("output_tokens", 0)),
            "requests": 0 if cached else int(meta_r.get("requests", 1)),
            # Estimated total (including cached) to preserve the 'true' spend for logging
            "est_cost_usd": float(meta_r.get("cost_usd", 0.0)),
            "est_input_tokens": int(meta_r.get("input_tokens", 0)),
            "est_cached_input_tokens": int(meta_r.get("cached_input_tokens", 0)),
            "est_output_tokens": int(meta_r.get("output_tokens", 0)),
            "est_requests": int(meta_r.get("requests", 1)),
        }
    )

scores = pd.DataFrame(score_rows)

totals = {
    "seconds": float(seconds),
    "requests": float(scores["requests"].sum()),
    "input_tokens": float(scores["input_tokens"].sum()),
    "cached_input_tokens": float(scores["cached_input_tokens"].sum()),
    "output_tokens": float(scores["output_tokens"].sum()),
    "cost_usd": float(scores["cost_usd"].sum()),
    "cached_files": int(scores["llm_cached"].sum()),
}
print("Rerank totals (this run):", totals)

totals_est = {
    "seconds": float(seconds),
    "requests": float(scores["est_requests"].sum()),
    "input_tokens": float(scores["est_input_tokens"].sum()),
    "cached_input_tokens": float(scores["est_cached_input_tokens"].sum()),
    "output_tokens": float(scores["est_output_tokens"].sum()),
    "cost_usd": float(scores["est_cost_usd"].sum()),
    "cached_files": int(scores["llm_cached"].sum()),
}
print("Rerank totals (estimated total incl cached):", totals_est)

# Merge into dataset
df_r = df.merge(scores[["chapter_id", "merge_key", "llm_score", "llm_notes"]], on=["chapter_id", "merge_key"], how="left")
df_r["score_llm_rerank_v1"] = pd.to_numeric(df_r["llm_score"], errors="coerce").fillna(0.0) / 100.0

# Stage C finalized score (for comparison)
STAGEC_W_EMBED_MAX = 0.0
STAGEC_W_EMBED = 0.7
STAGEC_CITE_WEIGHT = 0.08

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

emb_max = pd.to_numeric(df_r["score_embed_max"], errors="coerce").fillna(0.0)
emb_b = pd.to_numeric(df_r["score_embed_mean_top3"], errors="coerce").fillna(0.0)
df_r["_emb_raw"] = float(STAGEC_W_EMBED_MAX) * emb_max + (1.0 - float(STAGEC_W_EMBED_MAX)) * emb_b
df_r["_emb_n"] = minmax_by_group(df_r, "_emb_raw")
df_r["_tf_n"] = minmax_by_group(df_r, "score_tfidf")
df_r["_cite_n"] = minmax_by_group(df_r, "score_cite_norm")
base = float(STAGEC_W_EMBED) * df_r["_emb_n"] + (1.0 - float(STAGEC_W_EMBED)) * df_r["_tf_n"]
df_r["score_stageC_final"] = (1.0 - float(STAGEC_CITE_WEIGHT)) * base + float(STAGEC_CITE_WEIGHT) * df_r["_cite_n"]

# Evaluate
score_cols = ["score_hybrid_pool", "score_stageC_final", "score_llm_rerank_v1"]
res = evaluate_all(df_r, score_cols=score_cols, ks=KS)

macro = res[res["chapter_id"] == "__ALL__"].sort_values("ndcg@20", ascending=False).reset_index(drop=True)
display(macro)

# Per-chapter ndcg@20 (quick sanity)
per_ndcg = res[res["chapter_id"] != "__ALL__"].pivot_table(index="chapter_id", columns="score_col", values="ndcg@20")
display(per_ndcg)

# Macro delta: LLM vs StageC_final
need = {"score_llm_rerank_v1", "score_stageC_final"}
if not macro.empty and need.issubset(set(macro["score_col"])):
    m = macro.set_index("score_col")
    cols = [c for c in ["ndcg@20", "p@20", "mrr_include", "auc_include"] if c in m.columns]
    delta = (m.loc["score_llm_rerank_v1", cols] - m.loc["score_stageC_final", cols]).to_frame("delta")
    print("Delta (score_llm_rerank_v1 - score_stageC_final):")
    display(delta)

# Persist artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC3_rerank_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

scored_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_v1_scored.csv"
df_r.to_csv(scored_path, index=False)
print("Saved:", scored_path)

eval_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_v1_eval.csv"
res.assign(**meta_use, experiment="stageC3_rerank_v1_eval").to_csv(eval_path, index=False)
print("Saved:", eval_path)

cost_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_v1_cost.csv"
pd.DataFrame([totals]).assign(**meta_use, model=RERANK_MODEL, prompt_version=PROMPT_VERSION).to_csv(cost_path, index=False)
print("Saved:", cost_path)

# Append best macro row (LLM rerank) to runs.csv
best_macro = macro[macro["score_col"] == "score_llm_rerank_v1"].copy()
if not best_macro.empty:
    best_macro = best_macro.assign(**meta_use, experiment="stageC3_rerank_v1")
    runs_csv = EXP_DIR / "runs.csv"
    if runs_csv.exists():
        header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
        for c in header:
            if c not in best_macro.columns:
                best_macro[c] = None
        best_macro = best_macro[header]
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended LLM rerank macro row to:", runs_csv)
    '''
    import textwrap
    _wrapped = "async def _run_stageC3_llm_rerank_v1():\n" + textwrap.indent(_CODE, "    ")
    exec(_wrapped, globals())
    await _run_stageC3_llm_rerank_v1()


Skipping st_stageC3_llm_rerank_v1 (RUN_MODE=label_adjudication)


In [15]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_rerank_offline_analysis_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 offline analysis (no API calls)
#
# We already ran `st_stageC3_llm_rerank_v1` once and saved a scored CSV.
# Now we test cheap ways to use the LLM signal without re-calling the API:
# - score fusion (weighted mix)
# - rerank only within Stage C top-N (simulates production usage)

from pathlib import Path

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)

df_scored = pd.read_csv(SCORED_PATH)

required = [
    "chapter_id",
    "merge_key",
    "final_label",
    "score_stageC_final",
    "score_llm_rerank_v1",
]
missing = [c for c in required if c not in df_scored.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out


# Normalize LLM score per chapter for score fusion
df_base = df_scored.copy()
df_base["_llm_n"] = minmax_by_group(df_base, "score_llm_rerank_v1")


# Baseline macro
base_res = evaluate_all(df_base, score_cols=["score_stageC_final", "score_llm_rerank_v1"], ks=KS)
base_macro = base_res[base_res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
print("Baseline macro:")
display(base_macro)


# 1) Score fusion sweep
ALPHAS = [round(float(x), 2) for x in np.linspace(0, 1, 21)]
mix_rows = []
for a in ALPHAS:
    d = df_base.copy()
    d["score_stageC3_mix_v1"] = (1.0 - float(a)) * pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0) + float(a) * d["_llm_n"]
    r = evaluate_all(d, score_cols=["score_stageC3_mix_v1"], ks=KS)
    m = r[r["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    mix_rows.append({
        "alpha_llm": float(a),
        "ndcg@20": float(m.get("ndcg@20", float("nan"))),
        "p@20": float(m.get("p@20", float("nan"))),
        "mrr_include": float(m.get("mrr_include", float("nan"))),
        "auc_include": float(m.get("auc_include", float("nan"))),
    })

mix_df = pd.DataFrame(mix_rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
print("\nScore fusion sweep (top 15):")
display(mix_df.head(15))
best_mix = mix_df.iloc[0].to_dict() if not mix_df.empty else {}
print("Best mix by ndcg@20:", best_mix)


# 2) Rerank within Stage C top-N (simulates production rerank on a shortlist)
TOPN_GRID = [20, 50, 100, 150, 220]
topn_rows = []
for topn in TOPN_GRID:
    d = df_base.copy()
    d["_in_topn"] = False
    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False).head(int(topn)).index
        d.loc[idx, "_in_topn"] = True

    # force top-N to appear above the rest, ordered by the LLM score
    d["score_stageC3_topn_v1"] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)
    d.loc[d["_in_topn"], "score_stageC3_topn_v1"] = 1.0 + d.loc[d["_in_topn"], "_llm_n"]

    r = evaluate_all(d, score_cols=["score_stageC3_topn_v1"], ks=KS)
    m = r[r["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    topn_rows.append({
        "topn": int(topn),
        "ndcg@20": float(m.get("ndcg@20", float("nan"))),
        "p@20": float(m.get("p@20", float("nan"))),
        "mrr_include": float(m.get("mrr_include", float("nan"))),
        "auc_include": float(m.get("auc_include", float("nan"))),
    })

topn_df = pd.DataFrame(topn_rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
print("\nTop-N rerank sweep:")
display(topn_df)
best_topn = topn_df.iloc[0].to_dict() if not topn_df.empty else {}
print("Best top-N by ndcg@20:", best_topn)


# Compare against Stage C final
stagec_ndcg = float(base_macro[base_macro["score_col"] == "score_stageC_final"].iloc[0]["ndcg@20"]) if (base_macro["score_col"] == "score_stageC_final").any() else float("nan")
print("\nStage C final ndcg@20:", stagec_ndcg)
if best_mix:
    print("Best mix delta ndcg@20:", float(best_mix["ndcg@20"]) - stagec_ndcg)
if best_topn:
    print("Best top-N delta ndcg@20:", float(best_topn["ndcg@20"]) - stagec_ndcg)


# Persist artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC3_rerank_offline_analysis_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

mix_path = EXP_DIR / f"{run_id_use}_stageC3_mix_sweep_v1.csv"
mix_df.assign(**meta_use, experiment="stageC3_mix_sweep_v1", scored_path=str(SCORED_PATH)).to_csv(mix_path, index=False)
print("Saved:", mix_path)

topn_path = EXP_DIR / f"{run_id_use}_stageC3_topn_sweep_v1.csv"
topn_df.assign(**meta_use, experiment="stageC3_topn_sweep_v1", scored_path=str(SCORED_PATH)).to_csv(topn_path, index=False)
print("Saved:", topn_path)
    '''
    import textwrap
    _wrapped = "def _run_stageC3_rerank_offline_analysis_v1():\n" + textwrap.indent(_CODE, "    ")
    exec(_wrapped, globals())
    _run_stageC3_rerank_offline_analysis_v1()


Skipping st_stageC3_rerank_offline_analysis_v1 (RUN_MODE=label_adjudication)


## Stage B — Blueprint tests (stability + quality)

These tests validate that blueprint generation is stable and that facet queries are diverse/useful.

- Some cells call the OpenAI API (cost-tracked).
- All results are saved under `eval_dataset/experiments/<stageB_run_id>_stageB/`.


In [16]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_config (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
import os
import re
import json
import time
import asyncio
import hashlib
import difflib
from itertools import combinations
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer

# API-only dependencies
try:
    from openai import OpenAI
    from agents import Agent, Runner, ModelSettings
    STAGEB_AGENTS_OK = True
except Exception as e:
    STAGEB_AGENTS_OK = False
    print("Warning: openai/agents not available -> API-based Stage B tests will be skipped.")
    print("Import error:", repr(e))

# -----------------------------
# Stage B config
# -----------------------------
DO_STAGEB_API_CALLS = True
STAGEB_FORCE_REGEN = False
STAGEB_CONCURRENCY = 5

BLUEPRINT_MODEL = "gpt-5-mini"
BLUEPRINT_RUNS_PER_CHAPTER = 5  # stability reps

# Pricing for cost estimates (update if pricing changes)
MODEL_PRICES_USD_PER_1M = {
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
}

def _price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})

def cost_from_usage(usage, model: str) -> dict:
    prices = _price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)
        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)
    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }

# Local StageB run folder
dataset_sha12 = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
stageB_run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha12}"
STAGEB_DIR = EXP_DIR / f"{stageB_run_id}_stageB"
STAGEB_DIR.mkdir(parents=True, exist_ok=True)
STABILITY_DIR = STAGEB_DIR / "stability"
STABILITY_DIR.mkdir(parents=True, exist_ok=True)
VARIANTS_DIR = STAGEB_DIR / "variants"
VARIANTS_DIR.mkdir(parents=True, exist_ok=True)

# Load CHAPTERS from eval_dataset_builder.ipynb to avoid duplication/drift
nb = json.loads(Path("eval_dataset_builder.ipynb").read_text(encoding="utf-8"))
spec_cell = next(c for c in nb.get("cells", []) if c.get("id") == "eval_chapter_specs")
spec_src = "".join(spec_cell.get("source", []))
ns: Dict[str, Any] = {"List": List, "Dict": Dict, "Any": Any}
exec(spec_src, ns, ns)
CHAPTERS: List[Dict[str, Any]] = ns["CHAPTERS"]

# Blueprint output schema (same as eval_dataset_builder.ipynb)
class ChapterBlueprint(BaseModel):
    chapter_id: str
    language: str = Field("en")
    scope_statement: str
    must_cover: List[str]
    should_cover: List[str]
    must_avoid: List[str]
    main_query: str
    facet_queries: List[str]
    keywords: List[str]
    key_concepts: List[str]
    preferred_source_types: Optional[List[str]] = None
    negative_query_terms: Optional[List[str]] = None
    scoring_guidance: str
    notes: Optional[str] = None

BASE_BLUEPRINT_INSTRUCTIONS = (
    "You create a chapter blueprint (rubric) and search queries for academic literature retrieval.\n"
    "Return ONLY the structured output fields (no extra text).\n\n"
    "Constraints:\n"
    "- language must be 'en'\n"
    "- scope_statement: 1 sentence, <= 30 words\n"
    "- must_cover: 4–8 bullets, each <= 16 words\n"
    "- should_cover: 3–8 bullets, each <= 16 words\n"
    "- must_avoid: 3–8 bullets, each <= 16 words\n"
    "- main_query: <= 18 words\n"
    "- facet_queries: 8–14 items, each <= 14 words\n"
    "- keywords: 20–45 items\n"
    "- key_concepts: 10–22 items\n"
    "- preferred_source_types: 2–6 items\n"
    "- negative_query_terms: 0–12 items derived from must_avoid (soft negatives)\n"
    "- scoring_guidance: <= 80 words\n"
    "- Do NOT contradict yourself: if something is in must_avoid, do not emphasize it in keywords/facets.\n"
    "- Do NOT hardcode any domain. Follow the given chapter spec.\n"
)

VARIANT_COVERAGE_V1_INSTRUCTIONS = BASE_BLUEPRINT_INSTRUCTIONS + (
    "\nAdditional requirements (coverage_v1):\n"
    "- facet_queries must be semantically diverse (avoid near-duplicates).\n"
    "- Ensure each must_cover bullet is explicitly targeted by at least one facet_query.\n"
)

print("Stage B config OK")
print("- stageB_run_id:", stageB_run_id)
print("- STAGEB_DIR:", STAGEB_DIR)
print("- CHAPTERS:", [c["chapter_id"] for c in CHAPTERS])
    '''
    exec(_CODE, globals())


Skipping st_stageB_config (RUN_MODE=label_adjudication)


In [17]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_load_base_blueprints (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Load the current baseline blueprints that were used to build the dataset
BASE_BLUEPRINT_DIR = DATASET_PATH.parent / "blueprints"
if not BASE_BLUEPRINT_DIR.exists():
    # Backwards compatibility (older datasets)
    BASE_BLUEPRINT_DIR = Path("eval_dataset/blueprints")

assert BASE_BLUEPRINT_DIR.exists(), f"Missing {BASE_BLUEPRINT_DIR}. Run eval_dataset_builder.ipynb first."

base_blueprints: Dict[str, dict] = {}
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    p = BASE_BLUEPRINT_DIR / f"{cid}.json"
    assert p.exists(), f"Missing blueprint: {p}"
    base_blueprints[cid] = json.loads(p.read_text(encoding="utf-8"))

rows = []
for cid, bp in base_blueprints.items():
    rows.append({
        "chapter_id": cid,
        "n_facets": len(bp.get("facet_queries", []) or []),
        "n_keywords": len(bp.get("keywords", []) or []),
        "n_key_concepts": len(bp.get("key_concepts", []) or []),
        "main_query": str(bp.get("main_query", ""))[:90],
    })

display(pd.DataFrame(rows))
print("Loaded baseline blueprints from:", BASE_BLUEPRINT_DIR)
    '''
    exec(_CODE, globals())


Skipping st_stageB_load_base_blueprints (RUN_MODE=label_adjudication)


In [18]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_blueprint_stability_runs (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 1 (must-do): Blueprint stability
# Generate multiple blueprints per chapter with identical settings and measure overlap.
# Cost: ~ (3 * BLUEPRINT_RUNS_PER_CHAPTER) calls to BLUEPRINT_MODEL.

if not DO_STAGEB_API_CALLS:
    print("Skipping blueprint stability runs (DO_STAGEB_API_CALLS=False)")
elif not STAGEB_AGENTS_OK:
    print("Skipping blueprint stability runs (openai/agents import failed)")
else:
    assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."
    _ = OpenAI()  # ensure client init

    blueprint_agent = Agent(
        name="Chapter Blueprint Builder (stability)",
        model=BLUEPRINT_MODEL,
        model_settings=ModelSettings(top_p=1.0, verbosity="low"),
        instructions=BASE_BLUEPRINT_INSTRUCTIONS,
        output_type=ChapterBlueprint,
    )

    sem = asyncio.Semaphore(int(STAGEB_CONCURRENCY))

    async def gen_one(chapter: dict, rep_i: int) -> dict:
        out_path = STABILITY_DIR / f"{chapter['chapter_id']}__r{rep_i:02d}.json"
        if out_path.exists() and not STAGEB_FORCE_REGEN:
            return {"chapter_id": chapter["chapter_id"], "rep": rep_i, "path": str(out_path), "cached": True, **{"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}}

        prompt = (
            "Create a ChapterBlueprint for academic literature retrieval.\n"
            "Return ONLY the structured output fields required by the schema.\n\n"
            "CHAPTER_SPEC_JSON:\n" + json.dumps(chapter, ensure_ascii=False, indent=2)
        )

        async with sem:
            res = await Runner.run(blueprint_agent, prompt)

        bp = res.final_output.model_dump()
        bp["_meta"] = {
            "chapter_id": chapter["chapter_id"],
            "rep": int(rep_i),
            "model": BLUEPRINT_MODEL,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        out_path.write_text(json.dumps(bp, ensure_ascii=False, indent=2), encoding="utf-8")

        usage = res.context_wrapper.usage
        u = cost_from_usage(usage, model=BLUEPRINT_MODEL)
        return {"chapter_id": chapter["chapter_id"], "rep": rep_i, "path": str(out_path), "cached": False, **u}

    t0 = time.time()
    tasks = [gen_one(ch, rep_i) for ch in CHAPTERS for rep_i in range(1, int(BLUEPRINT_RUNS_PER_CHAPTER) + 1)]
    run_rows = await asyncio.gather(*tasks)
    dt = time.time() - t0

    runs_df = pd.DataFrame(run_rows)
    display(runs_df)

    totals = {k: float(runs_df[k].sum()) for k in ["requests", "input_tokens", "cached_input_tokens", "output_tokens", "cost_usd"] if k in runs_df.columns}
    totals["cached_files"] = int((runs_df.get("cached") == True).sum())
    totals["seconds"] = float(dt)
    print("Blueprint stability run totals:", totals)

    out_csv = STAGEB_DIR / "blueprint_stability_runs.csv"
    runs_df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_blueprint_stability_runs (RUN_MODE=label_adjudication)


In [19]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_blueprint_stability_metrics (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Analyze stability runs: overlap of facets/keywords/etc across repeated blueprint generations.

def _norm_item(x: str) -> str:
    x = (x or "").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def _token_set(s: str) -> set:
    return set(re.findall(r"[a-z0-9]+", (s or "").lower()))

def _jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    u = a | b
    return float(len(a & b) / len(u)) if u else float("nan")

def _avg_min(vals: List[float]) -> tuple[float, float]:
    if not vals:
        return float("nan"), float("nan")
    return float(np.mean(vals)), float(np.min(vals))

def stability_list(bps: List[dict], field: str) -> tuple[float, float]:
    sets = [set(_norm_item(x) for x in (bp.get(field) or [])) for bp in bps]
    sims = [_jaccard(sets[i], sets[j]) for i, j in combinations(range(len(sets)), 2)]
    return _avg_min(sims)

def stability_text_tokens(bps: List[dict], field: str) -> tuple[float, float]:
    toks = [_token_set(bp.get(field) or "") for bp in bps]
    sims = [_jaccard(toks[i], toks[j]) for i, j in combinations(range(len(toks)), 2)]
    return _avg_min(sims)

def stability_text_seqratio(bps: List[dict], field: str) -> tuple[float, float]:
    texts = [_norm_item(bp.get(field) or "") for bp in bps]
    sims = [difflib.SequenceMatcher(None, texts[i], texts[j]).ratio() for i, j in combinations(range(len(texts)), 2)]
    return _avg_min(sims)

rows = []
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    files = sorted(STABILITY_DIR.glob(f"{cid}__r*.json"))
    if not files:
        print(f"No stability blueprints for {cid} in {STABILITY_DIR} (run the stability cell first)")
        continue

    bps = [json.loads(p.read_text(encoding="utf-8")) for p in files]
    hashes = [hashlib.sha1(p.read_bytes()).hexdigest() for p in files]

    facet_avg, facet_min = stability_list(bps, "facet_queries")
    kw_avg, kw_min = stability_list(bps, "keywords")
    kc_avg, kc_min = stability_list(bps, "key_concepts")
    mc_avg, mc_min = stability_list(bps, "must_cover")
    ma_avg, ma_min = stability_list(bps, "must_avoid")
    mq_avg, mq_min = stability_text_tokens(bps, "main_query")
    scope_avg, scope_min = stability_text_tokens(bps, "scope_statement")
    guide_avg, guide_min = stability_text_seqratio(bps, "scoring_guidance")

    rows.append({
        "stageB_run_id": stageB_run_id,
        "chapter_id": cid,
        "n_runs": int(len(files)),
        "n_unique_outputs": int(len(set(hashes))),
        "facet_jaccard_avg": facet_avg,
        "facet_jaccard_min": facet_min,
        "keywords_jaccard_avg": kw_avg,
        "keywords_jaccard_min": kw_min,
        "key_concepts_jaccard_avg": kc_avg,
        "key_concepts_jaccard_min": kc_min,
        "must_cover_jaccard_avg": mc_avg,
        "must_cover_jaccard_min": mc_min,
        "must_avoid_jaccard_avg": ma_avg,
        "must_avoid_jaccard_min": ma_min,
        "main_query_token_jaccard_avg": mq_avg,
        "main_query_token_jaccard_min": mq_min,
        "scope_token_jaccard_avg": scope_avg,
        "scope_token_jaccard_min": scope_min,
        "scoring_guidance_seqratio_avg": guide_avg,
        "scoring_guidance_seqratio_min": guide_min,
    })

stability_df = pd.DataFrame(rows)
display(stability_df)

out_csv = STAGEB_DIR / "blueprint_stability_metrics.csv"
stability_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_blueprint_stability_metrics (RUN_MODE=label_adjudication)


In [20]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_facet_redundancy (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 2 (must-do): Facet diversity / redundancy
# We compute TF-IDF cosine similarity across facet_queries (per blueprint) and flag near-duplicates.

def facet_redundancy_stats(bp: dict) -> dict:
    facets = [str(x) for x in (bp.get("facet_queries") or []) if str(x).strip()]
    main = str(bp.get("main_query") or "").strip()
    if len(facets) < 2:
        return {"n_facets": len(facets), "max_sim": float("nan"), "mean_sim": float("nan"), "pct_gt_0_8": float("nan"), "max_sim_to_main": float("nan")}

    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    X = vec.fit_transform(facets)
    sim = (X @ X.T).toarray()  # TF-IDF is L2 normalized -> dot == cosine
    n = sim.shape[0]
    off = sim[~np.eye(n, dtype=bool)]

    qsim = float("nan")
    if main:
        q = vec.transform([main])
        qsim = float((q @ X.T).toarray().ravel().max()) if X.shape[0] else float("nan")

    return {
        "n_facets": int(len(facets)),
        "max_sim": float(np.max(off)) if off.size else float("nan"),
        "mean_sim": float(np.mean(off)) if off.size else float("nan"),
        "pct_gt_0_8": float(np.mean(off > 0.8)) if off.size else float("nan"),
        "pct_gt_0_9": float(np.mean(off > 0.9)) if off.size else float("nan"),
        "max_sim_to_main": float(qsim),
    }

def top_similar_facet_pairs(bp: dict, min_sim: float = 0.85, top_n: int = 10) -> pd.DataFrame:
    facets = [str(x) for x in (bp.get("facet_queries") or []) if str(x).strip()]
    if len(facets) < 2:
        return pd.DataFrame([])

    vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    X = vec.fit_transform(facets)
    sim = (X @ X.T).toarray()
    pairs = []
    for i in range(len(facets)):
        for j in range(i + 1, len(facets)):
            s = float(sim[i, j])
            if s >= float(min_sim):
                pairs.append({"i": i, "j": j, "sim": s, "facet_i": facets[i], "facet_j": facets[j]})
    out = pd.DataFrame(pairs).sort_values("sim", ascending=False).head(int(top_n)) if pairs else pd.DataFrame([])
    return out

rows = []
for cid, bp in base_blueprints.items():
    rows.append({"blueprint_source": "baseline", "chapter_id": cid, **facet_redundancy_stats(bp)})

facet_df = pd.DataFrame(rows).sort_values(["chapter_id"]).reset_index(drop=True)
display(facet_df)

# Show near-duplicate facets if any
for cid, bp in base_blueprints.items():
    pairs = top_similar_facet_pairs(bp, min_sim=0.85, top_n=8)
    if not pairs.empty:
        print("\nNear-duplicate facets (baseline) for", cid)
        display(pairs)

out_csv = STAGEB_DIR / "facet_redundancy_baseline.csv"
facet_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_facet_redundancy (RUN_MODE=label_adjudication)


In [21]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_coverage_proxy (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 3 (should-do): Downstream coverage proxy
# Using the already-fetched StageA corpus (built from baseline blueprint queries),
# compare how many labeled relevant items are retrieved by:
# - main query only
# - union of main + facet queries (top-K per query)

STAGEA_DIR = DATASET_PATH.parent / "stageA"
if not STAGEA_DIR.exists():
    # Backwards compatibility (older datasets)
    STAGEA_DIR = Path("eval_dataset/stageA")

TFIDF_MAX_FEATURES = 200_000
TFIDF_MIN_DF = 2
TFIDF_NGRAM_RANGE = (1, 2)
TOPK_PER_QUERY = 250

def _clean_text(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()

def _build_text(title, abstract) -> str:
    t = _clean_text(title)
    a = _clean_text(abstract)
    if t and a:
        return f"{t}. {a}"
    return t or a

def _top_merge_keys(vec: TfidfVectorizer, X, df_docs: pd.DataFrame, query: str, topk: int) -> List[str]:
    q = vec.transform([str(query or "")])
    scores = (X @ q.T).toarray().ravel()
    topk = int(min(max(1, topk), len(scores)))
    idx = np.argpartition(-scores, topk - 1)[:topk]
    idx = idx[np.argsort(-scores[idx])]
    return df_docs.iloc[idx]["merge_key"].astype(str).tolist()

def _recall(rel: set, retrieved: set) -> float:
    return float(len(rel & retrieved) / len(rel)) if len(rel) > 0 else float("nan")

rows = []
for cid, bp in base_blueprints.items():
    stageA_path = STAGEA_DIR / f"stageA_combined_oa_s2_{cid}.csv"
    assert stageA_path.exists(), f"Missing {stageA_path}"

    docs = pd.read_csv(stageA_path)
    docs = docs.dropna(subset=["merge_key"]).copy()

    titles = docs["title"] if "title" in docs.columns else pd.Series([""] * len(docs))
    abstracts = docs["abstract"] if "abstract" in docs.columns else pd.Series([""] * len(docs))
    docs["_text"] = [_build_text(t, a) for t, a in zip(titles, abstracts)]

    # Relevant sets from labels (note: labels exist only for candidate subset)
    lab = df[df["chapter_id"].astype(str) == cid].copy()
    inc = set(lab[lab["final_label"] == "include"]["merge_key"].astype(str).tolist())
    incmaybe = set(lab[lab["final_label"].isin(["include", "maybe"])]["merge_key"].astype(str).tolist())

    vec = TfidfVectorizer(
        stop_words="english",
        ngram_range=TFIDF_NGRAM_RANGE,
        min_df=TFIDF_MIN_DF,
        max_features=TFIDF_MAX_FEATURES,
    )
    X = vec.fit_transform(docs["_text"].astype(str).tolist())

    main_q = str(bp.get("main_query") or "")
    facet_qs = [str(x) for x in (bp.get("facet_queries") or [])]
    queries = [main_q] + facet_qs

    union_keys = set()
    for q in queries:
        union_keys.update(_top_merge_keys(vec, X, docs, q, topk=TOPK_PER_QUERY))
    union_budget = len(union_keys)

    main_keys_equal = set(_top_merge_keys(vec, X, docs, main_q, topk=union_budget)) if union_budget > 0 else set()

    rows.append({
        "stageB_run_id": stageB_run_id,
        "chapter_id": cid,
        "stageA_docs": int(len(docs)),
        "n_facets": int(len(facet_qs)),
        "topk_per_query": int(TOPK_PER_QUERY),
        "union_budget": int(union_budget),
        "n_include": int(len(inc)),
        "n_include_or_maybe": int(len(incmaybe)),
        "recall_include_main_equalbudget": _recall(inc, main_keys_equal),
        "recall_include_facets_union": _recall(inc, union_keys),
        "recall_incmaybe_main_equalbudget": _recall(incmaybe, main_keys_equal),
        "recall_incmaybe_facets_union": _recall(incmaybe, union_keys),
    })

coverage_df = pd.DataFrame(rows)
display(coverage_df)

out_csv = STAGEB_DIR / "coverage_proxy_baseline.csv"
coverage_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageB_coverage_proxy (RUN_MODE=label_adjudication)


In [22]:
if RUN_MODE != "stageB_diag":
    print(f"Skipping st_stageB_prompt_variant_ab (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Test 4 (optional): A/B blueprint prompt variant (coverage_v1)
# NOTE: This evaluates blueprint-level quality metrics only.
# For a true end-to-end A/B, rebuild StageA+candidates+labels using the variant blueprint queries under a separate dataset tag.

variant_tag = "coverage_v1"
variant_dir = VARIANTS_DIR / variant_tag
variant_dir.mkdir(parents=True, exist_ok=True)

if not DO_STAGEB_API_CALLS:
    print("Skipping variant blueprint generation (DO_STAGEB_API_CALLS=False)")
elif not STAGEB_AGENTS_OK:
    print("Skipping variant blueprint generation (openai/agents import failed)")
else:
    assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY env var."
    _ = OpenAI()

    variant_agent = Agent(
        name=f"Chapter Blueprint Builder ({variant_tag})",
        model=BLUEPRINT_MODEL,
        model_settings=ModelSettings(top_p=1.0, verbosity="low"),
        instructions=VARIANT_COVERAGE_V1_INSTRUCTIONS,
        output_type=ChapterBlueprint,
    )

    sem = asyncio.Semaphore(int(STAGEB_CONCURRENCY))

    async def gen_variant(chapter: dict) -> dict:
        out_path = variant_dir / f"{chapter['chapter_id']}.json"
        if out_path.exists() and not STAGEB_FORCE_REGEN:
            return {"chapter_id": chapter["chapter_id"], "path": str(out_path), "cached": True, **{"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}}

        prompt = (
            "Create a ChapterBlueprint for academic literature retrieval.\n"
            "Return ONLY the structured output fields required by the schema.\n\n"
            "CHAPTER_SPEC_JSON:\n" + json.dumps(chapter, ensure_ascii=False, indent=2)
        )

        async with sem:
            res = await Runner.run(variant_agent, prompt)

        bp = res.final_output.model_dump()
        bp["_meta"] = {
            "chapter_id": chapter["chapter_id"],
            "variant": variant_tag,
            "model": BLUEPRINT_MODEL,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
        }
        out_path.write_text(json.dumps(bp, ensure_ascii=False, indent=2), encoding="utf-8")

        usage = res.context_wrapper.usage
        u = cost_from_usage(usage, model=BLUEPRINT_MODEL)
        return {"chapter_id": chapter["chapter_id"], "path": str(out_path), "cached": False, **u}

    t0 = time.time()
    rows = await asyncio.gather(*[gen_variant(ch) for ch in CHAPTERS])
    dt = time.time() - t0

    variant_runs = pd.DataFrame(rows)
    display(variant_runs)

    totals = {k: float(variant_runs[k].sum()) for k in ["requests", "input_tokens", "cached_input_tokens", "output_tokens", "cost_usd"] if k in variant_runs.columns}
    totals["cached_files"] = int((variant_runs.get("cached") == True).sum())
    totals["seconds"] = float(dt)
    print("Variant generation totals:", totals)

    out_csv = STAGEB_DIR / f"blueprint_variant_{variant_tag}_runs.csv"
    variant_runs.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

# Compare baseline vs variant on redundancy metrics (offline)
variant_blueprints = {}
for ch in CHAPTERS:
    cid = ch["chapter_id"]
    p = variant_dir / f"{cid}.json"
    if p.exists():
        variant_blueprints[cid] = json.loads(p.read_text(encoding="utf-8"))

rows = []
for cid, bp in base_blueprints.items():
    rows.append({"variant": "baseline", "chapter_id": cid, **facet_redundancy_stats(bp)})
for cid, bp in variant_blueprints.items():
    rows.append({"variant": variant_tag, "chapter_id": cid, **facet_redundancy_stats(bp)})

cmp_df = pd.DataFrame(rows).sort_values(["chapter_id", "variant"]).reset_index(drop=True)
display(cmp_df)

out_csv = STAGEB_DIR / f"facet_redundancy_{variant_tag}_compare.csv"
cmp_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

print("\nNOTE: True A/B requires re-running eval_dataset_builder.ipynb with the variant blueprints to fetch a new StageA corpus.")
    '''
    exec(_CODE, globals())


Skipping st_stageB_prompt_variant_ab (RUN_MODE=label_adjudication)


## Next steps (recommended)

1) **Freeze a baseline**: keep the `runs.csv` history and don’t change evaluation logic while tuning.
2) Make changes one-at-a-time (ablations), e.g.:
   - weights (`W_EMBED`, `W_TFIDF`, `CITE_WEIGHT`)
   - candidate pool size per facet (`TOP_PER_QUERY`)
   - candidate set size (`CAND_TARGET_N`)
3) After a few promising changes, re-run `eval_dataset_builder.ipynb` to regenerate candidates/labels and re-evaluate here.


In [23]:
if RUN_MODE != "stageB_ab":
    print(f"Skipping st_stageB_ab_compare (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage B end-to-end A/B compare (scientific workflow)
#
# Why this exists:
# - Stage B (blueprints) changes Stage A queries → fetched corpus → candidate pool.
# - So to compare Stage B variants scientifically, we must build separate datasets.
# Protocol:
# 1) Run `eval_dataset_builder.ipynb` twice (or more):
#    - Set `BLUEPRINT_VARIANT=baseline` (then run) → produces `eval_dataset/datasets/<tag>/labeled_dataset.csv`
#    - Set `BLUEPRINT_VARIANT=coverage_v1` (then run) → produces another dataset
#    - IMPORTANT: keep `LABEL_RUBRIC_SOURCE=eval_rubrics` so labels are comparable across datasets.
# 2) For each dataset, run the baseline evaluation cells in this notebook so it appends to `eval_dataset/experiments/runs.csv`.
#    - Easiest: set env var `DATASET_PATH` to the dataset's `.../labeled_dataset.csv` and re-run.
# 3) Finally run this cell to compare the macro metrics across datasets.

COMPARE_EXPERIMENT = STAGEB_AB_EXPERIMENT_TAG if RUN_MODE == "stageB_ab" else EXPERIMENT_TAG
COMPARE_SCORE_COL = STAGEB_AB_SCORE_COL if RUN_MODE == "stageB_ab" else "score_hybrid_pool"
DATASET_TAGS = STAGEB_AB_DATASET_TAGS if RUN_MODE == "stageB_ab" else []

runs_csv = EXP_DIR / "runs.csv"
if not runs_csv.exists():
    raise FileNotFoundError(f"Missing runs.csv at {runs_csv}. Run at least one evaluation first.")

runs = pd.read_csv(runs_csv)
runs = runs.copy()

if "experiment" not in runs.columns:
    raise ValueError("runs.csv schema is unexpected (missing 'experiment' column)")
if "score_col" not in runs.columns:
    raise ValueError("runs.csv schema is unexpected (missing 'score_col' column)")

runs = runs[(runs["experiment"].astype(str) == str(COMPARE_EXPERIMENT)) & (runs["score_col"].astype(str) == str(COMPARE_SCORE_COL))]
if runs.empty:
    raise ValueError(
        "No rows matched (experiment, score_col). "
        "Did you run an evaluation that appended to runs.csv with the same EXPERIMENT_TAG and score_col?"
    )


def dataset_tag_from_path(p: str) -> str:
    s = ("" if p is None else str(p)).replace("\\", "/")
    needle = "eval_dataset/datasets/"
    if needle not in s:
        return "legacy"
    rest = s.split(needle, 1)[1]
    return rest.split("/", 1)[0]


runs["dataset_tag"] = runs["dataset_path"].apply(dataset_tag_from_path)
runs["created_at_utc"] = pd.to_datetime(runs.get("created_at_utc"), utc=True, errors="coerce")

if DATASET_TAGS:
    runs = runs[runs["dataset_tag"].isin(DATASET_TAGS)]

# Auto selection: compare the most recent datasets found in runs.csv
if not DATASET_TAGS:
    most_recent = (
        runs.dropna(subset=["created_at_utc"])
        .sort_values("created_at_utc", ascending=False)
        .drop_duplicates(subset=["dataset_tag"])
        .head(6)
    )
    DATASET_TAGS = list(most_recent["dataset_tag"].astype(str).values)
    print("Auto-selected dataset tags:", DATASET_TAGS)
    runs = runs[runs["dataset_tag"].isin(DATASET_TAGS)]

# Latest macro row per dataset_tag
latest = (
    runs.sort_values("created_at_utc", ascending=True)
    .groupby(["dataset_tag"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# Join dataset manifests (if present)
datasets_root = Path("eval_dataset/datasets")
manifest_rows = []
for tag in latest["dataset_tag"].astype(str).unique():
    if tag == "legacy":
        manifest_rows.append({"dataset_tag": tag})
        continue
    mp = datasets_root / tag / "manifest.json"
    if not mp.exists():
        manifest_rows.append({"dataset_tag": tag})
        continue
    try:
        m = json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        m = {}

    manifest_rows.append(
        {
            "dataset_tag": tag,
            "blueprint_variant": m.get("blueprint_variant"),
            "label_rubric_source": m.get("label_rubric_source"),
            "dataset_created_at_utc": m.get("created_at_utc"),
        }
    )

manifest_df = pd.DataFrame(manifest_rows).drop_duplicates(subset=["dataset_tag"], keep="first")
summary = latest.merge(manifest_df, on="dataset_tag", how="left")

cols = [
    "dataset_tag",
    "blueprint_variant",
    "label_rubric_source",
    "dataset_sha1_12",
    "created_at_utc",
    "ndcg@20",
    "p@20",
    "mrr_include",
    "auc_include",
    "n_docs",
    "n_include",
    "n_maybe",
    "dataset_path",
    "run_id",
]
cols = [c for c in cols if c in summary.columns]
summary = summary[cols].sort_values("ndcg@20", ascending=False).reset_index(drop=True)

display(summary)

# Optional: metric deltas if exactly two datasets are specified
if DATASET_TAGS and len(DATASET_TAGS) == 2 and {"ndcg@20", "p@20", "mrr_include", "auc_include"}.issubset(summary.columns):
    a, b = DATASET_TAGS
    s = summary.set_index("dataset_tag")
    if a in s.index and b in s.index:
        delta = (s.loc[b, ["ndcg@20", "p@20", "mrr_include", "auc_include"]] - s.loc[a, ["ndcg@20", "p@20", "mrr_include", "auc_include"]]).to_frame("delta")
        print(f"\nDelta ({b} - {a}):")
        display(delta)

# Persist comparison output
out_path = EXP_DIR / f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_stageB_ab_compare.csv"
summary.to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageB_ab_compare (RUN_MODE=label_adjudication)


In [24]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_rerank_offline_loco_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 offline LOCO validation (no API calls)
#
# Scientific intent:
# - We found that full LLM rerank v1 was worse than Stage C final.
# - But offline analysis showed big gains if we use the LLM signal only as:
#   (a) a weak score fusion signal, or
#   (b) a top-N shortlist reranker.
# - This cell validates those choices with LOCO (leave-one-chapter-out)
#   to reduce overfitting: choose alpha/topN on 2 chapters, evaluate on the 3rd.

from pathlib import Path

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df_scored = pd.read_csv(SCORED_PATH)

required = [
    "chapter_id",
    "merge_key",
    "final_label",
    "score_stageC_final",
    "score_llm_rerank_v1",
]
missing = [c for c in required if c not in df_scored.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out


df = df_scored.copy()
df["_llm_n"] = minmax_by_group(df, "score_llm_rerank_v1")

chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
print("Chapters:", chapters)

ALPHAS = [round(float(x), 2) for x in np.linspace(0, 1, 21)]
TOPN_GRID = [20, 50, 100, 150, 220]


def _macro_metric(df_in: pd.DataFrame, score_col: str, metric_col: str) -> float:
    r = evaluate_all(df_in, score_cols=[score_col], ks=KS)
    return float(r[r["chapter_id"] == "__ALL__"].iloc[0][metric_col])


def _score_mix(df_in: pd.DataFrame, alpha_llm: float) -> pd.DataFrame:
    d = df_in.copy()
    stagec = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)
    llm = pd.to_numeric(d["_llm_n"], errors="coerce").fillna(0.0)
    d["score_stageC3_mix_v1"] = (1.0 - float(alpha_llm)) * stagec + float(alpha_llm) * llm
    return d


def _score_topn(df_in: pd.DataFrame, topn: int) -> pd.DataFrame:
    d = df_in.copy()
    d["_in_topn"] = False
    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False).head(int(topn)).index
        d.loc[idx, "_in_topn"] = True

    # keep Stage C ordering for the tail; rerank only within the shortlist
    d["score_stageC3_topn_v1"] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)
    d.loc[d["_in_topn"], "score_stageC3_topn_v1"] = 1.0 + pd.to_numeric(d.loc[d["_in_topn"], "_llm_n"], errors="coerce").fillna(0.0)
    return d


# -----------------------------
# LOCO: score fusion
# -----------------------------
mix_rows = []
for holdout in chapters:
    train = df[df["chapter_id"].astype(str) != holdout].copy()
    hold = df[df["chapter_id"].astype(str) == holdout].copy()

    base_train = _macro_metric(train, "score_stageC_final", "ndcg@20")
    base_hold = _macro_metric(hold, "score_stageC_final", "ndcg@20")

    best_alpha = None
    best_train = -1e9
    for a in ALPHAS:
        t = _score_mix(train, alpha_llm=float(a))
        nd = _macro_metric(t, "score_stageC3_mix_v1", "ndcg@20")
        if nd > best_train:
            best_train = nd
            best_alpha = float(a)

    h = _score_mix(hold, alpha_llm=float(best_alpha))
    hold_nd = _macro_metric(h, "score_stageC3_mix_v1", "ndcg@20")

    mix_rows.append({
        "holdout_chapter": holdout,
        "selected_alpha_llm": float(best_alpha),
        "train_ndcg@20": float(best_train),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_stageC_ndcg@20": float(base_hold),
        "holdout_delta_vs_stageC": float(hold_nd - base_hold),
        "train_base_stageC_ndcg@20": float(base_train),
    })

mix_loco = pd.DataFrame(mix_rows)
print("\nLOCO (score fusion) results:")
display(mix_loco)
avg_mix = float(mix_loco["holdout_ndcg@20"].mean()) if len(mix_loco) else float("nan")
avg_mix_delta = float(mix_loco["holdout_delta_vs_stageC"].mean()) if len(mix_loco) else float("nan")
print("Avg LOCO holdout ndcg@20 (mix):", avg_mix)
print("Avg LOCO holdout delta vs Stage C (mix):", avg_mix_delta)


# -----------------------------
# LOCO: top-N shortlist rerank
# -----------------------------
topn_rows = []
for holdout in chapters:
    train = df[df["chapter_id"].astype(str) != holdout].copy()
    hold = df[df["chapter_id"].astype(str) == holdout].copy()

    base_train = _macro_metric(train, "score_stageC_final", "ndcg@20")
    base_hold = _macro_metric(hold, "score_stageC_final", "ndcg@20")

    best_topn = None
    best_train = -1e9
    for topn in TOPN_GRID:
        t = _score_topn(train, topn=int(topn))
        nd = _macro_metric(t, "score_stageC3_topn_v1", "ndcg@20")
        if nd > best_train:
            best_train = nd
            best_topn = int(topn)

    h = _score_topn(hold, topn=int(best_topn))
    hold_nd = _macro_metric(h, "score_stageC3_topn_v1", "ndcg@20")

    topn_rows.append({
        "holdout_chapter": holdout,
        "selected_topn": int(best_topn),
        "train_ndcg@20": float(best_train),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_stageC_ndcg@20": float(base_hold),
        "holdout_delta_vs_stageC": float(hold_nd - base_hold),
        "train_base_stageC_ndcg@20": float(base_train),
    })

topn_loco = pd.DataFrame(topn_rows)
print("\nLOCO (top-N shortlist rerank) results:")
display(topn_loco)
avg_topn = float(topn_loco["holdout_ndcg@20"].mean()) if len(topn_loco) else float("nan")
avg_topn_delta = float(topn_loco["holdout_delta_vs_stageC"].mean()) if len(topn_loco) else float("nan")
print("Avg LOCO holdout ndcg@20 (top-N):", avg_topn)
print("Avg LOCO holdout delta vs Stage C (top-N):", avg_topn_delta)


# -----------------------------
# Persist artifacts
# -----------------------------
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC3_rerank_analysis_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

mix_path = EXP_DIR / f"{run_id_use}_stageC3_mix_loco_v1.csv"
mix_loco.assign(**meta_use, experiment="stageC3_mix_loco_v1", scored_path=str(SCORED_PATH)).to_csv(mix_path, index=False)
print("Saved:", mix_path)

topn_path = EXP_DIR / f"{run_id_use}_stageC3_topn_loco_v1.csv"
topn_loco.assign(**meta_use, experiment="stageC3_topn_loco_v1", scored_path=str(SCORED_PATH)).to_csv(topn_path, index=False)
print("Saved:", topn_path)
    '''
    exec(_CODE, globals())


Skipping st_stageC3_rerank_offline_loco_v1 (RUN_MODE=label_adjudication)


In [25]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_topn_blend_sweep_loco_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 refinement (no API): blend Stage C + LLM within the top-N shortlist
#
# Scientific intent:
# - We found LOCO-confirmed gains for: topN=50 and pure LLM order within the shortlist.
# - This cell tests whether a *blend* inside the shortlist is even better:
#     score_in_topN = (1-beta)*score_stageC_final + beta*llm_norm
# - Outside topN we keep Stage C ordering.
# - We select beta by macro ndcg@20 and validate with LOCO.

from pathlib import Path

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df_scored = pd.read_csv(SCORED_PATH)

required = [
    "chapter_id",
    "merge_key",
    "final_label",
    "score_stageC_final",
    "score_llm_rerank_v1",
]
missing = [c for c in required if c not in df_scored.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out


TOPN = 50  # fixed from LOCO winner
BETAS = [round(float(x), 2) for x in np.linspace(0, 1, 21)]

df = df_scored.copy()
df["_llm_n"] = minmax_by_group(df, "score_llm_rerank_v1")

chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
print("Chapters:", chapters)
print("TOPN:", TOPN)


def _apply_topn_blend(df_in: pd.DataFrame, beta: float) -> pd.DataFrame:
    d = df_in.copy()
    d["_in_topn"] = False
    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False).head(int(TOPN)).index
        d.loc[idx, "_in_topn"] = True

    stagec = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)
    llm = pd.to_numeric(d["_llm_n"], errors="coerce").fillna(0.0)

    # Keep Stage C ordering for the tail; reorder only within shortlist.
    d["score_stageC3_topn_blend_v1"] = stagec
    d.loc[d["_in_topn"], "score_stageC3_topn_blend_v1"] = 1.0 + ((1.0 - float(beta)) * stagec + float(beta) * llm)
    return d


# -----------------------------
# Sweep betas (macro)
# -----------------------------
rows = []
for b in BETAS:
    d = _apply_topn_blend(df, beta=float(b))
    res = evaluate_all(d, score_cols=["score_stageC3_topn_blend_v1"], ks=KS)
    m = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "beta": float(b),
        "ndcg@20": float(m.get("ndcg@20")),
        "p@20": float(m.get("p@20")),
        "mrr_include": float(m.get("mrr_include")),
        "auc_include": float(m.get("auc_include")),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
print("\nTopN blend beta sweep (top 15):")
display(sweep_df.head(15))
best_beta = sweep_df.iloc[0].to_dict() if len(sweep_df) else {}
print("Best beta by ndcg@20:", best_beta)


# -----------------------------
# LOCO selection for beta
# -----------------------------
loco_rows = []
for holdout in chapters:
    train = df[df["chapter_id"].astype(str) != holdout].copy()
    hold = df[df["chapter_id"].astype(str) == holdout].copy()

    base_hold = float(evaluate_all(hold, score_cols=["score_stageC_final"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"]) 

    best_b = None
    best_train = -1e9
    for b in BETAS:
        t = _apply_topn_blend(train, beta=float(b))
        nd = float(evaluate_all(t, score_cols=["score_stageC3_topn_blend_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"]) 
        if nd > best_train:
            best_train = nd
            best_b = float(b)

    h = _apply_topn_blend(hold, beta=float(best_b))
    hold_nd = float(evaluate_all(h, score_cols=["score_stageC3_topn_blend_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"]) 

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_beta": float(best_b),
        "train_ndcg@20": float(best_train),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_stageC_ndcg@20": float(base_hold),
        "holdout_delta_vs_stageC": float(hold_nd - base_hold),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO (topN blend beta) results:")
display(loco_df)
avg_hold = float(loco_df["holdout_ndcg@20"].mean()) if len(loco_df) else float("nan")
avg_delta = float(loco_df["holdout_delta_vs_stageC"].mean()) if len(loco_df) else float("nan")
print("Avg LOCO holdout ndcg@20 (topN blend):", avg_hold)
print("Avg LOCO holdout delta vs Stage C (topN blend):", avg_delta)


# Persist artifacts
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC3_rerank_analysis_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_topn_blend_beta_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageC3_topn_blend_beta_sweep_v1", scored_path=str(SCORED_PATH), topn=int(TOPN)).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_topn_blend_beta_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageC3_topn_blend_beta_loco_v1", scored_path=str(SCORED_PATH), topn=int(TOPN)).to_csv(loco_path, index=False)
print("Saved:", loco_path)
    '''
    exec(_CODE, globals())


Skipping st_stageC3_topn_blend_sweep_loco_v1 (RUN_MODE=label_adjudication)


In [26]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageD_mmr_tfidf_sweep_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage D (candidate diversity): MMR sweep using TF-IDF similarity (no API)
#
# Scientific intent:
# - Stage D should increase *diversity* of the final top-K list without harming relevance too much.
# - Our labels only cover relevance, so we measure:
#     (A) relevance quality via ndcg@20/p@20/mrr_include
#     (B) redundancy via mean pairwise cosine similarity in the selected top-K
#
# Baseline for this test is the finalized Stage C.3 behavior:
# - top-50 shortlist by score_stageC_final
# - within that shortlist, order by LLM score (normalized)
# - remainder stays in Stage C order

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXP_DIR = Path("eval_dataset/experiments")

# Load latest rerank-scored CSV (contains llm score + labels)
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "title", "abstract", "score_stageC_final", "score_llm_rerank_v1"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_topn_final(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    df["_in_topn"] = False

    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False).head(int(topn)).index
        df.loc[idx, "_in_topn"] = True
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"])
        df.loc[idx, "score_stageC3_topn_final"] = 1.0 + llm_n

    return df


def build_text(df_in: pd.DataFrame) -> pd.Series:
    t = df_in["title"].fillna("").astype(str)
    a = df_in["abstract"].fillna("").astype(str)
    return (t + ". " + a).str.strip()


def mean_pairwise_sim(S: np.ndarray, chosen: List[int]) -> float:
    if len(chosen) <= 1:
        return float("nan")
    # mean of upper triangle (excluding diagonal)
    vals = []
    for i in range(len(chosen)):
        for j in range(i + 1, len(chosen)):
            vals.append(float(S[chosen[i], chosen[j]]))
    return float(np.mean(vals)) if vals else float("nan")


def mmr_select(relevance: np.ndarray, sim: np.ndarray, k: int, lam: float) -> List[int]:
    n = int(len(relevance))
    if n == 0:
        return []
    k = int(min(k, n))

    chosen: List[int] = []
    remaining = list(range(n))

    # start with highest relevance
    first = int(np.argmax(relevance))
    chosen.append(first)
    remaining.remove(first)

    eps = 1e-12
    while len(chosen) < k and remaining:
        best_i = None
        best_score = -1e18
        for i in remaining:
            max_sim = max(float(sim[i, j]) for j in chosen) if chosen else 0.0
            s = float(lam) * float(relevance[i]) - (1.0 - float(lam)) * float(max_sim)
            if (s > best_score + eps) or (abs(s - best_score) <= eps and (best_i is None or int(i) < int(best_i))):
                best_score = s
                best_i = int(i)
        chosen.append(int(best_i))
        remaining.remove(int(best_i))

    return chosen


# -----------------------------
# Baseline score (Stage C.3 top-50 rerank)
# -----------------------------
df = add_stagec3_topn_final(df0, topn=50)

base_res = evaluate_all(df, score_cols=["score_stageC3_topn_final", "score_stageC_final"], ks=KS)
base_macro = base_res[base_res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
print("Baseline macro:")
display(base_macro)


# -----------------------------
# Sweep MMR params
# -----------------------------
K_SELECT = 20
TOPM_GRID = [50, 100, 150, 220]
LAM_GRID = [0.6, 0.7, 0.8, 0.9, 1.0]

rows = []
for topm in TOPM_GRID:
    for lam in LAM_GRID:
        df_v = df.copy()
        df_v["score_stageD_mmr_tfidf_v1"] = pd.to_numeric(df_v["score_stageC3_topn_final"], errors="coerce").fillna(0.0)

        div_sims = []
        for cid, g in df_v.groupby("chapter_id"):
            g = g.sort_values("score_stageC3_topn_final", ascending=False)
            pool = g.head(int(topm)).copy()

            texts = build_text(pool).tolist()
            tfidf = TfidfVectorizer(max_features=20000)
            X = tfidf.fit_transform(texts)
            S = cosine_similarity(X)

            rel = pd.to_numeric(pool["score_stageC3_topn_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
            chosen_local = mmr_select(rel, S, k=K_SELECT, lam=float(lam))
            div_sims.append(mean_pairwise_sim(S, chosen_local))

            chosen_idx = pool.iloc[chosen_local].index.tolist()
            for r, ix in enumerate(chosen_idx):
                df_v.loc[ix, "score_stageD_mmr_tfidf_v1"] = 2.0 - (r * 1e-6)

        res = evaluate_all(df_v, score_cols=["score_stageD_mmr_tfidf_v1"], ks=KS)
        macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
        rows.append({
            "topm": int(topm),
            "lambda": float(lam),
            "ndcg@20": float(macro.get("ndcg@20")),
            "p@20": float(macro.get("p@20")),
            "mrr_include": float(macro.get("mrr_include")),
            "auc_include": float(macro.get("auc_include")),
            "mean_pairwise_sim_top20": float(np.nanmean(div_sims)) if div_sims else float("nan"),
        })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mean_pairwise_sim_top20"], ascending=[False, True]).reset_index(drop=True)
print("\nStage D MMR TF-IDF sweep (top 20 rows):")
display(sweep_df.head(20))


# Baseline reference numbers
base_ndcg = float(base_macro[base_macro["score_col"] == "score_stageC3_topn_final"].iloc[0]["ndcg@20"])
print("Baseline ndcg@20 (StageC3 top50):", base_ndcg)

# Best-by-diversity under a small ndcg drop constraint
MAX_NDCG_DROP = 0.01
ok = sweep_df[sweep_df["ndcg@20"] >= (base_ndcg - MAX_NDCG_DROP)].copy()
if len(ok):
    ok = ok.sort_values(["mean_pairwise_sim_top20", "ndcg@20"], ascending=[True, False]).reset_index(drop=True)
    print(f"\nBest diversity with ndcg@20 drop <= {MAX_NDCG_DROP}:")
    display(ok.head(10))
else:
    print("\nNo MMR settings met the ndcg constraint; consider relaxing MAX_NDCG_DROP or using MMR only as a final UX step.")


# Persist sweep
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageD_diversity_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

out_path = EXP_DIR / f"{run_id_use}_stageD_mmr_tfidf_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageD_mmr_tfidf_sweep_v1", scored_path=str(SCORED_PATH), k_select=int(K_SELECT)).to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageD_mmr_tfidf_sweep_v1 (RUN_MODE=label_adjudication)


In [27]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageD_mmr_tfidf_loco_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage D LOCO validation: choose (topm, lambda) on 2 chapters, evaluate on holdout.
# No API calls.

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXP_DIR = Path("eval_dataset/experiments")

cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "title", "abstract", "score_stageC_final", "score_llm_rerank_v1"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_topn_final(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    df["_in_topn"] = False
    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False).head(int(topn)).index
        df.loc[idx, "_in_topn"] = True
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"])
        df.loc[idx, "score_stageC3_topn_final"] = 1.0 + llm_n
    return df


def build_text(df_in: pd.DataFrame) -> pd.Series:
    t = df_in["title"].fillna("").astype(str)
    a = df_in["abstract"].fillna("").astype(str)
    return (t + ". " + a).str.strip()


def mean_pairwise_sim(S: np.ndarray, chosen: List[int]) -> float:
    if len(chosen) <= 1:
        return float("nan")
    vals = []
    for i in range(len(chosen)):
        for j in range(i + 1, len(chosen)):
            vals.append(float(S[chosen[i], chosen[j]]))
    return float(np.mean(vals)) if vals else float("nan")


def mmr_select(relevance: np.ndarray, sim: np.ndarray, k: int, lam: float) -> List[int]:
    n = int(len(relevance))
    if n == 0:
        return []
    k = int(min(k, n))

    chosen: List[int] = []
    remaining = list(range(n))

    first = int(np.argmax(relevance))
    chosen.append(first)
    remaining.remove(first)

    eps = 1e-12
    while len(chosen) < k and remaining:
        best_i = None
        best_score = -1e18
        for i in remaining:
            max_sim = max(float(sim[i, j]) for j in chosen) if chosen else 0.0
            s = float(lam) * float(relevance[i]) - (1.0 - float(lam)) * float(max_sim)
            if (s > best_score + eps) or (abs(s - best_score) <= eps and (best_i is None or int(i) < int(best_i))):
                best_score = s
                best_i = int(i)
        chosen.append(int(best_i))
        remaining.remove(int(best_i))
    return chosen


def topk_similarity(g: pd.DataFrame, score_col: str, k: int) -> float:
    g = g.sort_values(score_col, ascending=False).head(int(k)).copy()
    if len(g) <= 1:
        return float("nan")
    tfidf = TfidfVectorizer(max_features=20000)
    X = tfidf.fit_transform(build_text(g).tolist())
    S = cosine_similarity(X)
    return mean_pairwise_sim(S, list(range(len(g))))


def apply_stageD_mmr_tfidf(df_in: pd.DataFrame, topm: int, lam: float, k_select: int = 20) -> Tuple[pd.DataFrame, float]:
    df_v = df_in.copy()
    df_v["score_stageD_mmr_tfidf_v1"] = pd.to_numeric(df_v["score_stageC3_topn_final"], errors="coerce").fillna(0.0)
    sims = []

    for cid, g in df_v.groupby("chapter_id"):
        g = g.sort_values("score_stageC3_topn_final", ascending=False)
        pool = g.head(int(topm)).copy()

        texts = build_text(pool).tolist()
        tfidf = TfidfVectorizer(max_features=20000)
        X = tfidf.fit_transform(texts)
        S = cosine_similarity(X)

        rel = pd.to_numeric(pool["score_stageC3_topn_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        chosen_local = mmr_select(rel, S, k=int(k_select), lam=float(lam))
        sims.append(mean_pairwise_sim(S, chosen_local))

        chosen_idx = pool.iloc[chosen_local].index.tolist()
        for r, ix in enumerate(chosen_idx):
            df_v.loc[ix, "score_stageD_mmr_tfidf_v1"] = 2.0 - (r * 1e-6)

    return df_v, float(np.nanmean(sims)) if sims else float("nan")


# Build baseline Stage C.3 score
df = add_stagec3_topn_final(df0, topn=50)
chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
print("Chapters:", chapters)

K_SELECT = 20
TOPM_GRID = [50, 100, 150, 220]
LAM_GRID = [0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99, 1.0]


def _macro_ndcg(df_in: pd.DataFrame, score_col: str) -> float:
    r = evaluate_all(df_in, score_cols=[score_col], ks=KS)
    return float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"]) 


rows = []
for holdout in chapters:
    train = df[df["chapter_id"].astype(str) != holdout].copy()
    hold = df[df["chapter_id"].astype(str) == holdout].copy()

    base_train_nd = _macro_ndcg(train, "score_stageC3_topn_final")
    base_hold_nd = _macro_ndcg(hold, "score_stageC3_topn_final")
    base_hold_sim = topk_similarity(hold, "score_stageC3_topn_final", k=int(K_SELECT))

    best = None
    best_train_nd = -1e9
    best_train_sim = float("inf")

    for topm in TOPM_GRID:
        for lam in LAM_GRID:
            t_scored, t_sim = apply_stageD_mmr_tfidf(train, topm=int(topm), lam=float(lam), k_select=int(K_SELECT))
            nd = _macro_ndcg(t_scored, "score_stageD_mmr_tfidf_v1")

            if (nd > best_train_nd + 1e-12) or (abs(nd - best_train_nd) <= 1e-12 and float(t_sim) < float(best_train_sim)):
                best_train_nd = float(nd)
                best_train_sim = float(t_sim)
                best = {"topm": int(topm), "lambda": float(lam)}

    h_scored, h_sim = apply_stageD_mmr_tfidf(hold, topm=int(best["topm"]), lam=float(best["lambda"]), k_select=int(K_SELECT))
    hold_nd = _macro_ndcg(h_scored, "score_stageD_mmr_tfidf_v1")

    rows.append({
        "holdout_chapter": holdout,
        "selected_topm": int(best["topm"]),
        "selected_lambda": float(best["lambda"]),
        "train_ndcg@20": float(best_train_nd),
        "train_base_ndcg@20": float(base_train_nd),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_hold_nd),
        "holdout_delta_vs_base": float(hold_nd - base_hold_nd),
        "holdout_sim_top20": float(h_sim),
        "holdout_base_sim_top20": float(base_hold_sim),
        "holdout_sim_delta": float(h_sim - base_hold_sim),
    })

loco_df = pd.DataFrame(rows)
print("\nStage D LOCO results:")
display(loco_df)
print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))
print("Avg LOCO holdout sim_top20:", float(loco_df["holdout_sim_top20"].mean()))
print("Avg LOCO holdout sim delta:", float(loco_df["holdout_sim_delta"].mean()))


# Persist
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageD_diversity_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

out_path = EXP_DIR / f"{run_id_use}_stageD_mmr_tfidf_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageD_mmr_tfidf_loco_v1", scored_path=str(SCORED_PATH), k_select=int(K_SELECT)).to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageD_mmr_tfidf_loco_v1 (RUN_MODE=label_adjudication)


In [28]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageD_facet_cap_sweep_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage D alternative (no API): facet-cap selection using `facet_best_i`
#
# Scientific intent:
# - Reduce redundancy by preventing the top-K list from being dominated by a single facet.
# - Keep relevance high (measure via ndcg@20/p@20/mrr_include).

from pathlib import Path
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = [
    "chapter_id",
    "merge_key",
    "final_label",
    "title",
    "abstract",
    "score_stageC_final",
    "score_llm_rerank_v1",
    "facet_best_i",
]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_topn_final(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"])
        df.loc[idx, "score_stageC3_topn_final"] = 1.0 + llm_n
    return df


def build_text(df_in: pd.DataFrame) -> pd.Series:
    t = df_in["title"].fillna("").astype(str)
    a = df_in["abstract"].fillna("").astype(str)
    return (t + ". " + a).str.strip()


def mean_pairwise_sim(g: pd.DataFrame) -> float:
    if len(g) <= 1:
        return float("nan")
    tfidf = TfidfVectorizer(max_features=20000)
    X = tfidf.fit_transform(build_text(g).tolist())
    S = cosine_similarity(X)
    vals = []
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            vals.append(float(S[i, j]))
    return float(np.mean(vals)) if vals else float("nan")


def facet_cap_select(pool: pd.DataFrame, k: int, max_per_facet: int) -> List[int]:
    counts = defaultdict(int)
    chosen = []
    for ix, row in pool.iterrows():
        v = pd.to_numeric(row.get("facet_best_i"), errors="coerce")
        f = int(-1 if pd.isna(v) else v)
        if counts[f] < int(max_per_facet):
            chosen.append(ix)
            counts[f] += 1
        if len(chosen) >= int(k):
            break
    if len(chosen) < int(k):
        for ix in pool.index:
            if ix in chosen:
                continue
            chosen.append(ix)
            if len(chosen) >= int(k):
                break
    return chosen


df = add_stagec3_topn_final(df0, topn=50)
base = evaluate_all(df, score_cols=["score_stageC3_topn_final"], ks=KS)
base_macro = base[base["chapter_id"] == "__ALL__"].iloc[0].to_dict()
print("Baseline ndcg@20 (StageC3 top50):", float(base_macro.get("ndcg@20")))

K_SELECT = 20
TOPM = 150
CAP_GRID = [1, 2, 3, 4, 5]

rows = []
for cap in CAP_GRID:
    df_v = df.copy()
    df_v["score_stageD_facet_cap_v1"] = pd.to_numeric(df_v["score_stageC3_topn_final"], errors="coerce").fillna(0.0)

    sims = []
    facet_cov = []
    for cid, g in df_v.groupby("chapter_id"):
        g_sorted = g.sort_values("score_stageC3_topn_final", ascending=False, kind="mergesort")
        pool = g_sorted.head(int(TOPM)).copy()
        chosen_idx = facet_cap_select(pool, k=int(K_SELECT), max_per_facet=int(cap))

        chosen_g = pool.loc[chosen_idx].copy()
        sims.append(mean_pairwise_sim(chosen_g))
        facet_cov.append(int(pd.to_numeric(chosen_g["facet_best_i"], errors="coerce").fillna(-1).nunique()))

        # Push chosen set to front, ordered by baseline relevance
        chosen_sorted = chosen_g.sort_values("score_stageC3_topn_final", ascending=False, kind="mergesort").index.tolist()
        for r, ix in enumerate(chosen_sorted):
            df_v.loc[ix, "score_stageD_facet_cap_v1"] = 2.0 - (r * 1e-6)

    res = evaluate_all(df_v, score_cols=["score_stageD_facet_cap_v1"], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "topm": int(TOPM),
        "k": int(K_SELECT),
        "max_per_facet": int(cap),
        "ndcg@20": float(macro.get("ndcg@20")),
        "p@20": float(macro.get("p@20")),
        "mrr_include": float(macro.get("mrr_include")),
        "auc_include": float(macro.get("auc_include")),
        "mean_pairwise_sim_top20": float(np.nanmean(sims)) if sims else float("nan"),
        "unique_facets_top20": float(np.nanmean(facet_cov)) if facet_cov else float("nan"),
    })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mean_pairwise_sim_top20"], ascending=[False, True]).reset_index(drop=True)
print("\nStage D facet-cap sweep:")
display(sweep_df)

run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageD_diversity_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

out_path = EXP_DIR / f"{run_id_use}_stageD_facet_cap_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageD_facet_cap_sweep_v1", scored_path=str(SCORED_PATH)).to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageD_facet_cap_sweep_v1 (RUN_MODE=label_adjudication)


In [29]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageD_facet_cap_loco_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage D LOCO validation: facet-cap selection (no API)
#
# Choose `max_per_facet` on 2 chapters, evaluate on the holdout.

from pathlib import Path
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = [
    "chapter_id",
    "merge_key",
    "final_label",
    "title",
    "abstract",
    "score_stageC_final",
    "score_llm_rerank_v1",
    "facet_best_i",
]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_topn_final(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"])
        df.loc[idx, "score_stageC3_topn_final"] = 1.0 + llm_n
    return df


def build_text(df_in: pd.DataFrame) -> pd.Series:
    t = df_in["title"].fillna("").astype(str)
    a = df_in["abstract"].fillna("").astype(str)
    return (t + ". " + a).str.strip()


def mean_pairwise_sim(g: pd.DataFrame) -> float:
    if len(g) <= 1:
        return float("nan")
    tfidf = TfidfVectorizer(max_features=20000)
    X = tfidf.fit_transform(build_text(g).tolist())
    S = cosine_similarity(X)
    vals = []
    for i in range(len(g)):
        for j in range(i + 1, len(g)):
            vals.append(float(S[i, j]))
    return float(np.mean(vals)) if vals else float("nan")


def facet_cap_select(pool: pd.DataFrame, k: int, max_per_facet: int) -> List[int]:
    counts = defaultdict(int)
    chosen = []
    for ix, row in pool.iterrows():
        v = pd.to_numeric(row.get("facet_best_i"), errors="coerce")
        f = int(-1 if pd.isna(v) else v)
        if counts[f] < int(max_per_facet):
            chosen.append(ix)
            counts[f] += 1
        if len(chosen) >= int(k):
            break
    if len(chosen) < int(k):
        for ix in pool.index:
            if ix in chosen:
                continue
            chosen.append(ix)
            if len(chosen) >= int(k):
                break
    return chosen


def apply_facet_cap(df_in: pd.DataFrame, *, topm: int, k_select: int, cap: int) -> Tuple[pd.DataFrame, float, float]:
    d = df_in.copy()
    d["score_stageD_facet_cap_v1"] = pd.to_numeric(d["score_stageC3_topn_final"], errors="coerce").fillna(0.0)

    sims = []
    covs = []
    for _, g in d.groupby("chapter_id"):
        g_sorted = g.sort_values("score_stageC3_topn_final", ascending=False, kind="mergesort")
        pool = g_sorted.head(int(topm)).copy()

        chosen_idx = facet_cap_select(pool, k=int(k_select), max_per_facet=int(cap))
        chosen_g = pool.loc[chosen_idx].copy()

        sims.append(mean_pairwise_sim(chosen_g))
        covs.append(float(pd.to_numeric(chosen_g["facet_best_i"], errors="coerce").fillna(-1).nunique()))

        chosen_sorted = chosen_g.sort_values("score_stageC3_topn_final", ascending=False, kind="mergesort").index.tolist()
        for r, ix in enumerate(chosen_sorted):
            d.loc[ix, "score_stageD_facet_cap_v1"] = 2.0 - (r * 1e-6)

    return d, float(np.nanmean(sims)) if sims else float("nan"), float(np.nanmean(covs)) if covs else float("nan")


df = add_stagec3_topn_final(df0, topn=50)
chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
print("Chapters:", chapters)

TOPM = 150
K_SELECT = 20
CAP_GRID = [1, 2, 3, 4, 5]

rows = []
EPS = 1e-12
for holdout in chapters:
    train = df[df["chapter_id"].astype(str) != holdout].copy()
    hold = df[df["chapter_id"].astype(str) == holdout].copy()

    base_hold_nd = float(evaluate_all(hold, score_cols=["score_stageC3_topn_final"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
    hold_sorted = hold.sort_values("score_stageC3_topn_final", ascending=False, kind="mergesort").head(int(K_SELECT)).copy()
    base_hold_sim = mean_pairwise_sim(hold_sorted)
    base_hold_cov = float(pd.to_numeric(hold_sorted["facet_best_i"], errors="coerce").fillna(-1).nunique())

    best = None
    best_train_nd = -1e18
    best_train_sim = float("inf")
    best_train_cov = -1e18

    for cap in CAP_GRID:
        t_scored, t_sim, t_cov = apply_facet_cap(train, topm=int(TOPM), k_select=int(K_SELECT), cap=int(cap))
        t_nd = float(evaluate_all(t_scored, score_cols=["score_stageD_facet_cap_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])

        if (t_nd > best_train_nd + EPS) or (abs(t_nd - best_train_nd) <= EPS and (t_sim < best_train_sim - EPS)) or (abs(t_nd - best_train_nd) <= EPS and abs(t_sim - best_train_sim) <= EPS and t_cov > best_train_cov + EPS):
            best_train_nd = float(t_nd)
            best_train_sim = float(t_sim)
            best_train_cov = float(t_cov)
            best = {"cap": int(cap)}

    h_scored, h_sim, h_cov = apply_facet_cap(hold, topm=int(TOPM), k_select=int(K_SELECT), cap=int(best["cap"]))
    hold_nd = float(evaluate_all(h_scored, score_cols=["score_stageD_facet_cap_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])

    rows.append({
        "holdout_chapter": holdout,
        "selected_max_per_facet": int(best["cap"]),
        "train_ndcg@20": float(best_train_nd),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_hold_nd),
        "holdout_delta_vs_base": float(hold_nd - base_hold_nd),
        "holdout_sim_top20": float(h_sim),
        "holdout_base_sim_top20": float(base_hold_sim),
        "holdout_sim_delta": float(h_sim - base_hold_sim),
        "holdout_unique_facets_top20": float(h_cov),
        "holdout_base_unique_facets_top20": float(base_hold_cov),
        "holdout_unique_facets_delta": float(h_cov - base_hold_cov),
    })

loco_df = pd.DataFrame(rows)
print("\nStage D facet-cap LOCO results:")
display(loco_df)
print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))
print("Avg LOCO holdout sim delta:", float(loco_df["holdout_sim_delta"].mean()))
print("Avg LOCO holdout unique facets delta:", float(loco_df["holdout_unique_facets_delta"].mean()))


run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageD_diversity_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

out_path = EXP_DIR / f"{run_id_use}_stageD_facet_cap_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageD_facet_cap_loco_v1", scored_path=str(SCORED_PATH), topm=int(TOPM), k_select=int(K_SELECT)).to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageD_facet_cap_loco_v1 (RUN_MODE=label_adjudication)


In [30]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_topn_tiebreak_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 sanity test (no API): deterministic tie-break within top-N shortlist
#
# Motivation:
# - Stage D (TF-IDF MMR) LOCO selected lambda=1.0, suggesting the observed gains might
#   be dominated by deterministic tie-breaking rather than diversity penalties.
# - This cell tests a simpler and more principled approach:
#     shortlist = top-50 by score_stageC_final
#     primary order = LLM score (normalized)
#     tie-breakers = score_stageC_final (and score_cite_norm if present)
#   and then assigns strict ranks (2.0 - r*1e-6) to avoid sort instability.

from pathlib import Path

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "score_stageC_final", "score_llm_rerank_v1"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_topn_final(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    df["score_stageC3_topn_final"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"])
        df.loc[idx, "score_stageC3_topn_final"] = 1.0 + llm_n
    return df


def add_stagec3_topn_tiebreak_v1(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()

    cite_ok = "score_cite_norm" in df.columns
    if not cite_ok:
        df["score_cite_norm"] = 0.0

    df["score_stageC3_topn_tiebreak_v1"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index

        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(df.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(df.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        top = pd.DataFrame({
            "ix": idx.to_list(),
            "llm_n": llm_n,
            "stagec": stagec,
            "cite": cite,
        })

        top = top.sort_values(["llm_n", "stagec", "cite", "ix"], ascending=[False, False, False, True], kind="mergesort")

        for r, ix in enumerate(top["ix"].tolist()):
            df.loc[ix, "score_stageC3_topn_tiebreak_v1"] = 2.0 - (r * 1e-6)

    if not cite_ok:
        df = df.drop(columns=["score_cite_norm"], errors="ignore")

    return df


df = add_stagec3_topn_final(df0, topn=50)
df = add_stagec3_topn_tiebreak_v1(df, topn=50)

res = evaluate_all(df, score_cols=["score_stageC3_topn_final", "score_stageC3_topn_tiebreak_v1"], ks=KS)
macro = res[res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
print("Macro:")
display(macro)

if set(macro["score_col"]) == {"score_stageC3_topn_final", "score_stageC3_topn_tiebreak_v1"}:
    m = macro.set_index("score_col")
    cols = [c for c in ["ndcg@20", "p@20", "mrr_include", "auc_include"] if c in m.columns]
    delta = (m.loc["score_stageC3_topn_tiebreak_v1", cols] - m.loc["score_stageC3_topn_final", cols]).to_frame("delta")
    print("Delta (tiebreak - baseline):")
    display(delta)


run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC3_rerank_analysis_v1",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

eval_path = EXP_DIR / f"{run_id_use}_stageC3_topn_tiebreak_v1_eval.csv"
res.assign(**meta_use, experiment="stageC3_topn_tiebreak_v1_eval", scored_path=str(SCORED_PATH)).to_csv(eval_path, index=False)
print("Saved:", eval_path)

runs_csv = EXP_DIR / "runs.csv"
best_macro = macro[macro["score_col"] == "score_stageC3_topn_tiebreak_v1"].copy()
if not best_macro.empty:
    best_macro = best_macro.assign(**meta_use, experiment="stageC3_topn_tiebreak_v1")
    if runs_csv.exists():
        header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
        for c in header:
            if c not in best_macro.columns:
                best_macro[c] = None
        best_macro = best_macro[header]
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended macro row to:", runs_csv)
    '''
    exec(_CODE, globals())


Skipping st_stageC3_topn_tiebreak_v1 (RUN_MODE=label_adjudication)


In [31]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageD_mmr_tfidf_sweep_v2 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage D (candidate diversity) sweep v2 (no API)
#
# Scientific intent:
# - We improved Stage C.3 with a deterministic tie-break (`score_stageC3_topn_tiebreak_v1`).
# - Re-test Stage D MMR on top of that improved baseline, because earlier Stage D gains
#   may have been partially driven by tie-breaking artifacts.
#
# This sweep:
# - baseline ranking: `score_stageC3_topn_tiebreak_v1`
# - selection pool: top-`topm` by that baseline
# - MMR uses relevance signal that matches production knowledge:
#     score_stageC3_signal_v1 = 1 + llm_norm (within top-50 by Stage C) else score_stageC_final
# - assigns chosen top-20 a strict rank score so evaluate_all is deterministic.

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "title", "abstract", "score_stageC_final", "score_llm_rerank_v1"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def build_text(df_in: pd.DataFrame) -> pd.Series:
    t = df_in["title"].fillna("").astype(str)
    a = df_in["abstract"].fillna("").astype(str)
    return (t + ". " + a).str.strip()


def add_stagec3_signals_v1(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    df["_in_stagec_topn"] = False
    df["_llm_n_topn"] = 0.0
    df["score_stageC3_signal_v1"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index
        df.loc[idx, "_in_stagec_topn"] = True
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"])
        df.loc[idx, "_llm_n_topn"] = llm_n
        df.loc[idx, "score_stageC3_signal_v1"] = 1.0 + llm_n

    return df


def add_stagec3_topn_tiebreak_v1(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()

    cite_ok = "score_cite_norm" in df.columns
    if not cite_ok:
        df["score_cite_norm"] = 0.0

    df["score_stageC3_topn_tiebreak_v1"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(df.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(df.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        top = pd.DataFrame({
            "ix": idx.to_list(),
            "llm_n": llm_n,
            "stagec": stagec,
            "cite": cite,
        })
        top = top.sort_values(["llm_n", "stagec", "cite", "ix"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            df.loc[ix, "score_stageC3_topn_tiebreak_v1"] = 2.0 - (r * 1e-6)

    if not cite_ok:
        df = df.drop(columns=["score_cite_norm"], errors="ignore")

    return df


def mean_pairwise_sim(S: np.ndarray, chosen: List[int]) -> float:
    if len(chosen) <= 1:
        return float("nan")
    vals = []
    for i in range(len(chosen)):
        for j in range(i + 1, len(chosen)):
            vals.append(float(S[chosen[i], chosen[j]]))
    return float(np.mean(vals)) if vals else float("nan")


def mmr_select(relevance: np.ndarray, sim: np.ndarray, k: int, lam: float) -> List[int]:
    n = int(len(relevance))
    if n == 0:
        return []
    k = int(min(k, n))

    chosen: List[int] = []
    remaining = list(range(n))

    first = int(np.argmax(relevance))
    chosen.append(first)
    remaining.remove(first)

    eps = 1e-12
    while len(chosen) < k and remaining:
        best_i = None
        best_score = -1e18
        for i in remaining:
            max_sim = max(float(sim[i, j]) for j in chosen) if chosen else 0.0
            s = float(lam) * float(relevance[i]) - (1.0 - float(lam)) * float(max_sim)
            if (s > best_score + eps) or (abs(s - best_score) <= eps and (best_i is None or int(i) < int(best_i))):
                best_score = s
                best_i = int(i)
        chosen.append(int(best_i))
        remaining.remove(int(best_i))

    return chosen


# -----------------------------
# Build baseline + signals
# -----------------------------
TOPN = 50
K_SELECT = 20

df = add_stagec3_signals_v1(df0, topn=TOPN)
df = add_stagec3_topn_tiebreak_v1(df, topn=TOPN)

base_res = evaluate_all(df, score_cols=["score_stageC3_topn_tiebreak_v1"], ks=KS)
base_macro = base_res.query("chapter_id=='__ALL__'").copy().reset_index(drop=True)
print("Baseline macro (Stage C.3 tiebreak):")
display(base_macro)

# Baseline redundancy proxy: mean pairwise sim in baseline top-20 per chapter
base_sims = []
for _, g in df.groupby("chapter_id"):
    top20 = g.sort_values("score_stageC3_topn_tiebreak_v1", ascending=False, kind="mergesort").head(int(K_SELECT)).copy()
    texts = build_text(top20).tolist()
    tfidf = TfidfVectorizer(max_features=20000)
    X = tfidf.fit_transform(texts)
    S = cosine_similarity(X)
    base_sims.append(mean_pairwise_sim(S, list(range(len(top20)))))

base_ndcg = float(base_macro.iloc[0]["ndcg@20"]) if len(base_macro) else float("nan")
base_sim = float(np.nanmean(base_sims)) if base_sims else float("nan")
print("Baseline ndcg@20:", base_ndcg)
print("Baseline mean_pairwise_sim_top20:", base_sim)


# -----------------------------
# Precompute pools per (chapter, topm)
# -----------------------------
TOPM_GRID = [50, 100, 150]
LAM_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
pre = {}
for cid in chapters:
    g = df[df["chapter_id"].astype(str) == cid].copy()
    g_sorted = g.sort_values("score_stageC3_topn_tiebreak_v1", ascending=False, kind="mergesort")
    for topm in TOPM_GRID:
        pool = g_sorted.head(int(topm)).copy()
        texts = build_text(pool).tolist()
        tfidf = TfidfVectorizer(max_features=20000)
        X = tfidf.fit_transform(texts)
        S = cosine_similarity(X)
        rel = pd.to_numeric(pool["score_stageC3_signal_v1"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        pre[(cid, int(topm))] = {"pool": pool, "S": S, "rel": rel}


# -----------------------------
# Sweep MMR params
# -----------------------------
rows = []
for topm in TOPM_GRID:
    for lam in LAM_GRID:
        df_v = df.copy()
        df_v["score_stageD_mmr_tfidf_v2"] = pd.to_numeric(df_v["score_stageC3_topn_tiebreak_v1"], errors="coerce").fillna(0.0)

        sims = []
        for cid in chapters:
            pack = pre[(cid, int(topm))]
            pool = pack["pool"]
            S = pack["S"]
            rel = pack["rel"]

            chosen_local = mmr_select(rel, S, k=int(K_SELECT), lam=float(lam))
            sims.append(mean_pairwise_sim(S, chosen_local))

            chosen_idx = pool.iloc[chosen_local].index.tolist()
            chosen_sorted = (
                pool.loc[chosen_idx]
                .sort_values(["score_stageC3_signal_v1", "score_stageC3_topn_tiebreak_v1"], ascending=[False, False], kind="mergesort")
                .index.tolist()
            )

            for r, ix in enumerate(chosen_sorted):
                df_v.loc[ix, "score_stageD_mmr_tfidf_v2"] = 3.0 - (r * 1e-6)

        res = evaluate_all(df_v, score_cols=["score_stageD_mmr_tfidf_v2"], ks=KS)
        macro = res.query("chapter_id=='__ALL__'").iloc[0].to_dict()
        rows.append({
            "topm": int(topm),
            "lambda": float(lam),
            "ndcg@20": float(macro.get("ndcg@20")),
            "p@20": float(macro.get("p@20")),
            "mrr_include": float(macro.get("mrr_include")),
            "auc_include": float(macro.get("auc_include")),
            "mean_pairwise_sim_top20": float(np.nanmean(sims)) if sims else float("nan"),
        })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mean_pairwise_sim_top20"], ascending=[False, True]).reset_index(drop=True)
print("\nStage D v2 MMR TF-IDF sweep (top 20 rows):")
display(sweep_df.head(20))


MAX_NDCG_DROP = 0.01
ok = sweep_df[sweep_df["ndcg@20"] >= (base_ndcg - MAX_NDCG_DROP)].copy()
if len(ok):
    ok = ok.sort_values(["mean_pairwise_sim_top20", "ndcg@20"], ascending=[True, False]).reset_index(drop=True)
    print(f"\nBest diversity with ndcg@20 drop <= {MAX_NDCG_DROP}:")
    display(ok.head(10))
else:
    print("\nNo settings met the ndcg constraint; consider relaxing MAX_NDCG_DROP if you want a diversity mode.")


# Persist sweep
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageD_diversity_v2",
        "experiment_notes": "auto-meta (run_id/meta missing)",
    }

out_path = EXP_DIR / f"{run_id_use}_stageD_mmr_tfidf_sweep_v2.csv"
sweep_df.assign(
    **meta_use,
    experiment="stageD_mmr_tfidf_sweep_v2",
    scored_path=str(SCORED_PATH),
    baseline_score_col="score_stageC3_topn_tiebreak_v1",
    k_select=int(K_SELECT),
    topn=int(TOPN),
).to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageD_mmr_tfidf_sweep_v2 (RUN_MODE=label_adjudication)


In [32]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageD_mmr_tfidf_loco_v2 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage D LOCO validation v2 (no API): TF‑IDF MMR on top of Stage C.3 tie-break baseline
#
# Scientific intent:
# - Validate a *diversity mode* selection that does NOT materially harm relevance.
# - Parameter selection rule (train-only):
#     keep ndcg@20 within NDCG_DROP_MAX of the baseline Stage C.3 tie-break,
#     then pick the setting with lowest redundancy proxy (mean TF‑IDF cosine sim in top‑K).

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "title", "abstract", "score_stageC_final", "score_llm_rerank_v1"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def build_text(df_in: pd.DataFrame) -> pd.Series:
    t = df_in["title"].fillna("").astype(str)
    a = df_in["abstract"].fillna("").astype(str)
    return (t + ". " + a).str.strip()


def mean_pairwise_sim(S: np.ndarray, chosen: List[int]) -> float:
    if len(chosen) <= 1:
        return float("nan")
    vals = []
    for i in range(len(chosen)):
        for j in range(i + 1, len(chosen)):
            vals.append(float(S[chosen[i], chosen[j]]))
    return float(np.mean(vals)) if vals else float("nan")


def topk_similarity(g: pd.DataFrame, score_col: str, k: int) -> float:
    g = g.sort_values(score_col, ascending=False, kind="mergesort").head(int(k)).copy()
    if len(g) <= 1:
        return float("nan")
    tfidf = TfidfVectorizer(max_features=20000)
    X = tfidf.fit_transform(build_text(g).tolist())
    S = cosine_similarity(X)
    return mean_pairwise_sim(S, list(range(len(g))))


def mmr_select(relevance: np.ndarray, sim: np.ndarray, k: int, lam: float) -> List[int]:
    n = int(len(relevance))
    if n == 0:
        return []
    k = int(min(k, n))

    chosen: List[int] = []
    remaining = list(range(n))

    first = int(np.argmax(relevance))
    chosen.append(first)
    remaining.remove(first)

    eps = 1e-12
    while len(chosen) < k and remaining:
        best_i = None
        best_score = -1e18
        for i in remaining:
            max_sim = max(float(sim[i, j]) for j in chosen) if chosen else 0.0
            s = float(lam) * float(relevance[i]) - (1.0 - float(lam)) * float(max_sim)
            if (s > best_score + eps) or (abs(s - best_score) <= eps and (best_i is None or int(i) < int(best_i))):
                best_score = s
                best_i = int(i)
        chosen.append(int(best_i))
        remaining.remove(int(best_i))
    return chosen


def add_stagec3_signals_v1(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    df["_in_stagec_topn"] = False
    df["_llm_n_topn"] = 0.0
    df["score_stageC3_signal_v1"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index
        df.loc[idx, "_in_stagec_topn"] = True
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"])
        df.loc[idx, "_llm_n_topn"] = llm_n
        df.loc[idx, "score_stageC3_signal_v1"] = 1.0 + llm_n

    return df


def add_stagec3_topn_tiebreak_v1(df_in: pd.DataFrame, topn: int = 50) -> pd.DataFrame:
    df = df_in.copy()
    cite_ok = "score_cite_norm" in df.columns
    if not cite_ok:
        df["score_cite_norm"] = 0.0

    df["score_stageC3_topn_tiebreak_v1"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)
    for cid, g in df.groupby("chapter_id"):
        idx = g.sort_values("score_stageC_final", ascending=False, kind="mergesort").head(int(topn)).index
        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(df.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(df.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        top = pd.DataFrame({"ix": idx.to_list(), "llm_n": llm_n, "stagec": stagec, "cite": cite})
        top = top.sort_values(["llm_n", "stagec", "cite", "ix"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            df.loc[ix, "score_stageC3_topn_tiebreak_v1"] = 2.0 - (r * 1e-6)

    if not cite_ok:
        df = df.drop(columns=["score_cite_norm"], errors="ignore")

    return df


def apply_stageD_mmr_tfidf_v2(df_in: pd.DataFrame, topm: int, lam: float, k_select: int = 20) -> Tuple[pd.DataFrame, float]:
    df_v = df_in.copy()
    df_v["score_stageD_mmr_tfidf_v2"] = pd.to_numeric(df_v["score_stageC3_topn_tiebreak_v1"], errors="coerce").fillna(0.0)
    sims = []

    for cid, g in df_v.groupby("chapter_id"):
        g = g.sort_values("score_stageC3_topn_tiebreak_v1", ascending=False, kind="mergesort")
        pool = g.head(int(topm)).copy()

        tfidf = TfidfVectorizer(max_features=20000)
        X = tfidf.fit_transform(build_text(pool).tolist())
        S = cosine_similarity(X)

        rel = pd.to_numeric(pool["score_stageC3_signal_v1"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        chosen_local = mmr_select(rel, S, k=int(k_select), lam=float(lam))
        chosen_idx = pool.iloc[chosen_local].index.tolist()

        chosen_sorted = (
            pool.loc[chosen_idx]
            .sort_values(["score_stageC3_signal_v1", "score_stageC3_topn_tiebreak_v1"], ascending=[False, False], kind="mergesort")
            .index.tolist()
        )
        for r, ix in enumerate(chosen_sorted):
            df_v.loc[ix, "score_stageD_mmr_tfidf_v2"] = 3.0 - (r * 1e-6)

        sims.append(topk_similarity(df_v[df_v["chapter_id"].astype(str) == str(cid)], "score_stageD_mmr_tfidf_v2", k=int(k_select)))

    return df_v, float(np.nanmean(sims)) if sims else float("nan")


def _macro_ndcg(df_in: pd.DataFrame, score_col: str) -> float:
    r = evaluate_all(df_in, score_cols=[score_col], ks=KS)
    return float(r.query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"]) 


TOPN = 50
K_SELECT = 20
TOPM_GRID = [50, 100, 150]
LAM_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
NDCG_DROP_MAX = 0.01

df = add_stagec3_signals_v1(df0, topn=int(TOPN))
df = add_stagec3_topn_tiebreak_v1(df, topn=int(TOPN))

chapters = sorted(df["chapter_id"].dropna().astype(str).unique().tolist())
print("Chapters:", chapters)
print("NDCG_DROP_MAX:", NDCG_DROP_MAX)

rows = []
EPS = 1e-12
for holdout in chapters:
    train = df[df["chapter_id"].astype(str) != holdout].copy()
    hold = df[df["chapter_id"].astype(str) == holdout].copy()

    base_train_nd = _macro_ndcg(train, "score_stageC3_topn_tiebreak_v1")
    base_hold_nd = _macro_ndcg(hold, "score_stageC3_topn_tiebreak_v1")
    base_hold_sim = topk_similarity(hold, "score_stageC3_topn_tiebreak_v1", k=int(K_SELECT))

    cand_rows = []
    for topm in TOPM_GRID:
        for lam in LAM_GRID:
            t_scored, t_sim = apply_stageD_mmr_tfidf_v2(train, topm=int(topm), lam=float(lam), k_select=int(K_SELECT))
            nd = _macro_ndcg(t_scored, "score_stageD_mmr_tfidf_v2")
            cand_rows.append({"topm": int(topm), "lambda": float(lam), "train_ndcg@20": float(nd), "train_sim_top20": float(t_sim)})

    cand = pd.DataFrame(cand_rows)
    thr = float(base_train_nd - float(NDCG_DROP_MAX))
    ok = cand[cand["train_ndcg@20"] >= thr].copy()
    if len(ok) == 0:
        ok = cand.copy()

    ok["_sim"] = pd.to_numeric(ok["train_sim_top20"], errors="coerce")
    ok.loc[ok["_sim"].isna(), "_sim"] = float("inf")
    ok = ok.sort_values(["_sim", "train_ndcg@20", "topm", "lambda"], ascending=[True, False, True, True]).reset_index(drop=True)
    best = ok.iloc[0].to_dict()

    h_scored, h_sim = apply_stageD_mmr_tfidf_v2(hold, topm=int(best["topm"]), lam=float(best["lambda"]), k_select=int(K_SELECT))
    hold_nd = _macro_ndcg(h_scored, "score_stageD_mmr_tfidf_v2")

    rows.append({
        "holdout_chapter": holdout,
        "selected_topm": int(best["topm"]),
        "selected_lambda": float(best["lambda"]),
        "train_ndcg@20": float(best["train_ndcg@20"]),
        "train_base_ndcg@20": float(base_train_nd),
        "train_ndcg_threshold": float(thr),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_hold_nd),
        "holdout_delta_vs_base": float(hold_nd - base_hold_nd),
        "holdout_sim_top20": float(h_sim),
        "holdout_base_sim_top20": float(base_hold_sim),
        "holdout_sim_delta": float(h_sim - base_hold_sim),
    })

loco_df = pd.DataFrame(rows)
print("\nStage D LOCO v2 results:")
display(loco_df)
print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))
print("Avg LOCO holdout sim delta:", float(loco_df["holdout_sim_delta"].mean()))


run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageD_diversity_v2",
        "experiment_notes": f"ndcg_drop_max={NDCG_DROP_MAX}",
    }

out_path = EXP_DIR / f"{run_id_use}_stageD_mmr_tfidf_loco_v2.csv"
loco_df.assign(**meta_use, experiment="stageD_mmr_tfidf_loco_v2", scored_path=str(SCORED_PATH), k_select=int(K_SELECT), topn=int(TOPN)).to_csv(out_path, index=False)
print("Saved:", out_path)
    '''
    exec(_CODE, globals())


Skipping st_stageD_mmr_tfidf_loco_v2 (RUN_MODE=label_adjudication)


In [33]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_topn_tradeoff_sweep_v2 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 top-N tradeoff sweep v2 (no API)
#
# Scientific intent:
# - We finalized a deterministic Stage C.3 tie-break within the Stage C shortlist.
# - Now tune `TOPN` for cost/quality: smaller TOPN = cheaper production runs.
# - This is fully offline because we already have per-doc LLM scores in the scored CSV.

from pathlib import Path

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 first.")

SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "score_stageC_final", "score_llm_rerank_v1"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def stagec3_topn_tiebreak(df_in: pd.DataFrame, topn: int) -> pd.DataFrame:
    df = df_in.copy()
    cite_ok = "score_cite_norm" in df.columns
    if not cite_ok:
        df["score_cite_norm"] = 0.0

    df["score_stageC3_topn_tiebreak_v1"] = pd.to_numeric(df["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in df.groupby("chapter_id"):
        # deterministic shortlist selection
        idx = g.sort_values(["score_stageC_final", "merge_key"], ascending=[False, True], kind="mergesort").head(int(topn)).index

        llm_n = minmax_series(df.loc[idx, "score_llm_rerank_v1"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(df.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(df.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        top = pd.DataFrame({"ix": idx.to_list(), "llm_n": llm_n, "stagec": stagec, "cite": cite})
        top = top.sort_values(["llm_n", "stagec", "cite", "ix"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            df.loc[ix, "score_stageC3_topn_tiebreak_v1"] = 2.0 - (r * 1e-6)

    if not cite_ok:
        df = df.drop(columns=["score_cite_norm"], errors="ignore")

    return df


TOPN_GRID = [10, 20, 30, 40, 50, 75, 100]
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
print("Chapters:", chapters)

rows = []
for topn in TOPN_GRID:
    df = stagec3_topn_tiebreak(df0, topn=int(topn))
    res = evaluate_all(df, score_cols=["score_stageC3_topn_tiebreak_v1"], ks=KS)
    macro = res.query("chapter_id=='__ALL__'").iloc[0].to_dict()
    rows.append({
        "topn": int(topn),
        "ndcg@20": float(macro.get("ndcg@20")),
        "p@20": float(macro.get("p@20")),
        "mrr_include": float(macro.get("mrr_include")),
        "auc_include": float(macro.get("auc_include")),
        "requests_per_chapter_est": int(topn),
    })

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
print("\nStage C.3 TOPN sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict()
print("Best topn by ndcg@20:", best)


# LOCO: choose topn on 2 chapters, evaluate on holdout
loco_rows = []
EPS = 1e-12
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    hold = df0[df0["chapter_id"].astype(str) == holdout].copy()

    best_topn = None
    best_train = -1e18
    for topn in TOPN_GRID:
        t = stagec3_topn_tiebreak(train, topn=int(topn))
        nd = float(evaluate_all(t, score_cols=["score_stageC3_topn_tiebreak_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
        if (nd > best_train + EPS) or (abs(nd - best_train) <= EPS and (best_topn is None or int(topn) < int(best_topn))):
            best_train = float(nd)
            best_topn = int(topn)

    h = stagec3_topn_tiebreak(hold, topn=int(best_topn))
    hold_nd = float(evaluate_all(h, score_cols=["score_stageC3_topn_tiebreak_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_topn": int(best_topn),
        "train_ndcg@20": float(best_train),
        "holdout_ndcg@20": float(hold_nd),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nStage C.3 TOPN LOCO results:")
display(loco_df)
print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))


# Persist sweep + LOCO
run_id_use = globals().get("run_id")
meta_use = globals().get("meta")
if run_id_use is None or meta_use is None:
    dataset_sha = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(SCORED_PATH),
        "dataset_sha1_12": dataset_sha,
        "git_head": "unknown",
        "confidence_min": CONFIDENCE_MIN,
        "experiment_tag": "stageC3_rerank_analysis_v2",
        "experiment_notes": "topn_tradeoff_sweep_v2",
    }

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_topn_tradeoff_sweep_v2.csv"
sweep_df.assign(**meta_use, experiment="stageC3_topn_tradeoff_sweep_v2", scored_path=str(SCORED_PATH)).to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_topn_tradeoff_loco_v2.csv"
loco_df.assign(**meta_use, experiment="stageC3_topn_tradeoff_loco_v2", scored_path=str(SCORED_PATH)).to_csv(loco_path, index=False)
print("Saved:", loco_path)
    '''
    exec(_CODE, globals())


Skipping st_stageC3_topn_tradeoff_sweep_v2 (RUN_MODE=label_adjudication)


In [34]:
if RUN_MODE != "rerank_topn":
    print(f"Skipping st_stageC3_rerank_topn_model_v2 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 model test v2 (API calls, but only for shortlist)
#
# Scientific intent:
# - Stage C.3 is now finalized as: rerank within Stage C top-50 + deterministic tie-break.
# - We confirmed TOPN=50 is optimal for relevance.
# - Now test whether a stronger model improves shortlist ordering (cost/quality tradeoff).
#
# This run only scores the Stage C top-50 per chapter (~150 calls total).

import os
import json
import time
import random
import asyncio
import hashlib
import textwrap
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field

from openai import OpenAI
from agents import Agent, Runner, ModelSettings

if "evaluate_all" not in globals():
    raise RuntimeError("Missing evaluate_all(). Run the notebook top-to-bottom so metrics utilities are loaded.")


# -----------------------------
# Settings
# -----------------------------
TOPN = 50
RERANK_MODEL = os.getenv("RERANK_TOPN_MODEL", "gpt-5-mini").strip() or "gpt-5-mini"
PROMPT_VERSION = "v2_topn_model"
FORCE_RERANK = False

CONCURRENCY = 8
MAX_RETRIES = 8
BACKOFF_INITIAL = 1.0
BACKOFF_MAX = 30.0
ABSTRACT_MAX_CHARS = 2000

MODEL_PRICES_USD_PER_1M = {
    "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
    "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
}


def _price_for_model(model: str) -> dict:
    return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})


def cost_from_usage(usage, model: str) -> dict:
    prices = _price_for_model(model)

    def req_cost(input_tokens, cached_tokens, output_tokens):
        cached_tokens = int(cached_tokens or 0)
        input_tokens = int(input_tokens or 0)
        output_tokens = int(output_tokens or 0)
        non_cached = max(0, input_tokens - cached_tokens)
        cost = (
            (non_cached / 1_000_000) * prices["input"]
            + (cached_tokens / 1_000_000) * prices["cached"]
            + (output_tokens / 1_000_000) * prices["output"]
        )
        return non_cached, cached_tokens, output_tokens, cost

    entries = getattr(usage, "request_usage_entries", None) or []
    if entries:
        total_in = total_cached = total_out = 0
        total_cost = 0.0
        for r in entries:
            inp = getattr(r, "input_tokens", 0) or 0
            out = getattr(r, "output_tokens", 0) or 0
            itd = getattr(r, "input_tokens_details", None)
            cached = getattr(itd, "cached_tokens", 0) if itd is not None else 0
            non_cached, cached, out, c = req_cost(inp, cached, out)
            total_in += non_cached
            total_cached += cached
            total_out += out
            total_cost += c
        return {
            "requests": int(getattr(usage, "requests", len(entries)) or len(entries)),
            "input_tokens": int(total_in + total_cached),
            "cached_input_tokens": int(total_cached),
            "output_tokens": int(total_out),
            "cost_usd": float(total_cost),
        }

    inp = int(getattr(usage, "input_tokens", 0) or 0)
    out = int(getattr(usage, "output_tokens", 0) or 0)
    itd = getattr(usage, "input_tokens_details", None)
    cached = int(getattr(itd, "cached_tokens", 0) if itd is not None else 0)

    non_cached, cached, out, c = req_cost(inp, cached, out)
    return {
        "requests": int(getattr(usage, "requests", 1) or 1),
        "input_tokens": int(non_cached + cached),
        "cached_input_tokens": int(cached),
        "output_tokens": int(out),
        "cost_usd": float(c),
    }


def _truncate(s: str, max_chars: int) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + "…"


# -----------------------------
# Load baseline scored CSV (contains Stage C final + v1 LLM score)
# -----------------------------
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using baseline scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "title", "abstract", "score_stageC_final", "score_llm_rerank_v1"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Baseline scored CSV missing required columns: {missing}")


# -----------------------------
# Rubrics + prompt
# -----------------------------
RUBRIC_DIR = Path("eval_dataset/eval_rubrics")
if not RUBRIC_DIR.exists():
    raise FileNotFoundError(f"Missing rubric dir: {RUBRIC_DIR}")

rubrics: Dict[str, dict] = {}
rubric_sigs: Dict[str, str] = {}
for rp in sorted(RUBRIC_DIR.glob("*.json")):
    cid = rp.stem
    r = json.loads(rp.read_text(encoding="utf-8"))
    rubrics[cid] = r
    payload = {
        "scope_statement": r.get("scope_statement"),
        "must_cover": r.get("must_cover", []),
        "must_avoid": r.get("must_avoid", []),
        "scoring_guidance": r.get("scoring_guidance"),
        "prompt_version": PROMPT_VERSION,
    }
    rubric_sigs[cid] = hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:8]


class RerankOut(BaseModel):
    score: int = Field(..., ge=0, le=100)
    notes: str = Field("", max_length=140)


rerank_agent = Agent(
    name=f"StageC3 Shortlist Rerank ({PROMPT_VERSION})",
    model=RERANK_MODEL,
    model_settings=ModelSettings(top_p=1.0, verbosity="low"),
    instructions=(
        "You score academic papers for inclusion in a specific thesis chapter. "
        "Use the rubric strictly. Return ONLY the structured output."
    ),
    output_type=RerankOut,
)


def build_prompt(rubric: dict, title: str, abstract: str) -> str:
    scope = rubric.get("scope_statement", "")
    must_cover = rubric.get("must_cover", [])
    must_avoid = rubric.get("must_avoid", [])
    guidance = rubric.get("scoring_guidance", "")

    lines = []
    lines.append("Score this paper for inclusion in the chapter.")
    lines.append("Return only the schema fields.")
    lines.append("")
    lines.append("RUBRIC")
    lines.append(f"SCOPE: {scope}")
    if guidance:
        lines.append(f"GUIDANCE: {guidance}")
    if must_cover:
        lines.append("MUST_COVER:")
        for b in must_cover:
            lines.append(f"- {b}")
    if must_avoid:
        lines.append("MUST_AVOID:")
        for b in must_avoid:
            lines.append(f"- {b}")

    lines.append("")
    lines.append("PAPER")
    lines.append(f"TITLE: {title}")
    if abstract:
        lines.append(f"ABSTRACT: {abstract}")
    else:
        lines.append("ABSTRACT: (missing)")
    lines.append("")
    lines.append("Scoring:")
    lines.append("- 100 = must cite for this chapter")
    lines.append("- 50 = partially relevant / might cite")
    lines.append("- 0 = irrelevant or off-scope")
    return "\n".join(lines)


# -----------------------------
# Select shortlist to score (top-N by Stage C)
# -----------------------------
short_idx = []
for cid, g in df0.groupby("chapter_id"):
    g = g.sort_values(["score_stageC_final", "merge_key"], ascending=[False, True], kind="mergesort")
    short_idx.extend(g.head(int(TOPN)).index.tolist())

df_short = df0.loc[short_idx].copy()
print("Shortlist docs:", len(df_short), "(expected ~", int(TOPN) * int(df0["chapter_id"].nunique()), ")")
print("Model:", RERANK_MODEL)


# -----------------------------
# Cache
# -----------------------------
CACHE_DIR = EXP_DIR / "llm_rerank_cache_topn_v2" / PINNED_DATASET_TAG
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_path(chapter_id: str, merge_key: str) -> Path:
    mk = str(merge_key)
    mk_h = hashlib.sha1(mk.encode("utf-8")).hexdigest()[:16]
    sig = rubric_sigs.get(str(chapter_id), "nosig")
    safe_model = RERANK_MODEL.replace("/", "_")
    d = CACHE_DIR / str(chapter_id)
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{mk_h}_{sig}_{PROMPT_VERSION}_{safe_model}.json"


def atomic_write_json(path: Path, obj: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


async def score_one(row: pd.Series) -> dict:
    cid = str(row["chapter_id"])
    mk = str(row["merge_key"])
    title = str(row.get("title", "") or "").strip()
    abstract = _truncate(str(row.get("abstract", "") or ""), ABSTRACT_MAX_CHARS)

    out_path = cache_path(cid, mk)
    if (not FORCE_RERANK) and out_path.exists():
        cached = json.loads(out_path.read_text(encoding="utf-8"))
        cached["_cached"] = True
        return cached

    rubric = rubrics.get(cid)
    if rubric is None:
        raise KeyError(f"Missing rubric for chapter_id={cid}")

    prompt = build_prompt(rubric, title=title, abstract=abstract)
    last_err: Optional[str] = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = await Runner.run(rerank_agent, prompt)
            out = res.final_output.model_dump()
            usage = res.context_wrapper.usage
            cost = cost_from_usage(usage, model=RERANK_MODEL)
            payload = {
                "chapter_id": cid,
                "merge_key": mk,
                **out,
                "_meta": {
                    "model": RERANK_MODEL,
                    "prompt_version": PROMPT_VERSION,
                    "rubric_sig": rubric_sigs.get(cid),
                    **cost,
                },
                "_cached": False,
            }
            atomic_write_json(out_path, payload)
            return payload
        except Exception as e:
            last_err = repr(e)
            backoff = min(BACKOFF_MAX, BACKOFF_INITIAL * (2 ** (attempt - 1)))
            backoff *= (1.0 + random.uniform(-0.15, 0.15))
            await asyncio.sleep(max(0.1, backoff))

    raise RuntimeError(f"TopN rerank failed after {MAX_RETRIES} retries for {cid} {mk}: {last_err}")


async def run_all() -> List[dict]:
    sem = asyncio.Semaphore(CONCURRENCY)
    rows = df_short.copy()

    async def _bound(row: pd.Series) -> dict:
        async with sem:
            return await score_one(row)

    tasks = [_bound(r) for _, r in rows.iterrows()]
    out: List[dict] = []
    for fut in asyncio.as_completed(tasks):
        out.append(await fut)
        if len(out) % 25 == 0:
            print(f"Scored {len(out)}/{len(tasks)}")
    return out


t0 = time.time()
client = OpenAI()  # ensure API client init
print("Starting shortlist rerank… model=", RERANK_MODEL, "docs=", len(df_short))
results = await run_all()
seconds = time.time() - t0

scores = pd.DataFrame([
    {
        "chapter_id": r.get("chapter_id"),
        "merge_key": r.get("merge_key"),
        "llm_score_topn_v2": float(r.get("score", 0.0)),
        "llm_notes_topn_v2": r.get("notes", ""),
        "llm_cached": bool(r.get("_cached", False)),
        "cost_usd": 0.0 if bool(r.get("_cached", False)) else float((r.get("_meta") or {}).get("cost_usd", 0.0)),
        "input_tokens": 0 if bool(r.get("_cached", False)) else int((r.get("_meta") or {}).get("input_tokens", 0)),
        "cached_input_tokens": 0 if bool(r.get("_cached", False)) else int((r.get("_meta") or {}).get("cached_input_tokens", 0)),
        "output_tokens": 0 if bool(r.get("_cached", False)) else int((r.get("_meta") or {}).get("output_tokens", 0)),
        "requests": 0 if bool(r.get("_cached", False)) else int((r.get("_meta") or {}).get("requests", 1)),
        "est_cost_usd": float((r.get("_meta") or {}).get("cost_usd", 0.0)),
        "est_input_tokens": int((r.get("_meta") or {}).get("input_tokens", 0)),
        "est_cached_input_tokens": int((r.get("_meta") or {}).get("cached_input_tokens", 0)),
        "est_output_tokens": int((r.get("_meta") or {}).get("output_tokens", 0)),
        "est_requests": int((r.get("_meta") or {}).get("requests", 1)),
    }
    for r in results
])

totals = {
    "seconds": float(seconds),
    "requests": float(scores["requests"].sum()),
    "input_tokens": float(scores["input_tokens"].sum()),
    "cached_input_tokens": float(scores["cached_input_tokens"].sum()),
    "output_tokens": float(scores["output_tokens"].sum()),
    "cost_usd": float(scores["cost_usd"].sum()),
    "cached_files": int(scores["llm_cached"].sum()),
}
totals_est = {
    "requests": float(scores["est_requests"].sum()),
    "input_tokens": float(scores["est_input_tokens"].sum()),
    "cached_input_tokens": float(scores["est_cached_input_tokens"].sum()),
    "output_tokens": float(scores["est_output_tokens"].sum()),
    "cost_usd": float(scores["est_cost_usd"].sum()),
}
print("Rerank totals (this run):", totals)
print("Rerank totals (estimated total incl cached):", totals_est)

df_r = df0.merge(scores[["chapter_id", "merge_key", "llm_score_topn_v2", "llm_notes_topn_v2"]], on=["chapter_id", "merge_key"], how="left")
df_r["score_llm_rerank_topn_v2"] = pd.to_numeric(df_r["llm_score_topn_v2"], errors="coerce").fillna(0.0) / 100.0


def _minmax(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def add_stagec3_topn_tiebreak(df_in: pd.DataFrame, *, llm_col: str, out_col: str, topn: int = 50) -> pd.DataFrame:
    d = df_in.copy()

    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0

    d[out_col] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values(["score_stageC_final", "merge_key"], ascending=[False, True], kind="mergesort").head(int(topn)).index

        llm_n = _minmax(d.loc[idx, llm_col]).to_numpy(dtype=float)
        stagec = pd.to_numeric(d.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({"ix": idx.to_list(), "llm_n": llm_n, "stagec": stagec, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "stagec", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")

        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)

    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")

    return d


# Baseline (existing v1 score) vs new model
df_r = add_stagec3_topn_tiebreak(df_r, llm_col="score_llm_rerank_v1", out_col="score_stageC3_topn_tiebreak_v1", topn=int(TOPN))
df_r = add_stagec3_topn_tiebreak(df_r, llm_col="score_llm_rerank_topn_v2", out_col="score_stageC3_topn_tiebreak_v2", topn=int(TOPN))

res = evaluate_all(df_r, score_cols=["score_stageC3_topn_tiebreak_v1", "score_stageC3_topn_tiebreak_v2"], ks=KS)
macro = res.query("chapter_id=='__ALL__'").copy().reset_index(drop=True)
print("Macro:")
display(macro)

if set(macro["score_col"]) == {"score_stageC3_topn_tiebreak_v1", "score_stageC3_topn_tiebreak_v2"}:
    m = macro.set_index("score_col")
    cols = [c for c in ["ndcg@20", "p@20", "mrr_include", "auc_include"] if c in m.columns]
    delta = (m.loc["score_stageC3_topn_tiebreak_v2", cols] - m.loc["score_stageC3_topn_tiebreak_v1", cols]).to_frame("delta")
    print("Delta (v2 - v1):")
    display(delta)


# Persist
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_rerank_topn_v2",
    "experiment_notes": f"pinned_dataset_tag={PINNED_DATASET_TAG}; model={RERANK_MODEL}; topn={TOPN}",
}

scored_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_topn_model_v2_scored.csv"
df_r.assign(**meta_use, rerank_model=RERANK_MODEL, prompt_version=PROMPT_VERSION, topn=int(TOPN)).to_csv(scored_path, index=False)
print("Saved:", scored_path)

eval_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_topn_model_v2_eval.csv"
res.assign(**meta_use, experiment="stageC3_rerank_topn_model_v2_eval", rerank_model=RERANK_MODEL, prompt_version=PROMPT_VERSION, topn=int(TOPN)).to_csv(eval_path, index=False)
print("Saved:", eval_path)

cost_path = EXP_DIR / f"{run_id_use}_stageC3_rerank_topn_model_v2_cost.csv"
cost_row = {
    **totals,
    **{f"est_{k}": v for k, v in totals_est.items()},
}
pd.DataFrame([cost_row]).assign(**meta_use, experiment="stageC3_rerank_topn_model_v2_cost", rerank_model=RERANK_MODEL, prompt_version=PROMPT_VERSION, topn=int(TOPN)).to_csv(cost_path, index=False)
print("Saved:", cost_path)

# Append macro row to runs.csv
runs_csv = EXP_DIR / "runs.csv"
best_macro = macro[macro["score_col"] == "score_stageC3_topn_tiebreak_v2"].copy()
if not best_macro.empty:
    best_macro = best_macro.assign(**meta_use, experiment="stageC3_rerank_topn_model_v2")
    if runs_csv.exists():
        header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
        for c in header:
            if c not in best_macro.columns:
                best_macro[c] = None
        best_macro = best_macro[header]
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended macro row to:", runs_csv)
    '''
    import textwrap
    _wrapped = "async def _run_stageC3_rerank_topn_model_v2():\n" + textwrap.indent(_CODE, "    ")
    exec(_wrapped, globals())
    await _run_stageC3_rerank_topn_model_v2()


Skipping st_stageC3_rerank_topn_model_v2 (RUN_MODE=label_adjudication)


In [35]:
if RUN_MODE not in ("rerank_topn", "rerank_topn_analysis"):
    print(f"Skipping st_stageC3_rerank_topn_model_v2_analysis (RUN_MODE={RUN_MODE})")
else:
    from pathlib import Path
    import numpy as np
    import pandas as pd

    EXP_DIR = Path("eval_dataset/experiments")
    cands = sorted(EXP_DIR.glob("*_stageC3_rerank_topn_model_v2_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not cands:
        raise FileNotFoundError("No *_stageC3_rerank_topn_model_v2_scored.csv found. Run st_stageC3_rerank_topn_model_v2 first.")
    SCORED_PATH = cands[0]
    print("Using scored CSV:", SCORED_PATH)
    df = pd.read_csv(SCORED_PATH)

    need = [
        "chapter_id", "merge_key", "final_label", "title",
        "score_stageC3_topn_tiebreak_v1", "score_stageC3_topn_tiebreak_v2",
    ]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise RuntimeError(f"Scored CSV missing required columns: {missing}")

    score_cols = ["score_stageC3_topn_tiebreak_v1", "score_stageC3_topn_tiebreak_v2"]
    res = evaluate_all(df, score_cols=score_cols, ks=KS)
    per = res[res["chapter_id"] != "__ALL__"].copy()
    macro = res[res["chapter_id"] == "__ALL__"].copy()

    print("Per-chapter metrics:")
    display(per.sort_values(["chapter_id", "score_col"], ascending=True).reset_index(drop=True))
    print("Macro:")
    display(macro.reset_index(drop=True))

    # Delta per chapter (v2 - v1) for key metrics
    key_metrics = [c for c in ["ndcg@20", "p@20", "mrr_include", "auc_include"] if c in per.columns]
    if set(per["score_col"].unique().tolist()) >= set(score_cols):
        pvt = per.pivot(index="chapter_id", columns="score_col", values=key_metrics)
        # flatten columns
        pvt.columns = [f"{m}|{sc}" for m, sc in pvt.columns]
        out = pd.DataFrame(index=pvt.index)
        for m in key_metrics:
            out[m] = pvt[f"{m}|score_stageC3_topn_tiebreak_v2"] - pvt[f"{m}|score_stageC3_topn_tiebreak_v1"]
        print("Per-chapter delta (v2 - v1):")
        display(out.reset_index())

    def first_label_rank(df_in: pd.DataFrame, cid: str, score_col: str, label: str) -> int | None:
        g = df_in[df_in["chapter_id"].astype(str) == str(cid)].copy()
        g["_score"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
        g = g.sort_values(["_score", "merge_key"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
        hits = g.index[g["final_label"].astype(str) == str(label)].to_numpy()
        return int(hits[0] + 1) if len(hits) else None

    rows = []
    for cid in sorted(df["chapter_id"].dropna().astype(str).unique().tolist()):
        rows.append({
            "chapter_id": cid,
            "first_include_rank_v1": first_label_rank(df, cid, "score_stageC3_topn_tiebreak_v1", "include"),
            "first_include_rank_v2": first_label_rank(df, cid, "score_stageC3_topn_tiebreak_v2", "include"),
            "first_maybe_rank_v1": first_label_rank(df, cid, "score_stageC3_topn_tiebreak_v1", "maybe"),
            "first_maybe_rank_v2": first_label_rank(df, cid, "score_stageC3_topn_tiebreak_v2", "maybe"),
        })
    print("First relevant positions (lower is better):")
    display(pd.DataFrame(rows))


Skipping st_stageC3_rerank_topn_model_v2_analysis (RUN_MODE=label_adjudication)


In [36]:
if RUN_MODE not in ("rerank_analysis", "sweeps"):
    print(f"Skipping st_stageC3_stageC_joint_grid_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3: tune Stage C weights for downstream Stage C.3 (offline; no API)
#
# Scientific intent:
# - Our Stage C weights were originally tuned for Stage C alone.
# - But production ranking is Stage C + Stage C.3 (LLM rerank inside top-50).
# - This grid search re-tunes (w_embed_max, w_embed, cite_weight) to maximize
#   final ranking quality AFTER applying the Stage C.3 deterministic tie-break,
#   using the cached full LLM scores from *_stageC3_rerank_v1_scored.csv.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

if "evaluate_all" not in globals():
    raise RuntimeError("Missing evaluate_all(). Run the notebook top-to-bottom so metrics utilities are loaded.")

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

REQUIRED = [
    "chapter_id", "merge_key", "final_label",
    "score_embed_max", "score_embed_mean_top3", "score_tfidf", "score_cite_norm",
    "score_llm_rerank_v1",
]
missing = [c for c in REQUIRED if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

# Baseline (current finalized Stage C weights)
BASE = {"w_embed_max": 0.0, "w_embed": 0.7, "cite_weight": 0.08}
TOPN = 50

# Grid (small but covers interactions)
W_EMBED_MAX_GRID = [0.0, 0.2, 0.5, 0.8, 1.0]
W_EMBED_GRID = [round(float(x), 2) for x in np.linspace(0, 1, 11)]
CITE_WEIGHT_GRID = [0.0, 0.01, 0.02, 0.05, 0.08, 0.12]

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)

def score_stagec(df_in: pd.DataFrame, w_embed_max: float, w_embed: float, cite_weight: float) -> pd.Series:
    emb_max = pd.to_numeric(df_in["score_embed_max"], errors="coerce").fillna(0.0)
    emb_b = pd.to_numeric(df_in["score_embed_mean_top3"], errors="coerce").fillna(0.0)
    tfidf = pd.to_numeric(df_in["score_tfidf"], errors="coerce").fillna(0.0)
    cite = pd.to_numeric(df_in["score_cite_norm"], errors="coerce").fillna(0.0)

    d = df_in[["chapter_id"]].copy()
    d["_emb_raw"] = float(w_embed_max) * emb_max + (1.0 - float(w_embed_max)) * emb_b
    d["_emb_n"] = minmax_by_group(d.assign(score=d["_emb_raw"]), "score")
    d["_tf_n"] = minmax_by_group(df_in.assign(score=tfidf), "score")
    d["_cite_n"] = minmax_by_group(df_in.assign(score=cite), "score")

    base = float(w_embed) * d["_emb_n"] + (1.0 - float(w_embed)) * d["_tf_n"]
    stagec = (1.0 - float(cite_weight)) * base + float(cite_weight) * d["_cite_n"]
    return stagec

def add_stagec3_topn_tiebreak(df_in: pd.DataFrame, *, stagec_col: str, llm_col: str, out_col: str, topn: int = 50) -> pd.DataFrame:
    d = df_in.copy()
    d[out_col] = pd.to_numeric(d[stagec_col], errors="coerce").fillna(0.0)

    for cid, g in d.groupby("chapter_id"):
        cid = str(cid)
        idx = g.sort_values([stagec_col, "merge_key"], ascending=[False, True], kind="mergesort").head(int(topn)).index
        llm_n = minmax_series(d.loc[idx, llm_col]).to_numpy(dtype=float)
        stagec = pd.to_numeric(d.loc[idx, stagec_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({"ix": idx.to_list(), "llm_n": llm_n, "stagec": stagec, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "stagec", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)
    return d

def eval_combo(df_in: pd.DataFrame, w_embed_max: float, w_embed: float, cite_weight: float) -> dict:
    d = df_in.copy()
    d["score_stageC_tmp"] = score_stagec(d, w_embed_max=w_embed_max, w_embed=w_embed, cite_weight=cite_weight)
    d = add_stagec3_topn_tiebreak(d, stagec_col="score_stageC_tmp", llm_col="score_llm_rerank_v1", out_col="score_stageC3_tmp", topn=int(TOPN))
    res = evaluate_all(d, score_cols=["score_stageC3_tmp"], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    return {
        "w_embed_max": float(w_embed_max),
        "w_embed": float(w_embed),
        "cite_weight": float(cite_weight),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    }

# Baseline reference
base_row = eval_combo(df0, **BASE)
print("Baseline (current) after Stage C.3:", base_row)

# -----------------------------
# Sweep (macro)
# -----------------------------
rows = []
for wmax in W_EMBED_MAX_GRID:
    for w in W_EMBED_GRID:
        for cw in CITE_WEIGHT_GRID:
            rows.append(eval_combo(df0, w_embed_max=wmax, w_embed=w, cite_weight=cw))

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("\nStage C weights → Stage C.3 macro sweep (top 25):")
display(sweep_df.head(25))
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best by ndcg@20:", {k: best[k] for k in ["w_embed_max", "w_embed", "cite_weight", "ndcg@20", "mrr_include", "p@20", "auc_include"] if k in best})

# -----------------------------
# LOCO (choose params on 2 chapters, evaluate on 3rd)
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    # baseline holdout
    dbase = test.copy()
    dbase["score_stageC_tmp"] = score_stagec(dbase, **BASE)
    dbase = add_stagec3_topn_tiebreak(dbase, stagec_col="score_stageC_tmp", llm_col="score_llm_rerank_v1", out_col="score_stageC3_tmp", topn=int(TOPN))
    base_hold = evaluate_all(dbase, score_cols=["score_stageC3_tmp"], ks=KS)
    base_hold_nd = float(base_hold[base_hold["chapter_id"] == holdout].iloc[0]["ndcg@20"])
    base_hold_mrr = float(base_hold[base_hold["chapter_id"] == holdout].iloc[0]["mrr_include"])

    best_train = None
    best_train_nd = -1e9
    for wmax in W_EMBED_MAX_GRID:
        for w in W_EMBED_GRID:
            for cw in CITE_WEIGHT_GRID:
                dtr = train.copy()
                dtr["score_stageC_tmp"] = score_stagec(dtr, w_embed_max=wmax, w_embed=w, cite_weight=cw)
                dtr = add_stagec3_topn_tiebreak(dtr, stagec_col="score_stageC_tmp", llm_col="score_llm_rerank_v1", out_col="score_stageC3_tmp", topn=int(TOPN))
                r = evaluate_all(dtr, score_cols=["score_stageC3_tmp"], ks=KS)
                nd = float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])
                if nd > best_train_nd:
                    best_train_nd = nd
                    best_train = {"w_embed_max": float(wmax), "w_embed": float(w), "cite_weight": float(cw)}

    dte = test.copy()
    dte["score_stageC_tmp"] = score_stagec(dte, **best_train)
    dte = add_stagec3_topn_tiebreak(dte, stagec_col="score_stageC_tmp", llm_col="score_llm_rerank_v1", out_col="score_stageC3_tmp", topn=int(TOPN))
    r2 = evaluate_all(dte, score_cols=["score_stageC3_tmp"], ks=KS)
    hold = r2[r2["chapter_id"] == holdout].iloc[0].to_dict()

    loco_rows.append({
        "holdout_chapter": holdout,
        **best_train,
        "train_ndcg@20": float(best_train_nd),
        "holdout_ndcg@20": float(hold.get("ndcg@20", float("nan"))),
        "holdout_mrr_include": float(hold.get("mrr_include", float("nan"))),
        "holdout_base_ndcg@20": float(base_hold_nd),
        "holdout_base_mrr_include": float(base_hold_mrr),
        "holdout_delta_ndcg@20": float(hold.get("ndcg@20", float("nan")) - base_hold_nd),
        "holdout_delta_mrr_include": float(hold.get("mrr_include", float("nan")) - base_hold_mrr),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO results (Stage C tuned for Stage C.3):")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta ndcg@20 vs baseline:", float(loco_df["holdout_delta_ndcg@20"].mean()))
    print("Avg LOCO holdout delta mrr_include vs baseline:", float(loco_df["holdout_delta_mrr_include"].mean()))

# Persist artifacts
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_stageC_joint_grid_v1",
    "experiment_notes": f"TOPN={TOPN}; baseline={BASE}; uses score_llm_rerank_v1"
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_stageC_joint_grid_v1_sweep.csv"
sweep_df.assign(**meta_use, experiment="stageC3_stageC_joint_grid_v1_sweep").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_stageC_joint_grid_v1_loco.csv"
loco_df.assign(**meta_use, experiment="stageC3_stageC_joint_grid_v1_loco").to_csv(loco_path, index=False)
print("Saved:", loco_path)

runs_csv = EXP_DIR / "runs.csv"
best_macro = None
try:
    # best macro row from sweep
    best_macro = pd.DataFrame([best]).assign(**meta_use, experiment="stageC3_stageC_joint_grid_v1_best")
except Exception:
    best_macro = None
if best_macro is not None and not best_macro.empty:
    if runs_csv.exists():
        header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
        for c in header:
            if c not in best_macro.columns:
                best_macro[c] = None
        best_macro = best_macro[header]
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended best macro row to:", runs_csv)

    
    
'''
    exec(_CODE, globals())


Skipping st_stageC3_stageC_joint_grid_v1 (RUN_MODE=label_adjudication)


In [37]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_llm_bucket_sweep_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3: LLM-score quantization sweep (no API calls)
#
# Scientific intent:
# - LLM rerank outputs integer 0–100 scores; small differences can be noise.
# - Instead of fully ordering by raw LLM score, bucketize it (e.g. 5-point bins),
#   then use Stage C score/citations as tie-breakers within bins.
# - This can improve stability and reduce overfitting to LLM noise.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "score_stageC_final", "llm_score"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

TOPN = 50
BUCKETS = [1, 2, 5, 10]  # LLM score bucket size (points)

def add_stagec3_bucketed(df_in: pd.DataFrame, bucket_size: int, out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0

    d[out_col] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values(["score_stageC_final", "merge_key"], ascending=[False, True], kind="mergesort").head(int(TOPN)).index

        llm = pd.to_numeric(d.loc[idx, "llm_score"], errors="coerce").fillna(0.0)
        bucket = np.floor(llm.to_numpy(dtype=float) / float(bucket_size)).astype(int)
        stagec = pd.to_numeric(d.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({
            "ix": idx.to_list(),
            "bucket": bucket,
            "llm": llm.to_numpy(dtype=float),
            "stagec": stagec,
            "cite": cite,
            "merge_key": mk,
        })
        top = top.sort_values(["bucket", "llm", "stagec", "cite", "merge_key"], ascending=[False, False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)

    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d

# -----------------------------
# Macro sweep
# -----------------------------
rows = []
for b in BUCKETS:
    out_col = f"score_stageC3_bucket_b{int(b)}"
    d = add_stagec3_bucketed(df0, bucket_size=int(b), out_col=out_col)
    r = evaluate_all(d, score_cols=[out_col], ks=KS)
    macro = r[r["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "bucket": int(b),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("Bucket sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best bucket by ndcg@20:", best)

# -----------------------------
# LOCO
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    best_b = None
    best_train = -1e9
    for b in BUCKETS:
        dtr = add_stagec3_bucketed(train, bucket_size=int(b), out_col="score_stageC3_tmp")
        nd = float(evaluate_all(dtr, score_cols=["score_stageC3_tmp"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
        if nd > best_train:
            best_train = nd
            best_b = int(b)

    dte = add_stagec3_bucketed(test, bucket_size=int(best_b), out_col="score_stageC3_tmp")
    hold_nd = float(evaluate_all(dte, score_cols=["score_stageC3_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    # Base = bucket=1
    dbase = add_stagec3_bucketed(test, bucket_size=1, out_col="score_stageC3_base")
    base_nd = float(evaluate_all(dbase, score_cols=["score_stageC3_base"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_bucket": int(best_b),
        "train_ndcg@20": float(best_train),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_nd),
        "holdout_delta_vs_base": float(hold_nd - base_nd),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO (bucket size) results:")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta vs base (bucket=1):", float(loco_df["holdout_delta_vs_base"].mean()))

# -----------------------------
# Persist artifacts
# -----------------------------
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_llm_bucket_sweep_v1",
    "experiment_notes": f"TOPN={TOPN}; BUCKETS={BUCKETS}; uses llm_score + score_stageC_final",
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_llm_bucket_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageC3_llm_bucket_sweep_v1").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_llm_bucket_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageC3_llm_bucket_loco_v1").to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv
best_bucket = int(best.get("bucket", 1)) if best else 1
d_best = add_stagec3_bucketed(df0, bucket_size=int(best_bucket), out_col="score_stageC3_bucket_best")
r_best = evaluate_all(d_best, score_cols=["score_stageC3_bucket_best"], ks=KS)
macro_best = r_best[r_best["chapter_id"] == "__ALL__"].copy()
macro_best = macro_best.assign(**meta_use, experiment="stageC3_llm_bucket_best_v1", bucket_size=int(best_bucket))
runs_csv = EXP_DIR / "runs.csv"
if runs_csv.exists():
    header = pd.read_csv(runs_csv, nrows=0).columns.tolist()
    for c in header:
        if c not in macro_best.columns:
            macro_best[c] = None
    macro_best = macro_best[header]
    macro_best.to_csv(runs_csv, mode="a", header=False, index=False)
else:
    macro_best.to_csv(runs_csv, index=False)
print("Appended macro row to:", runs_csv)

'''
    exec(_CODE, globals())


Skipping st_stageC3_llm_bucket_sweep_v1 (RUN_MODE=label_adjudication)


In [38]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_llm_bucket_sweep_v2 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3: LLM-score bucketization sweep v2 (no API calls)
#
# Fix vs v1:
# - v1 still sorted by raw llm score inside the bucket, so bucket size had no effect.
# - v2 uses ONLY the bucket id as the LLM signal, and breaks ties with Stage C / citations.
#
# Interpretation:
# - Larger buckets = trust LLM only coarsely; rely more on Stage C within buckets.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "score_stageC_final", "llm_score"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

TOPN = 50
BUCKETS = [1, 2, 5, 10, 20]  # LLM score bucket size (points)

def add_stagec3_bucketed_v2(df_in: pd.DataFrame, bucket_size: int, out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0

    d[out_col] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values(["score_stageC_final", "merge_key"], ascending=[False, True], kind="mergesort").head(int(TOPN)).index

        llm = pd.to_numeric(d.loc[idx, "llm_score"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        bucket = np.floor(llm / float(bucket_size)).astype(int)
        stagec = pd.to_numeric(d.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({
            "ix": idx.to_list(),
            "bucket": bucket,
            "stagec": stagec,
            "cite": cite,
            "merge_key": mk,
        })
        top = top.sort_values(["bucket", "stagec", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)

    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d

# -----------------------------
# Macro sweep
# -----------------------------
rows = []
for b in BUCKETS:
    out_col = f"score_stageC3_bucket_v2_b{int(b)}"
    d = add_stagec3_bucketed_v2(df0, bucket_size=int(b), out_col=out_col)
    r = evaluate_all(d, score_cols=[out_col], ks=KS)
    macro = r[r["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "bucket": int(b),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
    })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("Bucket sweep v2 (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best bucket v2 by ndcg@20:", best)

# -----------------------------
# LOCO
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    best_b = None
    best_train = -1e9
    for b in BUCKETS:
        dtr = add_stagec3_bucketed_v2(train, bucket_size=int(b), out_col="score_stageC3_tmp")
        nd = float(evaluate_all(dtr, score_cols=["score_stageC3_tmp"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
        if nd > best_train:
            best_train = nd
            best_b = int(b)

    dte = add_stagec3_bucketed_v2(test, bucket_size=int(best_b), out_col="score_stageC3_tmp")
    hold_nd = float(evaluate_all(dte, score_cols=["score_stageC3_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    # Base = bucket=1 (should match production tie-break behavior)
    dbase = add_stagec3_bucketed_v2(test, bucket_size=1, out_col="score_stageC3_base")
    base_nd = float(evaluate_all(dbase, score_cols=["score_stageC3_base"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_bucket": int(best_b),
        "train_ndcg@20": float(best_train),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_nd),
        "holdout_delta_vs_base": float(hold_nd - base_nd),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO (bucket size v2) results:")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta vs base (bucket=1):", float(loco_df["holdout_delta_vs_base"].mean()))

# Persist artifacts
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_llm_bucket_sweep_v2",
    "experiment_notes": f"TOPN={TOPN}; BUCKETS={BUCKETS}; v2 uses bucket only + StageC/cite tie-break; uses llm_score + score_stageC_final",
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_llm_bucket_sweep_v2.csv"
sweep_df.assign(**meta_use, experiment="stageC3_llm_bucket_sweep_v2").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_llm_bucket_loco_v2.csv"
loco_df.assign(**meta_use, experiment="stageC3_llm_bucket_loco_v2").to_csv(loco_path, index=False)
print("Saved:", loco_path)

'''
    exec(_CODE, globals())


Skipping st_stageC3_llm_bucket_sweep_v2 (RUN_MODE=label_adjudication)


In [39]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_stageC_norm_rank_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C / C.3: normalization experiment (no API calls)
#
# Hypothesis:
# - Per-chapter minmax normalization can be sensitive to outliers.
# - Rank/percentile normalization may generalize better across chapters.
#
# Test:
# - Recompute Stage C score using rank-normalized components (emb/tfidf/cite)
#   with the same finalized weights.
# - Apply the finalized Stage C.3 top-50 deterministic rerank using existing llm_score.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = [
    "chapter_id", "merge_key", "final_label",
    "score_embed_max", "score_embed_mean_top3", "score_tfidf", "score_cite_norm",
    "score_stageC_final", "llm_score",
]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

# Finalized Stage C weights (keep fixed)
W_EMBED_MAX = 0.0
W_EMBED = 0.7
CITE_WEIGHT = 0.08
TOPN = 50

def ranknorm_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0.0)
        # rank 1..n, higher score => higher rank; convert to [0,1]
        r = x.rank(method="average", ascending=True)
        n = float(len(r))
        out.loc[g.index] = ((r - 1.0) / (n - 1.0 + 1e-12)).to_numpy(dtype=float)
    return out

def add_stagec3_topn_tiebreak(df_in: pd.DataFrame, *, stagec_col: str, out_col: str, topn: int = 50) -> pd.DataFrame:
    d = df_in.copy()
    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0
    d[out_col] = pd.to_numeric(d[stagec_col], errors="coerce").fillna(0.0)
    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values([stagec_col, "merge_key"], ascending=[False, True], kind="mergesort").head(int(topn)).index
        llm = pd.to_numeric(d.loc[idx, "llm_score"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        llm_n = (llm - float(llm.min())) / (float(llm.max()) - float(llm.min()) + 1e-12)
        stagec = pd.to_numeric(d.loc[idx, stagec_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()
        top = pd.DataFrame({"ix": idx.to_list(), "llm_n": llm_n, "stagec": stagec, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "stagec", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)
    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d

# Compute Stage C rank-normalized score
d = df0.copy()
emb_raw = float(W_EMBED_MAX) * pd.to_numeric(d["score_embed_max"], errors="coerce").fillna(0.0) + (1.0 - float(W_EMBED_MAX)) * pd.to_numeric(d["score_embed_mean_top3"], errors="coerce").fillna(0.0)
d["_emb_raw"] = emb_raw
d["_emb_r"] = ranknorm_by_group(d, "_emb_raw")
d["_tf_r"] = ranknorm_by_group(d, "score_tfidf")
d["_cite_r"] = ranknorm_by_group(d, "score_cite_norm")
base = float(W_EMBED) * d["_emb_r"] + (1.0 - float(W_EMBED)) * d["_tf_r"]
d["score_stageC_ranknorm_v1"] = (1.0 - float(CITE_WEIGHT)) * base + float(CITE_WEIGHT) * d["_cite_r"]

# Apply Stage C.3 tie-break using the new Stage C score
d = add_stagec3_topn_tiebreak(d, stagec_col="score_stageC_ranknorm_v1", out_col="score_stageC3_ranknorm_tiebreak_v1", topn=int(TOPN))

# Compare against production baseline (existing Stage C + Stage C.3 tie-break)
d = add_stagec3_topn_tiebreak(d, stagec_col="score_stageC_final", out_col="score_stageC3_topn_tiebreak_v1", topn=int(TOPN))

res = evaluate_all(d, score_cols=["score_stageC3_topn_tiebreak_v1", "score_stageC3_ranknorm_tiebreak_v1"], ks=KS)
macro = res[res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)
print("Macro:")
display(macro)

m = macro.set_index("score_col")
cols = [c for c in ["ndcg@20", "p@20", "mrr_include", "auc_include"] if c in m.columns]
if set(m.index) >= {"score_stageC3_topn_tiebreak_v1", "score_stageC3_ranknorm_tiebreak_v1"}:
    delta = (m.loc["score_stageC3_ranknorm_tiebreak_v1", cols] - m.loc["score_stageC3_topn_tiebreak_v1", cols]).to_frame("delta")
    print("Delta (ranknorm - baseline):")
    display(delta)

def first_include_rank(df_in: pd.DataFrame, cid: str, score_col: str) -> int | None:
    g = df_in[df_in["chapter_id"].astype(str) == str(cid)].copy()
    g["_s"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
    g = g.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    hits = g.index[g["final_label"].astype(str) == "include"].to_numpy()
    return int(hits[0] + 1) if len(hits) else None

rows = []
for cid in sorted(d["chapter_id"].dropna().astype(str).unique().tolist()):
    rows.append({
        "chapter_id": cid,
        "first_include_rank_baseline": first_include_rank(d, cid, "score_stageC3_topn_tiebreak_v1"),
        "first_include_rank_ranknorm": first_include_rank(d, cid, "score_stageC3_ranknorm_tiebreak_v1"),
    })
print("First include ranks:")
display(pd.DataFrame(rows))

# Persist
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_stageC_norm_rank_v1",
    "experiment_notes": f"W_EMBED_MAX={W_EMBED_MAX}; W_EMBED={W_EMBED}; CITE_WEIGHT={CITE_WEIGHT}; TOPN={TOPN}",
}

scored_path = EXP_DIR / f"{run_id_use}_stageC3_stageC_norm_rank_v1_scored.csv"
d.assign(**meta_use, experiment="stageC3_stageC_norm_rank_v1_scored").to_csv(scored_path, index=False)
print("Saved:", scored_path)

eval_path = EXP_DIR / f"{run_id_use}_stageC3_stageC_norm_rank_v1_eval.csv"
res.assign(**meta_use, experiment="stageC3_stageC_norm_rank_v1_eval").to_csv(eval_path, index=False)
print("Saved:", eval_path)

'''
    exec(_CODE, globals())


Skipping st_stageC3_stageC_norm_rank_v1 (RUN_MODE=label_adjudication)


In [40]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_base_scorecol_sweep_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3: sweep which Stage C base score column we should use
# (offline; no API calls)
#
# Scientific intent:
# - Stage C.3 can only rerank within TOPN by the base score.
# - If an INCLUDE is outside top-50 (e.g. platform_methodology), LLM can
#   never lift it. So the *base score column* matters a lot.
# - We already tuned weights, but this tests whether a different existing
#   base scoring signal yields a better top-50 set and downstream metrics.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "llm_score", "score_cite_norm"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

TOPN = 50
BASE_COLS = [
    "score_stageC_final",
    "score_hybrid_pool",
    "score_relevance_hybrid",
    "score_embed_combo",
    "score_embed_max",
    "score_embed_mean_top3",
    "score_tfidf",
    "score_stageC1",
]
BASE_COLS = [c for c in BASE_COLS if c in df0.columns]
print("Base score columns:", BASE_COLS)

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)

def add_stagec3_from_base(df_in: pd.DataFrame, base_col: str, out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    d["_base_n"] = minmax_by_group(d, base_col)
    d[out_col] = pd.to_numeric(d["_base_n"], errors="coerce").fillna(0.0)

    for cid, g in d.groupby("chapter_id"):
        idx = g.sort_values(["_base_n", "merge_key"], ascending=[False, True], kind="mergesort").head(int(TOPN)).index
        llm_n = minmax_series(d.loc[idx, "llm_score"]).to_numpy(dtype=float)
        base = pd.to_numeric(d.loc[idx, "_base_n"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({"ix": idx.to_list(), "llm_n": llm_n, "base": base, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "base", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)
    return d

def first_include_rank(df_in: pd.DataFrame, cid: str, score_col: str) -> int | None:
    g = df_in[df_in["chapter_id"].astype(str) == str(cid)].copy()
    g["_s"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
    g = g.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    hits = g.index[g["final_label"].astype(str) == "include"].to_numpy()
    return int(hits[0] + 1) if len(hits) else None

# -----------------------------
# Sweep (macro)
# -----------------------------
rows = []
for base_col in BASE_COLS:
    d = add_stagec3_from_base(df0, base_col=base_col, out_col="score_tmp")
    res = evaluate_all(d, score_cols=["score_tmp"], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({
        "base_col": base_col,
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
        "first_include_rank_platform_methodology": first_include_rank(d, "platform_methodology", "score_tmp"),
    })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("Base-col sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best base_col by ndcg@20:", best)

# -----------------------------
# LOCO (select base_col on 2 chapters)
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    best_col = None
    best_train = -1e9
    for base_col in BASE_COLS:
        dtr = add_stagec3_from_base(train, base_col=base_col, out_col="score_tmp")
        nd = float(evaluate_all(dtr, score_cols=["score_tmp"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
        if nd > best_train:
            best_train = nd
            best_col = base_col

    dte = add_stagec3_from_base(test, base_col=str(best_col), out_col="score_tmp")
    hold_nd = float(evaluate_all(dte, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    # baseline for this holdout = base_col=score_stageC_final
    dbase = add_stagec3_from_base(test, base_col="score_stageC_final", out_col="score_tmp")
    base_nd = float(evaluate_all(dbase, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_base_col": str(best_col),
        "train_ndcg@20": float(best_train),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_nd),
        "holdout_delta_vs_base": float(hold_nd - base_nd),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO results:")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))

# Persist artifacts
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_base_scorecol_sweep_v1",
    "experiment_notes": f"TOPN={TOPN}; cols={BASE_COLS}",
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_base_scorecol_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageC3_base_scorecol_sweep_v1").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_base_scorecol_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageC3_base_scorecol_loco_v1").to_csv(loco_path, index=False)
print("Saved:", loco_path)

'''
    exec(_CODE, globals())


Skipping st_stageC3_base_scorecol_sweep_v1 (RUN_MODE=label_adjudication)


In [41]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_shortlist_union_sweep_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3: shortlist expansion via union-of-toplists (offline; no API calls)
#
# Scientific intent:
# - Stage C.3 can only change ordering inside the shortlist.
# - Some chapters (platform_methodology) have INCLUDEs far below top-50.
# - Instead of changing the base score globally (which hurts other chapters),
#   expand the shortlist as UNION(top-50 by Stage C final, top-K by another signal).
# - Then apply the finalized deterministic tie-break rerank using existing llm_score.
#
# Output:
# - Macro sweep + LOCO to see if any union config improves generalization.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "llm_score", "score_stageC_final"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

TOPN_BASE = 50
ALT_TOPK = [5, 10, 20, 50]
ALT_COLS = ["score_tfidf", "score_stageC1", "score_relevance_hybrid", "score_hybrid_pool"]
ALT_COLS = [c for c in ALT_COLS if c in df0.columns]
print("Alt cols:", ALT_COLS)

def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)

def shortlist_union_idx(df_in: pd.DataFrame, *, alt_col: str | None, alt_k: int) -> dict[str, list[int]]:
    out = {}
    for cid, g in df_in.groupby("chapter_id"):
        g = g.sort_values(["score_stageC_final", "merge_key"], ascending=[False, True], kind="mergesort")
        base_idx = g.head(int(TOPN_BASE)).index.to_list()
        if alt_col is None:
            out[str(cid)] = base_idx
            continue
        ga = g.sort_values([alt_col, "merge_key"], ascending=[False, True], kind="mergesort")
        alt_idx = ga.head(int(alt_k)).index.to_list()
        out[str(cid)] = list(dict.fromkeys(base_idx + alt_idx))
    return out

def apply_stagec3_tiebreak(df_in: pd.DataFrame, idx_by_cid: dict[str, list[int]], out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0

    d[out_col] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)
    for cid, idx in idx_by_cid.items():
        if not idx:
            continue
        llm_n = minmax_series(d.loc[idx, "llm_score"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(d.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({"ix": list(idx), "llm_n": llm_n, "stagec": stagec, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "stagec", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)

    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d

def first_include_rank(df_in: pd.DataFrame, cid: str, score_col: str) -> int | None:
    g = df_in[df_in["chapter_id"].astype(str) == str(cid)].copy()
    g["_s"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
    g = g.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    hits = g.index[g["final_label"].astype(str) == "include"].to_numpy()
    return int(hits[0] + 1) if len(hits) else None

# Baseline reference
idx_base = shortlist_union_idx(df0, alt_col=None, alt_k=0)
df_base = apply_stagec3_tiebreak(df0, idx_by_cid=idx_base, out_col="score_stageC3_union_base")
base_macro = evaluate_all(df_base, score_cols=["score_stageC3_union_base"], ks=KS).query("chapter_id=='__ALL__'").iloc[0].to_dict()
print("Baseline macro ndcg@20:", float(base_macro.get("ndcg@20")))

# -----------------------------
# Sweep
# -----------------------------
rows = []
for alt_col in ALT_COLS:
    for k in ALT_TOPK:
        idx = shortlist_union_idx(df0, alt_col=alt_col, alt_k=int(k))
        out_col = "score_stageC3_union_tmp"
        d = apply_stagec3_tiebreak(df0, idx_by_cid=idx, out_col=out_col)
        res = evaluate_all(d, score_cols=[out_col], ks=KS)
        macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
        union_sizes = [len(v) for v in idx.values()]
        rows.append({
            "alt_col": str(alt_col),
            "alt_k": int(k),
            "mean_union_size": float(np.mean(union_sizes)) if union_sizes else float("nan"),
            "max_union_size": int(max(union_sizes)) if union_sizes else 0,
            "mean_extra_calls": (float(np.mean(union_sizes)) - float(TOPN_BASE)) if union_sizes else float("nan"),
            "max_extra_calls": int(max(union_sizes) - int(TOPN_BASE)) if union_sizes else 0,
            "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
            "p@20": float(macro.get("p@20", float("nan"))),
            "mrr_include": float(macro.get("mrr_include", float("nan"))),
            "auc_include": float(macro.get("auc_include", float("nan"))),
            "first_include_rank_platform_methodology": first_include_rank(d, "platform_methodology", out_col),
        })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("\nShortlist union sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best by ndcg@20:", best)

# -----------------------------
# LOCO
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    # baseline on holdout
    idx_b = shortlist_union_idx(test, alt_col=None, alt_k=0)
    d_b = apply_stagec3_tiebreak(test, idx_by_cid=idx_b, out_col="score_tmp")
    base_nd = float(evaluate_all(d_b, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    best_cfg = None
    best_train_nd = -1e9
    for alt_col in ALT_COLS:
        for k in ALT_TOPK:
            idx_tr = shortlist_union_idx(train, alt_col=alt_col, alt_k=int(k))
            d_tr = apply_stagec3_tiebreak(train, idx_by_cid=idx_tr, out_col="score_tmp")
            nd = float(evaluate_all(d_tr, score_cols=["score_tmp"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
            if nd > best_train_nd:
                best_train_nd = nd
                best_cfg = {"alt_col": str(alt_col), "alt_k": int(k)}

    idx_te = shortlist_union_idx(test, alt_col=best_cfg["alt_col"], alt_k=int(best_cfg["alt_k"]))
    d_te = apply_stagec3_tiebreak(test, idx_by_cid=idx_te, out_col="score_tmp")
    hold_nd = float(evaluate_all(d_te, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])
    union_sizes = [len(v) for v in idx_te.values()]

    loco_rows.append({
        "holdout_chapter": holdout,
        **best_cfg,
        "train_ndcg@20": float(best_train_nd),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_nd),
        "holdout_delta_vs_base": float(hold_nd - base_nd),
        "holdout_union_size": int(np.mean(union_sizes)) if union_sizes else 0,
        "holdout_first_include_rank": first_include_rank(d_te, holdout, "score_tmp"),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO results:")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))

# Persist artifacts
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_shortlist_union_sweep_v1",
    "experiment_notes": f"TOPN_BASE={TOPN_BASE}; ALT_TOPK={ALT_TOPK}; ALT_COLS={ALT_COLS}",
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_union_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageC3_shortlist_union_sweep_v1").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_union_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageC3_shortlist_union_loco_v1").to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv + save a scored CSV for the best config
if not sweep_df.empty:
    best_alt_col = str(best.get("alt_col"))
    best_alt_k = int(best.get("alt_k"))
    idx_best = shortlist_union_idx(df0, alt_col=best_alt_col, alt_k=best_alt_k)
    out_col_best = "score_stageC3_shortlist_union_best_v1"
    d_best = apply_stagec3_tiebreak(df0, idx_by_cid=idx_best, out_col=out_col_best)
    best_res = evaluate_all(d_best, score_cols=[out_col_best], ks=KS)
    best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)

    best_meta = dict(meta_use)
    best_meta["experiment_tag"] = "stageC3_shortlist_union_best_v1"
    best_meta["experiment_notes"] = f"TOPN_BASE={TOPN_BASE}; alt_col={best_alt_col}; alt_k={best_alt_k}"

    runs_csv = EXP_DIR / "runs.csv"
    best_macro = best_macro.assign(**best_meta, experiment="stageC3_shortlist_union_best_v1")
    if runs_csv.exists():
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended best macro row to:", runs_csv)

    scored_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_union_best_v1_scored.csv"
    keep_cols = [
        c
        for c in ["chapter_id", "merge_key", "final_label", "score_stageC_final", "llm_score", out_col_best]
        if c in d_best.columns
    ]
    d_best[keep_cols].to_csv(scored_path, index=False)
    print("Saved best scored CSV:", scored_path)

'''
    exec(_CODE, globals())


Skipping st_stageC3_shortlist_union_sweep_v1 (RUN_MODE=label_adjudication)


In [42]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_stageC_rrf_sweep_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C (alternative) + Stage C.3: Reciprocal Rank Fusion (RRF) shortlist builder (offline; no API calls)
#
# Scientific intent:
# - We have a hard limitation: Stage C.3 only reranks within top-50.
# - `platform_methodology` has its first INCLUDE around rank ~58 in the current best pipeline.
# - Naive shortlist union helps only with big K and hurts LOCO.
# - Try a *robust rank-fusion* for Stage C scoring so that docs that rank high in TF-IDF
#   can still enter the top-50 even if embeddings underrate them.
# - Evaluate end-to-end by applying the finalized Stage C.3 deterministic rerank on top.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "llm_score", "score_stageC_final", "score_tfidf"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

TOPN = 50
EMB_COLS = [c for c in ["score_embed_mean_top3", "score_embed_max"] if c in df0.columns]
if not EMB_COLS:
    raise RuntimeError("No embedding score columns found (expected score_embed_mean_top3 or score_embed_max)")

HAS_CITE = "score_cite_norm" in df0.columns
print("Embedding cols:", EMB_COLS)
print("Has citations:", HAS_CITE)

RRF_KS = [10, 30, 60, 100]
TFIDF_WEIGHTS = [1.0, 2.0]  # small boost to help methodology-like chapters


def _sorted_rank(g: pd.DataFrame, score_col: str) -> pd.Series:
    # Stable: break ties by merge_key.
    gg = g[[score_col, "merge_key"]].copy()
    gg[score_col] = pd.to_numeric(gg[score_col], errors="coerce").fillna(-1e9)
    gg["merge_key"] = gg["merge_key"].fillna("").astype(str)
    gg = gg.sort_values([score_col, "merge_key"], ascending=[False, True], kind="mergesort")
    r = pd.Series(np.arange(1, len(gg) + 1, dtype=int), index=gg.index)
    return r.reindex(g.index)


def add_rrf_score(
    df_in: pd.DataFrame,
    *,
    out_col: str,
    rrf_k: int,
    emb_col: str,
    w_emb: float,
    w_tfidf: float,
    w_cite: float,
) -> pd.DataFrame:
    d = df_in.copy()
    if "score_cite_norm" not in d.columns:
        d["score_cite_norm"] = 0.0

    # Avoid pandas groupby.apply edge-case when only one group (can return a DataFrame)
    rank_emb = pd.Series(index=d.index, dtype=int)
    rank_tf = pd.Series(index=d.index, dtype=int)
    rank_cite = pd.Series(index=d.index, dtype=int)
    for _, g in d.groupby("chapter_id"):
        rank_emb.loc[g.index] = _sorted_rank(g, emb_col).to_numpy(dtype=int)
        rank_tf.loc[g.index] = _sorted_rank(g, "score_tfidf").to_numpy(dtype=int)
        rank_cite.loc[g.index] = _sorted_rank(g, "score_cite_norm").to_numpy(dtype=int)
    d["_rank_emb"] = rank_emb
    d["_rank_tf"] = rank_tf
    d["_rank_cite"] = rank_cite

    kk = float(rrf_k)
    d[out_col] = (
        float(w_emb) / (kk + d["_rank_emb"].astype(float))
        + float(w_tfidf) / (kk + d["_rank_tf"].astype(float))
        + float(w_cite) / (kk + d["_rank_cite"].astype(float))
    )

    d = d.drop(columns=["_rank_emb", "_rank_tf", "_rank_cite"], errors="ignore")
    if not HAS_CITE:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d


def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)


def shortlist_topn_idx(df_in: pd.DataFrame, base_score_col: str, topn: int = 50) -> dict[str, list[int]]:
    out: dict[str, list[int]] = {}
    for cid, g in df_in.groupby("chapter_id"):
        gg = g.copy()
        gg["_s"] = pd.to_numeric(gg[base_score_col], errors="coerce").fillna(-1e9)
        gg = gg.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort")
        out[str(cid)] = gg.head(int(topn)).index.to_list()
    return out


def apply_stagec3_tiebreak(df_in: pd.DataFrame, *, base_score_col: str, idx_by_cid: dict[str, list[int]], out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0

    d[out_col] = pd.to_numeric(d[base_score_col], errors="coerce").fillna(0.0)

    for cid, idx in idx_by_cid.items():
        if not idx:
            continue
        llm_n = minmax_series(d.loc[idx, "llm_score"]).to_numpy(dtype=float)
        base = pd.to_numeric(d.loc[idx, base_score_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({"ix": list(idx), "llm_n": llm_n, "base": base, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "base", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)

    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d


def first_include_rank(df_in: pd.DataFrame, cid: str, score_col: str) -> int | None:
    g = df_in[df_in["chapter_id"].astype(str) == str(cid)].copy()
    g["_s"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
    g = g.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    hits = g.index[g["final_label"].astype(str) == "include"].to_numpy()
    return int(hits[0] + 1) if len(hits) else None


# Baseline (production): Stage C final -> Stage C.3 top-50 tiebreak
idx_base = shortlist_topn_idx(df0, base_score_col="score_stageC_final", topn=TOPN)
df_base = apply_stagec3_tiebreak(df0, base_score_col="score_stageC_final", idx_by_cid=idx_base, out_col="score_stageC3_topn_tiebreak_v1")
base_macro = evaluate_all(df_base, score_cols=["score_stageC3_topn_tiebreak_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0].to_dict()
print("Baseline ndcg@20:", float(base_macro.get("ndcg@20")))
print("Baseline first include ranks:", {
    cid: first_include_rank(df_base, cid, "score_stageC3_topn_tiebreak_v1")
    for cid in sorted(df0["chapter_id"].astype(str).unique().tolist())
})


# -----------------------------
# Sweep
# -----------------------------
rows = []
for emb_col in EMB_COLS:
    for rrf_k in RRF_KS:
        for w_tf in TFIDF_WEIGHTS:
            out_stagec = "score_stageC_rrf_tmp"
            d = add_rrf_score(
                df0,
                out_col=out_stagec,
                rrf_k=int(rrf_k),
                emb_col=str(emb_col),
                w_emb=1.0,
                w_tfidf=float(w_tf),
                w_cite=0.25 if HAS_CITE else 0.0,
            )

            idx = shortlist_topn_idx(d, base_score_col=out_stagec, topn=TOPN)
            d2 = apply_stagec3_tiebreak(d, base_score_col=out_stagec, idx_by_cid=idx, out_col="score_stageC3_rrf_tmp")
            res = evaluate_all(d2, score_cols=["score_stageC3_rrf_tmp"], ks=KS)
            macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
            rows.append({
                "emb_col": str(emb_col),
                "rrf_k": int(rrf_k),
                "w_tfidf": float(w_tf),
                "w_cite": float(0.25 if HAS_CITE else 0.0),
                "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
                "p@20": float(macro.get("p@20", float("nan"))),
                "mrr_include": float(macro.get("mrr_include", float("nan"))),
                "auc_include": float(macro.get("auc_include", float("nan"))),
                "first_include_rank_platform_methodology": first_include_rank(d2, "platform_methodology", "score_stageC3_rrf_tmp"),
            })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("\nStage C RRF → Stage C.3 sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best by ndcg@20:", best)


# -----------------------------
# LOCO
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    # holdout baseline
    idx_b = shortlist_topn_idx(test, base_score_col="score_stageC_final", topn=TOPN)
    d_b = apply_stagec3_tiebreak(test, base_score_col="score_stageC_final", idx_by_cid=idx_b, out_col="score_tmp")
    base_nd = float(evaluate_all(d_b, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    best_cfg = None
    best_train_nd = -1e9
    for emb_col in EMB_COLS:
        for rrf_k in RRF_KS:
            for w_tf in TFIDF_WEIGHTS:
                tr = add_rrf_score(
                    train,
                    out_col="score_stageC_rrf_tmp",
                    rrf_k=int(rrf_k),
                    emb_col=str(emb_col),
                    w_emb=1.0,
                    w_tfidf=float(w_tf),
                    w_cite=0.25 if HAS_CITE else 0.0,
                )
                idx_tr = shortlist_topn_idx(tr, base_score_col="score_stageC_rrf_tmp", topn=TOPN)
                tr2 = apply_stagec3_tiebreak(tr, base_score_col="score_stageC_rrf_tmp", idx_by_cid=idx_tr, out_col="score_tmp")
                nd = float(evaluate_all(tr2, score_cols=["score_tmp"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
                if nd > best_train_nd:
                    best_train_nd = nd
                    best_cfg = {"emb_col": str(emb_col), "rrf_k": int(rrf_k), "w_tfidf": float(w_tf)}

    te = add_rrf_score(
        test,
        out_col="score_stageC_rrf_tmp",
        rrf_k=int(best_cfg["rrf_k"]),
        emb_col=str(best_cfg["emb_col"]),
        w_emb=1.0,
        w_tfidf=float(best_cfg["w_tfidf"]),
        w_cite=0.25 if HAS_CITE else 0.0,
    )
    idx_te = shortlist_topn_idx(te, base_score_col="score_stageC_rrf_tmp", topn=TOPN)
    te2 = apply_stagec3_tiebreak(te, base_score_col="score_stageC_rrf_tmp", idx_by_cid=idx_te, out_col="score_tmp")
    hold_nd = float(evaluate_all(te2, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    loco_rows.append({
        "holdout_chapter": holdout,
        **best_cfg,
        "train_ndcg@20": float(best_train_nd),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_nd),
        "holdout_delta_vs_base": float(hold_nd - base_nd),
        "holdout_first_include_rank": first_include_rank(te2, holdout, "score_tmp"),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO results:")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))


# Persist artifacts
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_stageC_rrf_sweep_v1",
    "experiment_notes": f"TOPN={TOPN}; EMB_COLS={EMB_COLS}; RRF_KS={RRF_KS}; TFIDF_WEIGHTS={TFIDF_WEIGHTS}; w_cite={(0.25 if HAS_CITE else 0.0)}",
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_stageC_rrf_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageC3_stageC_rrf_sweep_v1").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_stageC_rrf_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageC3_stageC_rrf_loco_v1").to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv (for quick tracking) + save a scored CSV for the best macro config
if not sweep_df.empty:
    best_emb = str(best.get("emb_col"))
    best_rrf_k = int(best.get("rrf_k"))
    best_w_tf = float(best.get("w_tfidf"))

    d_best = add_rrf_score(
        df0,
        out_col="score_stageC_rrf_best_v1",
        rrf_k=int(best_rrf_k),
        emb_col=best_emb,
        w_emb=1.0,
        w_tfidf=float(best_w_tf),
        w_cite=0.25 if HAS_CITE else 0.0,
    )
    idx_best = shortlist_topn_idx(d_best, base_score_col="score_stageC_rrf_best_v1", topn=TOPN)
    d_best = apply_stagec3_tiebreak(d_best, base_score_col="score_stageC_rrf_best_v1", idx_by_cid=idx_best, out_col="score_stageC3_rrf_best_v1")
    best_res = evaluate_all(d_best, score_cols=["score_stageC3_rrf_best_v1"], ks=KS)
    best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)

    best_meta = dict(meta_use)
    best_meta["experiment_tag"] = "stageC3_stageC_rrf_best_v1"
    best_meta["experiment_notes"] = f"TOPN={TOPN}; emb_col={best_emb}; rrf_k={best_rrf_k}; w_tfidf={best_w_tf}; w_cite={(0.25 if HAS_CITE else 0.0)}"  # macro-best (may overfit)

    runs_csv = EXP_DIR / "runs.csv"
    best_macro = best_macro.assign(**best_meta, experiment="stageC3_stageC_rrf_best_v1")
    if runs_csv.exists():
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended best macro row to:", runs_csv)

    scored_path = EXP_DIR / f"{run_id_use}_stageC3_stageC_rrf_best_v1_scored.csv"
    keep_cols = [
        c
        for c in ["chapter_id", "merge_key", "final_label", "score_stageC_final", "llm_score", "score_stageC_rrf_best_v1", "score_stageC3_rrf_best_v1"]
        if c in d_best.columns
    ]
    d_best[keep_cols].to_csv(scored_path, index=False)
    print("Saved best scored CSV:", scored_path)

'''
    exec(_CODE, globals())


Skipping st_stageC3_stageC_rrf_sweep_v1 (RUN_MODE=label_adjudication)


In [43]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_shortlist_mix_tfidf_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 shortlist membership test: mix Stage C final + TF-IDF (offline; no API calls)
#
# Scientific intent:
# - `platform_methodology` INCLUDEs are at ranks ~58/81 under `score_stageC_final`, so they never enter the top-50 rerank.
# - They are within TF-IDF top-50 (ranks ~43/48), but naive shortlist union hurts LOCO.
# - Try a *soft* approach: pick the top-50 shortlist by a convex combination of:
#     mix = (1-gamma)*norm(score_stageC_final) + gamma*norm(score_tfidf)
#   while keeping the final ranking outside the shortlist on `score_stageC_final`.
# - Then apply the finalized Stage C.3 deterministic rerank on that shortlist using existing `llm_score`.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "llm_score", "score_stageC_final", "score_tfidf"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

TOPN = 50
GAMMAS = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

def minmax_by_group(df_in: pd.DataFrame, col: str, group_col: str = "chapter_id") -> pd.Series:
    out = pd.Series(index=df_in.index, dtype=float)
    for _, g in df_in.groupby(group_col):
        x = pd.to_numeric(g[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mn, mx = float(np.min(x)), float(np.max(x))
        out.loc[g.index] = (x - mn) / (mx - mn + 1e-12)
    return out

def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)

def shortlist_idx_mix(df_in: pd.DataFrame, gamma: float) -> dict[str, list[int]]:
    d = df_in.copy()
    d["_sc_n"] = minmax_by_group(d, "score_stageC_final")
    d["_tf_n"] = minmax_by_group(d, "score_tfidf")
    d["_mix"] = (1.0 - float(gamma)) * d["_sc_n"] + float(gamma) * d["_tf_n"]

    out: dict[str, list[int]] = {}
    for cid, g in d.groupby("chapter_id"):
        gg = g.copy()
        gg["_mk"] = gg["merge_key"].fillna("").astype(str)
        gg = gg.sort_values(["_mix", "_mk"], ascending=[False, True], kind="mergesort")
        out[str(cid)] = gg.head(int(TOPN)).index.to_list()
    return out

def shortlist_idx_stagec(df_in: pd.DataFrame) -> dict[str, list[int]]:
    out: dict[str, list[int]] = {}
    for cid, g in df_in.groupby("chapter_id"):
        gg = g.copy()
        gg["_s"] = pd.to_numeric(gg["score_stageC_final"], errors="coerce").fillna(-1e9)
        gg["_mk"] = gg["merge_key"].fillna("").astype(str)
        gg = gg.sort_values(["_s", "_mk"], ascending=[False, True], kind="mergesort")
        out[str(cid)] = gg.head(int(TOPN)).index.to_list()
    return out

def apply_stagec3_tiebreak(df_in: pd.DataFrame, idx_by_cid: dict[str, list[int]], out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0

    # Keep Stage C final for global ordering; overwrite only inside shortlist.
    d[out_col] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)

    for cid, idx in idx_by_cid.items():
        if not idx:
            continue
        llm_n = minmax_series(d.loc[idx, "llm_score"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(d.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({"ix": list(idx), "llm_n": llm_n, "stagec": stagec, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "stagec", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)

    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d

def first_include_rank(df_in: pd.DataFrame, cid: str, score_col: str) -> int | None:
    g = df_in[df_in["chapter_id"].astype(str) == str(cid)].copy()
    g["_s"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
    g = g.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    hits = g.index[g["final_label"].astype(str) == "include"].to_numpy()
    return int(hits[0] + 1) if len(hits) else None

def shortlist_changes(idx_a: dict[str, list[int]], idx_b: dict[str, list[int]]) -> dict[str, int]:
    out = {}
    for cid in idx_a.keys():
        a = set(idx_a.get(cid, []))
        b = set(idx_b.get(cid, []))
        out[cid] = int(TOPN - len(a & b))
    return out

# Baseline reference
idx_base = shortlist_idx_stagec(df0)
df_base = apply_stagec3_tiebreak(df0, idx_by_cid=idx_base, out_col="score_stageC3_topn_tiebreak_v1")
base_macro = evaluate_all(df_base, score_cols=["score_stageC3_topn_tiebreak_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0].to_dict()
print("Baseline ndcg@20:", float(base_macro.get("ndcg@20")))
print("Baseline first include ranks:", {
    cid: first_include_rank(df_base, cid, "score_stageC3_topn_tiebreak_v1")
    for cid in sorted(df0["chapter_id"].astype(str).unique().tolist())
})

# -----------------------------
# Sweep
# -----------------------------
rows = []
for gamma in GAMMAS:
    idx = shortlist_idx_mix(df0, gamma=float(gamma))
    d = apply_stagec3_tiebreak(df0, idx_by_cid=idx, out_col="score_stageC3_mix_tmp")
    res = evaluate_all(d, score_cols=["score_stageC3_mix_tmp"], ks=KS)
    macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    ch = shortlist_changes(idx_base, idx)
    rows.append({
        "gamma": float(gamma),
        "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
        "p@20": float(macro.get("p@20", float("nan"))),
        "mrr_include": float(macro.get("mrr_include", float("nan"))),
        "auc_include": float(macro.get("auc_include", float("nan"))),
        "first_include_rank_platform_methodology": first_include_rank(d, "platform_methodology", "score_stageC3_mix_tmp"),
        "mean_shortlist_changes": float(np.mean(list(ch.values()))) if ch else float("nan"),
        "changes_empirical": int(ch.get("platform_empirical_case", 0)),
        "changes_methodology": int(ch.get("platform_methodology", 0)),
        "changes_theory": int(ch.get("platform_theory", 0)),
    })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("\nShortlist mix sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best by ndcg@20:", best)

# -----------------------------
# LOCO
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    # baseline on holdout
    idx_b = shortlist_idx_stagec(test)
    d_b = apply_stagec3_tiebreak(test, idx_by_cid=idx_b, out_col="score_tmp")
    base_nd = float(evaluate_all(d_b, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    best_g = None
    best_train_nd = -1e9
    for gamma in GAMMAS:
        idx_tr = shortlist_idx_mix(train, gamma=float(gamma))
        d_tr = apply_stagec3_tiebreak(train, idx_by_cid=idx_tr, out_col="score_tmp")
        nd = float(evaluate_all(d_tr, score_cols=["score_tmp"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
        if nd > best_train_nd:
            best_train_nd = nd
            best_g = float(gamma)

    idx_te = shortlist_idx_mix(test, gamma=float(best_g))
    d_te = apply_stagec3_tiebreak(test, idx_by_cid=idx_te, out_col="score_tmp")
    hold_nd = float(evaluate_all(d_te, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    loco_rows.append({
        "holdout_chapter": holdout,
        "selected_gamma": float(best_g),
        "train_ndcg@20": float(best_train_nd),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_nd),
        "holdout_delta_vs_base": float(hold_nd - base_nd),
        "holdout_first_include_rank": first_include_rank(d_te, holdout, "score_tmp"),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO results:")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))

# Persist artifacts
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_shortlist_mix_tfidf_v1",
    "experiment_notes": f"TOPN={TOPN}; GAMMAS={GAMMAS}",
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_mix_tfidf_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageC3_shortlist_mix_tfidf_sweep_v1").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_mix_tfidf_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageC3_shortlist_mix_tfidf_loco_v1").to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv + save a scored CSV for the best gamma
if not sweep_df.empty:
    best_gamma = float(best.get("gamma", 0.0))
    idx_best = shortlist_idx_mix(df0, gamma=float(best_gamma))
    out_col_best = "score_stageC3_shortlist_mix_tfidf_best_v1"
    d_best = apply_stagec3_tiebreak(df0, idx_by_cid=idx_best, out_col=out_col_best)
    best_res = evaluate_all(d_best, score_cols=[out_col_best], ks=KS)
    best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)

    best_meta = dict(meta_use)
    best_meta["experiment_tag"] = "stageC3_shortlist_mix_tfidf_best_v1"
    best_meta["experiment_notes"] = f"TOPN={TOPN}; best_gamma={best_gamma}"

    runs_csv = EXP_DIR / "runs.csv"
    best_macro = best_macro.assign(**best_meta, experiment="stageC3_shortlist_mix_tfidf_best_v1")
    if runs_csv.exists():
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended best macro row to:", runs_csv)

    scored_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_mix_tfidf_best_v1_scored.csv"
    keep_cols = [
        c
        for c in ["chapter_id", "merge_key", "final_label", "score_stageC_final", "score_tfidf", "llm_score", out_col_best]
        if c in d_best.columns
    ]
    d_best[keep_cols].to_csv(scored_path, index=False)
    print("Saved best scored CSV:", scored_path)

'''
    exec(_CODE, globals())


Skipping st_stageC3_shortlist_mix_tfidf_v1 (RUN_MODE=label_adjudication)


In [44]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_shortlist_rescue_intersection_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Stage C.3 shortlist rescue (offline; no API calls)
#
# Scientific intent:
# - We need to keep LLM calls at TOPN=50 (LLM helps only within top-50).
# - But `platform_methodology` INCLUDEs are outside Stage C top-50.
# - TF-IDF ranks them better (within TF-IDF top-50), yet naive union/score-mixing harmed LOCO.
# - Here we try a *conservative swap*:
#   Start with Stage C top-50.
#   Identify candidate docs that are ALSO strong in an alternate rank signal (TF-IDF or StageC1)
#   and not too far down in Stage C (<= stagec_rank_max).
#   Swap at most `swaps` of them into the shortlist by replacing the worst Stage C docs.
#
# This keeps TOPN=50 (cost unchanged) and aims to rescue chapters where embeddings suppress good lexical matches.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found. Run st_stageC3_llm_rerank_v1 at least once.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "llm_score", "score_stageC_final", "score_tfidf"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

TOPN = 50
ALT_COLS = [c for c in ["score_tfidf", "score_stageC1"] if c in df0.columns]
ALT_RANK_MAXS = [20, 50]
STAGEC_RANK_MAXS = [60, 80, 100]
SWAPS = [0, 2, 5, 8, 10, 12]

print("Alt cols:", ALT_COLS)

def rank_by(g: pd.DataFrame, score_col: str) -> pd.Series:
    gg = g.copy()
    gg["_s"] = pd.to_numeric(gg[score_col], errors="coerce").fillna(-1e9)
    gg["_mk"] = gg["merge_key"].fillna("").astype(str)
    gg = gg.sort_values(["_s", "_mk"], ascending=[False, True], kind="mergesort").reset_index()
    ranks = pd.Series(np.arange(1, len(gg) + 1, dtype=int), index=gg["index"].to_numpy())
    return ranks

def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)

def shortlist_base_idx(df_in: pd.DataFrame) -> dict[str, list[int]]:
    out: dict[str, list[int]] = {}
    for cid, g in df_in.groupby("chapter_id"):
        r = rank_by(g, "score_stageC_final")
        base = r[r <= int(TOPN)].sort_values().index.to_list()
        out[str(cid)] = base
    return out

def shortlist_rescue_idx(df_in: pd.DataFrame, *, alt_col: str, alt_rank_max: int, stagec_rank_max: int, swaps: int) -> tuple[dict[str, list[int]], list[dict]]:
    idx_by_cid: dict[str, list[int]] = {}
    stats: list[dict] = []
    for cid, g in df_in.groupby("chapter_id"):
        cid = str(cid)
        g = g.copy()
        r_sc = rank_by(g, "score_stageC_final")
        r_alt = rank_by(g, alt_col)

        base = r_sc[r_sc <= int(TOPN)].sort_values().index.to_list()
        base_set = set(base)

        cand = []
        for ix in g.index:
            if ix in base_set:
                continue
            if int(r_alt.get(ix, 10**9)) > int(alt_rank_max):
                continue
            if int(r_sc.get(ix, 10**9)) > int(stagec_rank_max):
                continue
            cand.append(ix)

        # Conservative candidate order: closest to Stage C top-50 first
        g_mk = g["merge_key"].fillna("").astype(str)
        cand = sorted(cand, key=lambda ix: (int(r_sc[ix]), int(r_alt[ix]), g_mk.loc[ix]))
        take = cand[: int(swaps)]

        # Remove the worst Stage C docs from the base shortlist (by Stage C rank)
        base_sorted = sorted(base, key=lambda ix: (int(r_sc[ix]), g_mk.loc[ix]))
        remove = base_sorted[-len(take):] if len(take) else []

        new = [ix for ix in base if ix not in set(remove)] + list(take)
        new = list(dict.fromkeys(new))

        # Safety: fill if something went weird
        if len(new) < int(TOPN):
            for ix in base_sorted:
                if ix not in set(new):
                    new.append(ix)
                if len(new) >= int(TOPN):
                    break
        if len(new) > int(TOPN):
            keep_set = set(take)
            # drop worst non-take first
            new_sorted = sorted(new, key=lambda ix: (int(r_sc.get(ix, 10**9)), g_mk.loc[ix]))
            # new_sorted is best->worst; drop from end
            drop = []
            for ix in reversed(new_sorted):
                if len(new_sorted) - len(drop) <= int(TOPN):
                    break
                if ix in keep_set:
                    continue
                drop.append(ix)
            kept = [ix for ix in new if ix not in set(drop)]
            new = kept[: int(TOPN)]

        idx_by_cid[cid] = new
        stats.append({
            "chapter_id": cid,
            "alt_col": str(alt_col),
            "alt_rank_max": int(alt_rank_max),
            "stagec_rank_max": int(stagec_rank_max),
            "swaps": int(swaps),
            "candidates_available": int(len(cand)),
            "swaps_applied": int(len(take)),
        })

    return idx_by_cid, stats

def apply_stagec3_tiebreak(df_in: pd.DataFrame, idx_by_cid: dict[str, list[int]], out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    cite_ok = "score_cite_norm" in d.columns
    if not cite_ok:
        d["score_cite_norm"] = 0.0

    d[out_col] = pd.to_numeric(d["score_stageC_final"], errors="coerce").fillna(0.0)
    for cid, idx in idx_by_cid.items():
        if not idx:
            continue
        llm_n = minmax_series(d.loc[idx, "llm_score"]).to_numpy(dtype=float)
        stagec = pd.to_numeric(d.loc[idx, "score_stageC_final"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()

        top = pd.DataFrame({"ix": list(idx), "llm_n": llm_n, "stagec": stagec, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "stagec", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)

    if not cite_ok:
        d = d.drop(columns=["score_cite_norm"], errors="ignore")
    return d

def first_include_rank(df_in: pd.DataFrame, cid: str, score_col: str) -> int | None:
    g = df_in[df_in["chapter_id"].astype(str) == str(cid)].copy()
    g["_s"] = pd.to_numeric(g[score_col], errors="coerce").fillna(-1e9)
    g = g.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    hits = g.index[g["final_label"].astype(str) == "include"].to_numpy()
    return int(hits[0] + 1) if len(hits) else None

# Baseline reference
idx_base = shortlist_base_idx(df0)
df_base = apply_stagec3_tiebreak(df0, idx_by_cid=idx_base, out_col="score_stageC3_topn_tiebreak_v1")
base_macro = evaluate_all(df_base, score_cols=["score_stageC3_topn_tiebreak_v1"], ks=KS).query("chapter_id=='__ALL__'").iloc[0].to_dict()
print("Baseline ndcg@20:", float(base_macro.get("ndcg@20")))
print("Baseline first include ranks:", {
    cid: first_include_rank(df_base, cid, "score_stageC3_topn_tiebreak_v1")
    for cid in sorted(df0["chapter_id"].astype(str).unique().tolist())
})

# -----------------------------
# Sweep
# -----------------------------
rows = []
for alt_col in ALT_COLS:
    for alt_rank_max in ALT_RANK_MAXS:
        for stagec_rank_max in STAGEC_RANK_MAXS:
            for swaps in SWAPS:
                idx, st = shortlist_rescue_idx(
                    df0,
                    alt_col=str(alt_col),
                    alt_rank_max=int(alt_rank_max),
                    stagec_rank_max=int(stagec_rank_max),
                    swaps=int(swaps),
                )
                d = apply_stagec3_tiebreak(df0, idx_by_cid=idx, out_col="score_stageC3_rescue_tmp")
                res = evaluate_all(d, score_cols=["score_stageC3_rescue_tmp"], ks=KS)
                macro = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
                mean_swaps_applied = float(np.mean([x["swaps_applied"] for x in st])) if st else float("nan")
                rows.append({
                    "alt_col": str(alt_col),
                    "alt_rank_max": int(alt_rank_max),
                    "stagec_rank_max": int(stagec_rank_max),
                    "swaps": int(swaps),
                    "mean_swaps_applied": mean_swaps_applied,
                    "ndcg@20": float(macro.get("ndcg@20", float("nan"))),
                    "p@20": float(macro.get("p@20", float("nan"))),
                    "mrr_include": float(macro.get("mrr_include", float("nan"))),
                    "auc_include": float(macro.get("auc_include", float("nan"))),
                    "first_include_rank_platform_methodology": first_include_rank(d, "platform_methodology", "score_stageC3_rescue_tmp"),
                })

sweep_df = pd.DataFrame(rows).sort_values(["ndcg@20", "mrr_include"], ascending=[False, False]).reset_index(drop=True)
print("\nShortlist rescue sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict() if not sweep_df.empty else {}
print("Best by ndcg@20:", best)

# -----------------------------
# LOCO
# -----------------------------
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    test = df0[df0["chapter_id"].astype(str) == holdout].copy()

    # baseline on holdout
    idx_b = shortlist_base_idx(test)
    d_b = apply_stagec3_tiebreak(test, idx_by_cid=idx_b, out_col="score_tmp")
    base_nd = float(evaluate_all(d_b, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    best_cfg = None
    best_train_nd = -1e9
    for alt_col in ALT_COLS:
        for alt_rank_max in ALT_RANK_MAXS:
            for stagec_rank_max in STAGEC_RANK_MAXS:
                for swaps in SWAPS:
                    idx_tr, _ = shortlist_rescue_idx(
                        train,
                        alt_col=str(alt_col),
                        alt_rank_max=int(alt_rank_max),
                        stagec_rank_max=int(stagec_rank_max),
                        swaps=int(swaps),
                    )
                    d_tr = apply_stagec3_tiebreak(train, idx_by_cid=idx_tr, out_col="score_tmp")
                    nd = float(evaluate_all(d_tr, score_cols=["score_tmp"], ks=KS).query("chapter_id=='__ALL__'").iloc[0]["ndcg@20"])
                    if nd > best_train_nd:
                        best_train_nd = nd
                        best_cfg = {
                            "alt_col": str(alt_col),
                            "alt_rank_max": int(alt_rank_max),
                            "stagec_rank_max": int(stagec_rank_max),
                            "swaps": int(swaps),
                        }

    idx_te, _ = shortlist_rescue_idx(
        test,
        alt_col=str(best_cfg["alt_col"]),
        alt_rank_max=int(best_cfg["alt_rank_max"]),
        stagec_rank_max=int(best_cfg["stagec_rank_max"]),
        swaps=int(best_cfg["swaps"]),
    )
    d_te = apply_stagec3_tiebreak(test, idx_by_cid=idx_te, out_col="score_tmp")
    hold_nd = float(evaluate_all(d_te, score_cols=["score_tmp"], ks=KS).query(f"chapter_id=='{holdout}'").iloc[0]["ndcg@20"])

    loco_rows.append({
        "holdout_chapter": holdout,
        **best_cfg,
        "train_ndcg@20": float(best_train_nd),
        "holdout_ndcg@20": float(hold_nd),
        "holdout_base_ndcg@20": float(base_nd),
        "holdout_delta_vs_base": float(hold_nd - base_nd),
        "holdout_first_include_rank": first_include_rank(d_te, holdout, "score_tmp"),
    })

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO results:")
display(loco_df)
if not loco_df.empty:
    print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
    print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))

# Persist artifacts
dataset_sha = hashlib.sha1(SCORED_PATH.read_bytes()).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
meta_use = {
    "run_id": run_id_use,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(SCORED_PATH),
    "dataset_sha1_12": dataset_sha,
    "git_head": "unknown",
    "confidence_min": CONFIDENCE_MIN,
    "experiment_tag": "stageC3_shortlist_rescue_intersection_v1",
    "experiment_notes": f"TOPN={TOPN}; ALT_COLS={ALT_COLS}; ALT_RANK_MAXS={ALT_RANK_MAXS}; STAGEC_RANK_MAXS={STAGEC_RANK_MAXS}; SWAPS={SWAPS}",
}

sweep_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_rescue_intersection_sweep_v1.csv"
sweep_df.assign(**meta_use, experiment="stageC3_shortlist_rescue_intersection_sweep_v1").to_csv(sweep_path, index=False)
print("Saved:", sweep_path)

loco_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_rescue_intersection_loco_v1.csv"
loco_df.assign(**meta_use, experiment="stageC3_shortlist_rescue_intersection_loco_v1").to_csv(loco_path, index=False)
print("Saved:", loco_path)

# Append best macro row to runs.csv + save a scored CSV for best config
if not sweep_df.empty:
    best_alt_col = str(best.get("alt_col"))
    best_alt_rank_max = int(best.get("alt_rank_max"))
    best_stagec_rank_max = int(best.get("stagec_rank_max"))
    best_swaps = int(best.get("swaps"))

    idx_best, _ = shortlist_rescue_idx(
        df0,
        alt_col=best_alt_col,
        alt_rank_max=int(best_alt_rank_max),
        stagec_rank_max=int(best_stagec_rank_max),
        swaps=int(best_swaps),
    )
    out_col_best = "score_stageC3_shortlist_rescue_best_v1"
    d_best = apply_stagec3_tiebreak(df0, idx_by_cid=idx_best, out_col=out_col_best)
    best_res = evaluate_all(d_best, score_cols=[out_col_best], ks=KS)
    best_macro = best_res[best_res["chapter_id"] == "__ALL__"].copy().reset_index(drop=True)

    best_meta = dict(meta_use)
    best_meta["experiment_tag"] = "stageC3_shortlist_rescue_intersection_best_v1"
    best_meta["experiment_notes"] = f"TOPN={TOPN}; alt_col={best_alt_col}; alt_rank_max={best_alt_rank_max}; stagec_rank_max={best_stagec_rank_max}; swaps={best_swaps}"

    runs_csv = EXP_DIR / "runs.csv"
    best_macro = best_macro.assign(**best_meta, experiment="stageC3_shortlist_rescue_intersection_best_v1")
    if runs_csv.exists():
        best_macro.to_csv(runs_csv, mode="a", header=False, index=False)
    else:
        best_macro.to_csv(runs_csv, index=False)
    print("Appended best macro row to:", runs_csv)

    scored_path = EXP_DIR / f"{run_id_use}_stageC3_shortlist_rescue_intersection_best_v1_scored.csv"
    keep_cols = [
        c
        for c in ["chapter_id", "merge_key", "final_label", "score_stageC_final", "score_tfidf", "score_stageC1", "llm_score", out_col_best]
        if c in d_best.columns
    ]
    d_best[keep_cols].to_csv(scored_path, index=False)
    print("Saved best scored CSV:", scored_path)

'''
    exec(_CODE, globals())


Skipping st_stageC3_shortlist_rescue_intersection_v1 (RUN_MODE=label_adjudication)


In [45]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_audit_methodology_labels_v1 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r'''
# Audit: why `platform_methodology` is hard (offline; no API calls)
#
# Scientific intent:
# - We repeatedly failed to pull methodology INCLUDEs into the top-50 without hurting macro.
# - This cell prints the key INCLUDE items and how they rank under different signals.
# - It also shows the top-ranked docs so you can sanity-check whether the labels match the rubric.

from pathlib import Path
import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

CID = "platform_methodology"
g = df0[df0["chapter_id"].astype(str) == CID].copy()
print("Rows:", len(g))
print("Label counts:")
print(g["final_label"].fillna("exclude").astype(str).value_counts())

def _rank_col(df_in: pd.DataFrame, col: str) -> pd.Series:
    d = df_in.copy()
    d["_s"] = pd.to_numeric(d[col], errors="coerce").fillna(-1e9)
    d["_mk"] = d["merge_key"].fillna("").astype(str)
    d = d.sort_values(["_s", "_mk"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    return pd.Series(np.arange(1, len(d) + 1, dtype=int), index=d["merge_key"].astype(str))

cols = [c for c in ["score_stageC_final", "score_tfidf", "score_stageC1", "llm_score", "score_stageC3_topn_tiebreak_v1"] if c in g.columns]
rank_maps = {c: _rank_col(g, c) for c in cols}

inc = g[g["final_label"].fillna("exclude").astype(str) == "include"].copy()
if len(inc) == 0:
    print("No INCLUDE rows for this chapter in scored CSV.")
else:
    for c in cols:
        inc[f"rank_{c}"] = inc["merge_key"].astype(str).map(rank_maps[c])
    show_cols = [c for c in ["merge_key", "title"] if c in inc.columns] + [f"rank_{c}" for c in cols] + [c for c in cols if c in inc.columns]
    print("\nINCLUDE items + ranks under different signals:")
    display(inc[show_cols].sort_values([f"rank_score_stageC_final"], ascending=True, kind="mergesort").reset_index(drop=True))

# Show top of baseline final ranking for this chapter
BASE_COL = "score_stageC3_topn_tiebreak_v1" if "score_stageC3_topn_tiebreak_v1" in g.columns else "score_stageC_final"
g2 = g.copy()
g2["_s"] = pd.to_numeric(g2[BASE_COL], errors="coerce").fillna(-1e9)
g2["_mk"] = g2["merge_key"].fillna("").astype(str)
g2 = g2.sort_values(["_s", "_mk"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
g2["rank_final"] = np.arange(1, len(g2) + 1, dtype=int)

keep = [c for c in ["rank_final", "final_label", "llm_score", "score_stageC_final", "score_tfidf", "score_stageC1", "title", "year", "venue", "merge_key"] if c in g2.columns]
print(f"\nTop 30 for {CID} by {BASE_COL}:")
display(g2.head(30)[keep])

print(f"\nBottom 30 for {CID} by {BASE_COL} (to see low-ranked INCLUDEs):")
display(g2.tail(30)[keep])

'''
    exec(_CODE, globals())


Skipping st_audit_methodology_labels_v1 (RUN_MODE=label_adjudication)


In [46]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_stageC_rrf_cite_sweep_v2 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r"""
# Stage C (RRF) + Stage C.3: sweep citation weight inside RRF (offline; no API calls)
#
# Scientific intent:
# - Under the updated methodology labels, the best stable improvement so far is Stage C RRF + Stage C.3.
# - In v1 we fixed w_cite=0.25. This cell sweeps w_cite to see if we can keep the gains
#   while reducing regressions (e.g. empirical case holdout).

from pathlib import Path
from datetime import datetime, timezone
import hashlib

import numpy as np
import pandas as pd

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)

df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "score_stageC_final", "score_tfidf", "llm_score"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

EMB_COL = "score_embed_max" if "score_embed_max" in df0.columns else "score_embed_mean_top3"
if EMB_COL not in df0.columns:
    raise RuntimeError("Missing embedding columns: expected score_embed_max or score_embed_mean_top3")

if "score_cite_norm" not in df0.columns:
    df0 = df0.copy()
    df0["score_cite_norm"] = 0.0

TOPN = 50
RRF_K = 10
W_EMB = 1.0
W_TFIDF = 1.0
W_CITES = [0.0, 0.05, 0.08, 0.12, 0.2, 0.25, 0.35]

def _sorted_rank(g: pd.DataFrame, score_col: str) -> pd.Series:
    gg = g[[score_col, "merge_key"]].copy()
    gg[score_col] = pd.to_numeric(gg[score_col], errors="coerce").fillna(-1e9)
    gg["merge_key"] = gg["merge_key"].fillna("").astype(str)
    gg = gg.sort_values([score_col, "merge_key"], ascending=[False, True], kind="mergesort")
    r = pd.Series(np.arange(1, len(gg) + 1, dtype=int), index=gg.index)
    return r.reindex(g.index)

def add_rrf_score(df_in: pd.DataFrame, *, out_col: str, rrf_k: int, emb_col: str, w_emb: float, w_tfidf: float, w_cite: float) -> pd.DataFrame:
    d = df_in.copy()
    rank_emb = pd.Series(index=d.index, dtype=int)
    rank_tf = pd.Series(index=d.index, dtype=int)
    rank_cite = pd.Series(index=d.index, dtype=int)
    for _, g in d.groupby("chapter_id"):
        rank_emb.loc[g.index] = _sorted_rank(g, emb_col).to_numpy(dtype=int)
        rank_tf.loc[g.index] = _sorted_rank(g, "score_tfidf").to_numpy(dtype=int)
        rank_cite.loc[g.index] = _sorted_rank(g, "score_cite_norm").to_numpy(dtype=int)
    kk = float(rrf_k)
    d[out_col] = (float(w_emb) / (kk + rank_emb.astype(float)) + float(w_tfidf) / (kk + rank_tf.astype(float)) + float(w_cite) / (kk + rank_cite.astype(float)))
    return d

def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)

def shortlist_topn_idx(df_in: pd.DataFrame, base_score_col: str, topn: int = 50) -> dict[str, list[int]]:
    out: dict[str, list[int]] = {}
    for cid, g in df_in.groupby("chapter_id"):
        gg = g.copy()
        gg["_s"] = pd.to_numeric(gg[base_score_col], errors="coerce").fillna(-1e9)
        gg = gg.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort")
        out[str(cid)] = gg.head(int(topn)).index.to_list()
    return out

def apply_stagec3_tiebreak(df_in: pd.DataFrame, *, base_score_col: str, idx_by_cid: dict[str, list[int]], out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    d["_base_n"] = 0.0
    for cid, g in d.groupby("chapter_id"):
        d.loc[g.index, "_base_n"] = minmax_series(g[base_score_col]).to_numpy(dtype=float)
    d[out_col] = d["_base_n"].astype(float)
    for cid, idx in idx_by_cid.items():
        if not idx:
            continue
        llm_n = minmax_series(d.loc[idx, "llm_score"]).to_numpy(dtype=float)
        base = pd.to_numeric(d.loc[idx, base_score_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()
        top = pd.DataFrame({"ix": list(idx), "llm_n": llm_n, "base": base, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "base", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)
    d = d.drop(columns=["_base_n"], errors="ignore")
    return d

def macro_ndcg(df_in: pd.DataFrame, score_col: str) -> float:
    r = evaluate_all(df_in, score_cols=[score_col], ks=KS)
    return float(r[r["chapter_id"] == "__ALL__"].iloc[0]["ndcg@20"])

# Baseline
idx_base = shortlist_topn_idx(df0, base_score_col="score_stageC_final", topn=TOPN)
d_base = apply_stagec3_tiebreak(df0, base_score_col="score_stageC_final", idx_by_cid=idx_base, out_col="score_stageC3_topn_tiebreak_base")
base_nd = macro_ndcg(d_base, "score_stageC3_topn_tiebreak_base")
print("Baseline ndcg@20 (Stage C.3 tiebreak):", base_nd)

rows = []
for w_c in W_CITES:
    d = add_rrf_score(df0, out_col="score_stageC_rrf_v2", rrf_k=RRF_K, emb_col=EMB_COL, w_emb=W_EMB, w_tfidf=W_TFIDF, w_cite=float(w_c))
    idx = shortlist_topn_idx(d, base_score_col="score_stageC_rrf_v2", topn=TOPN)
    d2 = apply_stagec3_tiebreak(d, base_score_col="score_stageC_rrf_v2", idx_by_cid=idx, out_col="score_stageC3_rrf_cite_v2")
    res = evaluate_all(d2, score_cols=["score_stageC3_rrf_cite_v2"], ks=KS)
    m = res[res["chapter_id"] == "__ALL__"].iloc[0].to_dict()
    rows.append({"w_cite": float(w_c), "ndcg@20": float(m.get("ndcg@20")), "p@20": float(m.get("p@20")), "mrr_include": float(m.get("mrr_include")), "auc_include": float(m.get("auc_include")), "delta_vs_base_ndcg@20": float(m.get("ndcg@20")) - float(base_nd)})

sweep_df = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False).reset_index(drop=True)
print("\nRRF w_cite sweep (macro):")
display(sweep_df)
best = sweep_df.iloc[0].to_dict()
print("Best by ndcg@20:", best)

# LOCO
chapters = sorted(df0["chapter_id"].dropna().astype(str).unique().tolist())
loco_rows = []
for holdout in chapters:
    train = df0[df0["chapter_id"].astype(str) != holdout].copy()
    hold = df0[df0["chapter_id"].astype(str) == holdout].copy()
    idx_b = shortlist_topn_idx(hold, base_score_col="score_stageC_final", topn=TOPN)
    b = apply_stagec3_tiebreak(hold, base_score_col="score_stageC_final", idx_by_cid=idx_b, out_col="score_base")
    base_h = macro_ndcg(b, "score_base")
    best_w = None
    best_train = -1e9
    for w_c in W_CITES:
        t = add_rrf_score(train, out_col="score_stageC_rrf_v2", rrf_k=RRF_K, emb_col=EMB_COL, w_emb=W_EMB, w_tfidf=W_TFIDF, w_cite=float(w_c))
        idx_t = shortlist_topn_idx(t, base_score_col="score_stageC_rrf_v2", topn=TOPN)
        t2 = apply_stagec3_tiebreak(t, base_score_col="score_stageC_rrf_v2", idx_by_cid=idx_t, out_col="score_stageC3_rrf_cite_v2")
        nd = macro_ndcg(t2, "score_stageC3_rrf_cite_v2")
        if nd > best_train:
            best_train = nd
            best_w = float(w_c)
    h = add_rrf_score(hold, out_col="score_stageC_rrf_v2", rrf_k=RRF_K, emb_col=EMB_COL, w_emb=W_EMB, w_tfidf=W_TFIDF, w_cite=float(best_w))
    idx_h = shortlist_topn_idx(h, base_score_col="score_stageC_rrf_v2", topn=TOPN)
    h2 = apply_stagec3_tiebreak(h, base_score_col="score_stageC_rrf_v2", idx_by_cid=idx_h, out_col="score_stageC3_rrf_cite_v2")
    hold_nd = macro_ndcg(h2, "score_stageC3_rrf_cite_v2")
    loco_rows.append({"holdout_chapter": holdout, "selected_w_cite": float(best_w), "train_ndcg@20": float(best_train), "holdout_ndcg@20": float(hold_nd), "holdout_base_ndcg@20": float(base_h), "holdout_delta_vs_base": float(hold_nd - base_h)})

loco_df = pd.DataFrame(loco_rows)
print("\nLOCO results:")
display(loco_df)
print("Avg LOCO holdout ndcg@20:", float(loco_df["holdout_ndcg@20"].mean()))
print("Avg LOCO holdout delta vs base:", float(loco_df["holdout_delta_vs_base"].mean()))

# Persist
sha12 = hashlib.sha1(str(SCORED_PATH).encode("utf-8")).hexdigest()[:12]
run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{sha12}"
meta = {"run_id": run_id_use, "created_at_utc": datetime.now(timezone.utc).isoformat(), "dataset_path": str(SCORED_PATH), "dataset_sha1_12": sha12, "experiment_tag": "stageC3_stageC_rrf_cite_sweep_v2", "experiment_notes": f"TOPN={TOPN}; emb_col={EMB_COL}; rrf_k={RRF_K}; w_emb={W_EMB}; w_tfidf={W_TFIDF}; w_cites={W_CITES}"}
out_sweep = EXP_DIR / f"{run_id_use}_stageC3_stageC_rrf_cite_sweep_v2.csv"
sweep_df.assign(**meta, experiment="stageC3_stageC_rrf_cite_sweep_v2").to_csv(out_sweep, index=False)
print("Saved:", out_sweep)
out_loco = EXP_DIR / f"{run_id_use}_stageC3_stageC_rrf_cite_loco_v2.csv"
loco_df.assign(**meta, experiment="stageC3_stageC_rrf_cite_loco_v2").to_csv(out_loco, index=False)
print("Saved:", out_loco)
"""
    exec(_CODE, globals())


Skipping st_stageC3_stageC_rrf_cite_sweep_v2 (RUN_MODE=label_adjudication)


In [47]:
if RUN_MODE != "rerank_analysis":
    print(f"Skipping st_stageC3_stageC_rrf_cite_choice_v3 (RUN_MODE={RUN_MODE})")
else:
    _CODE = r"""
# Decide whether to adopt RRF cite-weight change (offline; no API calls)
#
# Scientific intent:
# - v2 showed a small macro gain at w_cite=0.35, but a regression on platform_empirical_case.
# - This cell evaluates per-chapter tradeoffs across a small candidate set and recommends
#   a conservative choice based on (A) macro ndcg@20 and (B) worst-chapter delta.

from pathlib import Path
import numpy as np
import pandas as pd

# Fallback: if you ran ONLY this cell, earlier helpers like `evaluate_all` may not exist.
if "evaluate_all" not in globals():
    from sklearn.metrics import roc_auc_score
    
    LABEL_TO_BIN_INCLUDE = {"include": 1, "maybe": 0, "exclude": 0}
    LABEL_TO_GRADE = {"include": 2, "maybe": 1, "exclude": 0}
    
    def _to_numeric_score(s: pd.Series) -> np.ndarray:
        return pd.to_numeric(s, errors="coerce").fillna(-1e9).to_numpy(dtype=float)
    
    def _dcg(rels: np.ndarray) -> float:
        rels = np.asarray(rels, dtype=float)
        if rels.size == 0:
            return 0.0
        discounts = 1.0 / np.log2(np.arange(2, rels.size + 2))
        gains = (2.0 ** rels - 1.0)
        return float(np.sum(gains * discounts))
    
    def _ndcg_at_k(rels: np.ndarray, k: int) -> float:
        rels_k = np.asarray(rels[:k], dtype=float)
        ideal = np.sort(rels)[::-1][:k]
        denom = _dcg(ideal)
        if denom <= 0:
            return float("nan")
        return _dcg(rels_k) / denom
    
    def _mrr(rel_bin: np.ndarray) -> float:
        rel_bin = np.asarray(rel_bin, dtype=float)
        hits = np.where(rel_bin > 0)[0]
        if hits.size == 0:
            return 0.0
        return float(1.0 / (hits[0] + 1))
    
    def evaluate_all(df_in: pd.DataFrame, *, score_cols: list[str], ks=(20,)) -> pd.DataFrame:
        rows = []
        for cid, g in df_in.groupby("chapter_id"):
            cid = str(cid)
            labels = g["final_label"].fillna("exclude").astype(str)
            rel_bin = labels.map(LABEL_TO_BIN_INCLUDE).fillna(0).to_numpy(dtype=float)
            rel_grade = labels.map(LABEL_TO_GRADE).fillna(0).to_numpy(dtype=float)
            for col in score_cols:
                s = _to_numeric_score(g[col])
                order = np.argsort(-s, kind="mergesort")
                rb = rel_bin[order]
                rg = rel_grade[order]
                # AUC for include vs not
                try:
                    auc = float(roc_auc_score(rel_bin, s)) if len(np.unique(rel_bin)) > 1 else float("nan")
                except Exception:
                    auc = float("nan")
                out = {
                    "chapter_id": cid,
                    "score_col": col,
                    "n_docs": int(len(g)),
                    "n_include": int((labels == "include").sum()),
                    "n_maybe": int((labels == "maybe").sum()),
                    "auc_include": auc,
                    "mrr_include": _mrr(rb),
                }
                for k in ks:
                    k = int(k)
                    out[f"ndcg@{k}"] = float(_ndcg_at_k(rg, k))
                rows.append(out)
        df_out = pd.DataFrame(rows)
        # macro row
        macros = []
        for col in score_cols:
            sub = df_out[df_out["score_col"] == col].copy()
            if sub.empty:
                continue
            m = {"chapter_id": "__ALL__", "score_col": col}
            for c in ["n_docs", "n_include", "n_maybe"]:
                m[c] = int(sub[c].sum())
            for c in ["auc_include", "mrr_include"] + [f"ndcg@{int(k)}" for k in ks]:
                m[c] = float(sub[c].mean())
            macros.append(m)
        df_out = pd.concat([df_out, pd.DataFrame(macros)], ignore_index=True)
        return df_out

EXP_DIR = Path("eval_dataset/experiments")
cands = sorted(EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not cands:
    raise FileNotFoundError("No *_stageC3_rerank_v1_scored.csv found.")
SCORED_PATH = cands[0]
print("Using scored CSV:", SCORED_PATH)
df0 = pd.read_csv(SCORED_PATH)

required = ["chapter_id", "merge_key", "final_label", "score_stageC_final", "score_tfidf", "llm_score"]
missing = [c for c in required if c not in df0.columns]
if missing:
    raise RuntimeError(f"Scored CSV missing required columns: {missing}")

EMB_COL = "score_embed_max" if "score_embed_max" in df0.columns else "score_embed_mean_top3"
if EMB_COL not in df0.columns:
    raise RuntimeError("Missing embedding columns")

if "score_cite_norm" not in df0.columns:
    df0 = df0.copy()
    df0["score_cite_norm"] = 0.0

TOPN = 50
RRF_K = 10
W_EMB = 1.0
W_TFIDF = 1.0
CAND_W_CITE = [0.2, 0.25, 0.35]

def _sorted_rank(g: pd.DataFrame, score_col: str) -> pd.Series:
    gg = g[[score_col, "merge_key"]].copy()
    gg[score_col] = pd.to_numeric(gg[score_col], errors="coerce").fillna(-1e9)
    gg["merge_key"] = gg["merge_key"].fillna("").astype(str)
    gg = gg.sort_values([score_col, "merge_key"], ascending=[False, True], kind="mergesort")
    r = pd.Series(np.arange(1, len(gg) + 1, dtype=int), index=gg.index)
    return r.reindex(g.index)

def add_rrf_score(df_in: pd.DataFrame, *, out_col: str, w_cite: float) -> pd.DataFrame:
    d = df_in.copy()
    rank_emb = pd.Series(index=d.index, dtype=int)
    rank_tf = pd.Series(index=d.index, dtype=int)
    rank_cite = pd.Series(index=d.index, dtype=int)
    for _, g in d.groupby("chapter_id"):
        rank_emb.loc[g.index] = _sorted_rank(g, EMB_COL).to_numpy(dtype=int)
        rank_tf.loc[g.index] = _sorted_rank(g, "score_tfidf").to_numpy(dtype=int)
        rank_cite.loc[g.index] = _sorted_rank(g, "score_cite_norm").to_numpy(dtype=int)
    kk = float(RRF_K)
    d[out_col] = (float(W_EMB) / (kk + rank_emb.astype(float)) + float(W_TFIDF) / (kk + rank_tf.astype(float)) + float(w_cite) / (kk + rank_cite.astype(float)))
    return d

def minmax_series(x: pd.Series) -> pd.Series:
    v = pd.to_numeric(x, errors="coerce").fillna(0.0)
    mn, mx = float(v.min()), float(v.max())
    return (v - mn) / (mx - mn + 1e-12)

def shortlist_topn_idx(df_in: pd.DataFrame, base_score_col: str, topn: int = 50) -> dict[str, list[int]]:
    out: dict[str, list[int]] = {}
    for cid, g in df_in.groupby("chapter_id"):
        gg = g.copy()
        gg["_s"] = pd.to_numeric(gg[base_score_col], errors="coerce").fillna(-1e9)
        gg = gg.sort_values(["_s", "merge_key"], ascending=[False, True], kind="mergesort")
        out[str(cid)] = gg.head(int(topn)).index.to_list()
    return out

def apply_stagec3_tiebreak(df_in: pd.DataFrame, *, base_score_col: str, idx_by_cid: dict[str, list[int]], out_col: str) -> pd.DataFrame:
    d = df_in.copy()
    d["_base_n"] = 0.0
    for cid, g in d.groupby("chapter_id"):
        d.loc[g.index, "_base_n"] = minmax_series(g[base_score_col]).to_numpy(dtype=float)
    d[out_col] = d["_base_n"].astype(float)
    for cid, idx in idx_by_cid.items():
        if not idx:
            continue
        llm_n = minmax_series(d.loc[idx, "llm_score"]).to_numpy(dtype=float)
        base = pd.to_numeric(d.loc[idx, base_score_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cite = pd.to_numeric(d.loc[idx, "score_cite_norm"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        mk = d.loc[idx, "merge_key"].fillna("").astype(str).to_list()
        top = pd.DataFrame({"ix": list(idx), "llm_n": llm_n, "base": base, "cite": cite, "merge_key": mk})
        top = top.sort_values(["llm_n", "base", "cite", "merge_key"], ascending=[False, False, False, True], kind="mergesort")
        for r, ix in enumerate(top["ix"].tolist()):
            d.loc[ix, out_col] = 2.0 - (r * 1e-6)
    d = d.drop(columns=["_base_n"], errors="ignore")
    return d

def eval_score(df_in: pd.DataFrame, score_col: str) -> pd.DataFrame:
    r = evaluate_all(df_in, score_cols=[score_col], ks=KS)
    return r[["chapter_id", "score_col", "ndcg@20", "p@20", "mrr_include", "auc_include"]].copy()

# Baseline
idx_base = shortlist_topn_idx(df0, base_score_col="score_stageC_final", topn=TOPN)
d_base = apply_stagec3_tiebreak(df0, base_score_col="score_stageC_final", idx_by_cid=idx_base, out_col="score_base")
base_res = eval_score(d_base, "score_base")
base_macro = float(base_res[base_res["chapter_id"]=="__ALL__"].iloc[0]["ndcg@20"])
print("Baseline macro ndcg@20:", base_macro)

rows = []
for w_c in CAND_W_CITE:
    d = add_rrf_score(df0, out_col="score_rrf", w_cite=float(w_c))
    idx = shortlist_topn_idx(d, base_score_col="score_rrf", topn=TOPN)
    d2 = apply_stagec3_tiebreak(d, base_score_col="score_rrf", idx_by_cid=idx, out_col="score_rrf_stageC3")
    res = eval_score(d2, "score_rrf_stageC3")
    res = res.rename(columns={"ndcg@20": "ndcg@20_new"})
    merged = base_res.merge(res[["chapter_id", "ndcg@20_new"]], on="chapter_id", how="left")
    merged["w_cite"] = float(w_c)
    merged["delta_ndcg@20"] = merged["ndcg@20_new"] - merged["ndcg@20"]
    macro_new = float(merged[merged["chapter_id"]=="__ALL__"].iloc[0]["ndcg@20_new"])
    per = merged[merged["chapter_id"]!="__ALL__"].copy()
    worst = float(per["delta_ndcg@20"].min())
    rows.append({"w_cite": float(w_c), "macro_ndcg@20": macro_new, "macro_delta": macro_new-base_macro, "worst_delta_chapter": worst})
    print("\nPer-chapter deltas for w_cite=", w_c)
    display(merged[["chapter_id", "ndcg@20", "ndcg@20_new", "delta_ndcg@20"]])

summary = pd.DataFrame(rows).sort_values(["macro_ndcg@20", "worst_delta_chapter"], ascending=[False, False]).reset_index(drop=True)
print("\nSummary (choose conservative option):")
display(summary)
best_macro = summary.iloc[0].to_dict()
best_worst = summary.sort_values(["worst_delta_chapter", "macro_ndcg@20"], ascending=[False, False]).iloc[0].to_dict()
print("Best by macro ndcg@20:", best_macro)
print("Best by worst-chapter delta:", best_worst)
"""
    exec(_CODE, globals())


Skipping st_stageC3_stageC_rrf_cite_choice_v3 (RUN_MODE=label_adjudication)


## Label quality

If metrics plateau, the next scientific step is to validate label/rubric reliability (especially when one chapter has very few INCLUDEs).


In [48]:
if RUN_MODE != "label_adjudication":
    print(f"Skipping st_label_adjudication_v2 (RUN_MODE={RUN_MODE})")
else:
    # Label adjudication v2 (API calls; cached)
    #
    # Scientific intent:
    # - We found evidence of label noise / rubric ambiguity in LLM labels.
    # - This cell re-labels a subset (or full chapter if MAX_ITEMS>=n_docs) for ONE chapter with explicit label definitions.

    import os
    import json
    import time
    import random
    import asyncio
    import hashlib
    from pathlib import Path
    from datetime import datetime, timezone
    from typing import Literal

    import numpy as np
    import pandas as pd
    import tiktoken
    from pydantic import BaseModel, Field

    from agents import Agent, Runner, ModelSettings

    if "df" not in globals():
        raise RuntimeError("Missing df. Run the notebook top-to-bottom so the dataset is loaded.")

    # -----------------------------
    # Settings (env overrides)
    # -----------------------------
    MODEL = os.getenv("LABEL_ADJUDICATION_MODEL", "gpt-5-mini").strip() or "gpt-5-mini"
    CHAPTER_ID = os.getenv("LABEL_ADJUDICATION_CHAPTER_ID", "platform_methodology").strip() or "platform_methodology"

    MAX_ITEMS = int(os.getenv("LABEL_ADJUDICATION_MAX_ITEMS", "40"))
    CONCURRENCY = int(os.getenv("LABEL_ADJUDICATION_CONCURRENCY", "10"))
    FORCE = bool(int(os.getenv("LABEL_ADJUDICATION_FORCE", "0")))

    PROMPT_VERSION = "v2_methodology_label_defs"

    MAX_RETRIES = 10
    BACKOFF_INITIAL = 1.0
    BACKOFF_MAX = 45.0
    ABSTRACT_MAX_CHARS = 2000

    CACHE_DIR = EXP_DIR / "label_adjudication_cache_v2"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    rubric_path = Path("eval_dataset/eval_rubrics") / f"{CHAPTER_ID}.json"
    if not rubric_path.exists():
        raise FileNotFoundError(f"Missing rubric: {rubric_path}")
    rubric = json.loads(rubric_path.read_text(encoding="utf-8"))

    # -----------------------------
    # Select items
    # -----------------------------
    df_c = df[df["chapter_id"].astype(str) == CHAPTER_ID].copy()
    if df_c.empty:
        raise RuntimeError(f"No rows for chapter_id={CHAPTER_ID}")

    # Prefer Stage-C-ish score if present
    STAGE_SCORE_COL = "score_stageC_final" if "score_stageC_final" in df_c.columns else ("score_hybrid_pool" if "score_hybrid_pool" in df_c.columns else "score_tfidf")
    if STAGE_SCORE_COL not in df_c.columns:
        df_c[STAGE_SCORE_COL] = 0.0
    if "score_tfidf" not in df_c.columns:
        df_c["score_tfidf"] = 0.0

    by_stage = df_c.sort_values([STAGE_SCORE_COL, "merge_key"], ascending=[False, True], kind="mergesort")
    by_tfidf = df_c.sort_values(["score_tfidf", "merge_key"], ascending=[False, True], kind="mergesort")

    facet_txt = df_c.get("facet_best_query")
    if facet_txt is None:
        facet_txt = ""
    facet_txt = facet_txt.fillna("").astype(str)
    facet_mask = (
        facet_txt.str.contains("systematic review methodology", case=False, regex=False)
        | facet_txt.str.contains("inclusion exclusion criteria", case=False, regex=False)
        | facet_txt.str.contains("synthesis methods", case=False, regex=False)
        | facet_txt.str.contains("validation and robustness", case=False, regex=False)
    )

    selected: list[str]

    if MAX_ITEMS >= len(df_c):
        # Full chapter mode
        selected = by_stage["merge_key"].astype(str).tolist()
        print(f"Full mode: selecting all {len(selected)} docs")
    else:
        selected = []

        def _add(mks: list[str]) -> None:
            for mk in mks:
                mk = str(mk or "").strip()
                if mk and (mk not in selected):
                    selected.append(mk)

        # Always include existing INCLUDEs first
        _add(df_c[df_c["final_label"] == "include"]["merge_key"].tolist())

        # Mix: top by stage score, top by tfidf, and some methodology-facet items
        _add(by_stage.head(20)["merge_key"].tolist())
        _add(by_tfidf.head(20)["merge_key"].tolist())

        if facet_mask.any():
            _add(df_c[facet_mask].sort_values([STAGE_SCORE_COL, "merge_key"], ascending=[False, True], kind="mergesort").head(40)["merge_key"].tolist())

        # Add a few MAYBEs for coverage
        _add(df_c[df_c["final_label"] == "maybe"].head(10)["merge_key"].tolist())

        selected = selected[:MAX_ITEMS]

    sel_df = df_c[df_c["merge_key"].astype(str).isin(selected)].copy()
    sel_df["_sel_order"] = sel_df["merge_key"].astype(str).apply(lambda mk: selected.index(mk) if mk in selected else 10**9)
    sel_df = sel_df.sort_values(["_sel_order"], kind="mergesort").drop(columns=["_sel_order"])

    print(f"Selected {len(sel_df)} docs for adjudication (MAX_ITEMS={MAX_ITEMS})")
    print(f"Stage score column: {STAGE_SCORE_COL}")
    print("Selection label mix (existing):")
    display(sel_df["final_label"].value_counts(dropna=False).to_frame("count"))

    # -----------------------------
    # Cost tracking (estimate via tiktoken if API usage is missing)
    # -----------------------------
    MODEL_PRICES_USD_PER_1M = {
        "gpt-5-nano": {"input": 0.05, "cached": 0.005, "output": 0.40},
        "gpt-5-mini": {"input": 0.25, "cached": 0.025, "output": 2.00},
    }

    def _price_for_model(model: str) -> dict:
        return MODEL_PRICES_USD_PER_1M.get(model, {"input": 0.0, "cached": 0.0, "output": 0.0})

    enc = tiktoken.encoding_for_model(MODEL)

    def _estimate_meta(prompt: str, output_obj: dict, model: str) -> dict:
        prices = _price_for_model(model)
        inp = int(len(enc.encode(prompt)))
        out_txt = json.dumps(output_obj, ensure_ascii=False)
        out = int(len(enc.encode(out_txt)))
        cost = (inp / 1_000_000) * prices["input"] + (out / 1_000_000) * prices["output"]
        return {
            "requests": 1,
            "input_tokens": inp,
            "cached_input_tokens": 0,
            "output_tokens": out,
            "cost_usd": float(cost),
        }

    def _truncate(s: str, max_chars: int) -> str:
        s = (s or "").strip()
        return s if len(s) <= max_chars else s[:max_chars].rstrip() + "…"

    LABEL_DEFS_DEFAULT = {
        "include": "Directly fits the chapter scope and satisfies multiple must_cover items; a strong core source for the chapter.",
        "maybe": "Partially relevant (covers a facet or useful background) but misses key rubric requirements, is too narrow, or lacks evidence/data.",
        "exclude": "Out of scope, wrong domain, or mainly violates must_avoid; not useful for the chapter.",
    }

    # Chapter-specific clarifications (optional but improves label consistency)
    LABEL_DEFS_BY_CHAPTER = {
        "platform_methodology": {
            "include": "Directly supports methodology: review protocol/search strategy/inclusion criteria OR framework development & operationalization/reproducibility.",
            "maybe": "Useful background for the methodology chapter (e.g., core platform-mechanism papers to justify search terms/dimensions), but not itself a methodology/process source.",
            "exclude": "Unrelated, wrong domain, or mainly violates must_avoid; not useful for the chapter.",
        },
        "platform_theory": {
            "include": "Foundational platform-economics theory on two-/multi-sided markets, network externalities, critical mass, multi-homing/switching costs, governance, monetization/price structure, and competition dynamics.",
            "maybe": "Related background (e.g., antitrust/digital strategy) but not focused on the required core mechanisms or too applied/narrow.",
            "exclude": "Other market/organization theories outside scope or unrelated domains.",
        },
        "platform_empirical_case": {
            "include": "Empirical/evidence-based source about a specific platform or platform market (e.g., Airbnb/Uber/eBay/app store/Grab/etc.) that provides usable evidence or metrics for applying the framework (network effects, multi-homing/switching costs, governance changes, fees/pricing, competition/market share, quality indicators). One dimension can be enough if it is clearly platform-specific and evidence-based.",
            "maybe": "Platform-related but not directly usable as empirical case evidence: theory/models, conceptual frameworks, or methods/metrics papers without a concrete platform case; also weak/unclear platform connection.",
            "exclude": "Not about platforms/two-sided markets (wrong domain) or mainly technical/system papers; not useful for the case chapter.",
        },
    }

    LABEL_DEFS = LABEL_DEFS_BY_CHAPTER.get(CHAPTER_ID, LABEL_DEFS_DEFAULT)

    # -----------------------------
    # LLM adjudication
    # -----------------------------
    class Adjudication(BaseModel):
        label: Literal["include", "maybe", "exclude"] = Field(...)
        confidence: int = Field(..., ge=0, le=100)
        rationale: str = Field(..., description="<=20 words")

    adjudicator = Agent(
        name="label_adjudicator",
        instructions=(
            """You label whether a paper is useful for the chapter rubric.
Follow the LABEL DEFINITIONS and CHAPTER RUBRIC in the prompt.
Be conservative with include.
If unsure between maybe and exclude, choose maybe."""
        ),
        model=MODEL,
        output_type=Adjudication,
        model_settings=ModelSettings(),
    )

    def _prompt(rub: dict, title: str, abstract: str) -> str:
        keep = {k: rub.get(k) for k in ["scope_statement", "must_cover", "should_cover", "must_avoid", "scoring_guidance"]}
        t = str(title or "").strip()
        a = _truncate(str(abstract or ""), ABSTRACT_MAX_CHARS)
        return (
            "LABEL DEFINITIONS:\n"
            + json.dumps(LABEL_DEFS, ensure_ascii=False)
            + "\n\nCHAPTER RUBRIC (JSON):\n"
            + json.dumps(keep, ensure_ascii=False)
            + "\n\nPAPER:\n"
            + f"Title: {t}\n"
            + f"Abstract: {a}\n\n"
            + "Return JSON only."
        )

    rubric_sig = hashlib.sha1(json.dumps(rubric, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:12]
    sig = hashlib.sha1(f"{CHAPTER_ID}|{MODEL}|{rubric_sig}|{PROMPT_VERSION}".encode("utf-8")).hexdigest()[:12]

    sem = asyncio.Semaphore(CONCURRENCY)

    def _cache_path(merge_key: str) -> Path:
        mk = str(merge_key or "").strip()
        mk_hash = hashlib.sha1(mk.encode("utf-8")).hexdigest()[:12]
        slug = "".join([c if (c.isalnum() or c in "._-") else "_" for c in mk])
        slug = slug.strip("._-") or "mk"
        slug = slug[:80] if len(slug) > 80 else slug
        return CACHE_DIR / f"{sig}_{mk_hash}_{slug}.json"

    async def _run_one(row: pd.Series) -> dict:
        mk = str(row.get("merge_key") or "").strip()
        title = str(row.get("title") or "")
        abstract = str(row.get("abstract") or "")

        prompt = _prompt(rubric, title=title, abstract=abstract)
        cache_path = _cache_path(mk)

        if cache_path.exists() and not FORCE:
            payload_disk = json.loads(cache_path.read_text(encoding="utf-8"))
            out_obj = {
                "label": payload_disk.get("label"),
                "confidence": payload_disk.get("confidence"),
                "rationale": payload_disk.get("rationale"),
            }
            payload_disk["_meta"] = {
                **_estimate_meta(prompt, out_obj, model=MODEL),
                "model": MODEL,
                "rubric_sig": rubric_sig,
                "prompt_version": PROMPT_VERSION,
            }
            payload_disk["_cached"] = True
            return payload_disk

        last_err = None
        for attempt in range(MAX_RETRIES):
            try:
                async with sem:
                    res = await Runner.run(adjudicator, prompt)
                out_obj = {
                    "label": str(res.final_output.label),
                    "confidence": int(res.final_output.confidence),
                    "rationale": str(res.final_output.rationale),
                }
                meta_est = _estimate_meta(prompt, out_obj, model=MODEL)
                out = {
                    "merge_key": mk,
                    **out_obj,
                    "_meta": {**meta_est, "model": MODEL, "rubric_sig": rubric_sig, "prompt_version": PROMPT_VERSION},
                    "_cached": False,
                }
                cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
                return out
            except Exception as e:
                last_err = e
                backoff = min(BACKOFF_MAX, BACKOFF_INITIAL * (2 ** attempt))
                backoff = backoff * (0.75 + 0.5 * random.random())
                await asyncio.sleep(backoff)

        raise RuntimeError(f"Adjudication failed for merge_key={mk}: {last_err}")

    async def _run_all() -> tuple[pd.DataFrame, dict, dict]:
        t0 = time.time()
        totals_run = {"seconds": 0.0, "requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0, "cached_files": 0}
        totals_est = {"seconds": 0.0, "requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0}

        mks: list[str] = []
        tasks = []
        for _, r in sel_df.iterrows():
            mk = str(r.get("merge_key") or "").strip()
            mks.append(mk)
            tasks.append(_run_one(r))

        outs = await asyncio.gather(*tasks, return_exceptions=True)

        rows: list[dict] = []
        errors = 0
        for mk, o in zip(mks, outs):
            if isinstance(o, Exception):
                errors += 1
                rows.append({
                    "merge_key": mk,
                    "label": None,
                    "confidence": None,
                    "rationale": "",
                    "_meta": {"requests": 0, "input_tokens": 0, "cached_input_tokens": 0, "output_tokens": 0, "cost_usd": 0.0},
                    "_cached": False,
                    "_error": repr(o),
                })
                continue

            rows.append(o)
            meta = o.get("_meta") or {}
            # estimated totals (include cached)
            totals_est["requests"] += int(meta.get("requests", 1) or 1)
            totals_est["input_tokens"] += int(meta.get("input_tokens", 0) or 0)
            totals_est["cached_input_tokens"] += int(meta.get("cached_input_tokens", 0) or 0)
            totals_est["output_tokens"] += int(meta.get("output_tokens", 0) or 0)
            totals_est["cost_usd"] += float(meta.get("cost_usd", 0.0) or 0.0)

            if bool(o.get("_cached", False)):
                totals_run["cached_files"] += 1
                continue

            # this-run totals (exclude cached)
            totals_run["requests"] += int(meta.get("requests", 1) or 1)
            totals_run["input_tokens"] += int(meta.get("input_tokens", 0) or 0)
            totals_run["cached_input_tokens"] += int(meta.get("cached_input_tokens", 0) or 0)
            totals_run["output_tokens"] += int(meta.get("output_tokens", 0) or 0)
            totals_run["cost_usd"] += float(meta.get("cost_usd", 0.0) or 0.0)

        if errors:
            print(f"Warning: {errors} adjudication items failed (kept as _error rows).")

        secs = float(time.time() - t0)
        totals_run["seconds"] = secs
        totals_est["seconds"] = secs
        return pd.DataFrame(rows), totals_run, totals_est

    adj_df, totals_run, totals_est = await _run_all()
    print("Adjudication totals (this run):", totals_run)
    print("Adjudication totals (estimated incl cached):", totals_est)

    out = sel_df.merge(adj_df, on="merge_key", how="left")
    out = out.rename(columns={"final_label": "old_label", "label": "new_label", "confidence": "new_confidence", "rationale": "new_rationale"})

    print()
    print("Label comparison counts:")
    display(out.groupby(["old_label", "new_label"]).size().unstack(fill_value=0))

    changed = out[out["old_label"] != out["new_label"]].copy()
    print()
    print(f"Changed labels: {len(changed)}")
    cols = ["merge_key", "title", "old_label", "new_label", "new_confidence", "new_rationale", STAGE_SCORE_COL, "score_tfidf", "facet_best_query"]
    cols = [c for c in cols if c in changed.columns]
    display(changed[cols].head(25))

    low_conf = out[pd.to_numeric(out["new_confidence"], errors="coerce").fillna(0) < 70].copy()
    print()
    print(f"Low-confidence (<70) items: {len(low_conf)}")
    display(low_conf[["merge_key", "title", "old_label", "new_label", "new_confidence", "new_rationale"]].head(25))

    # -----------------------------
    # Persist artifacts
    # -----------------------------
    dataset_sha = hashlib.sha1(DATASET_PATH.read_bytes()).hexdigest()[:12]
    run_id_use = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + f"_{dataset_sha}"
    meta_use = {
        "run_id": run_id_use,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dataset_path": str(DATASET_PATH),
        "dataset_sha1_12": dataset_sha,
        "experiment_tag": "label_adjudication_v2",
        "experiment_notes": f"chapter_id={CHAPTER_ID}; model={MODEL}; max_items={MAX_ITEMS}; prompt={PROMPT_VERSION}; stage_score_col={STAGE_SCORE_COL}",
    }

    out_path = EXP_DIR / f"{run_id_use}_label_adjudication_{CHAPTER_ID}_v2.csv"
    out.assign(**meta_use, adjudication_model=MODEL, prompt_version=PROMPT_VERSION, stage_score_col=STAGE_SCORE_COL).to_csv(out_path, index=False)
    print("Saved:", out_path)

    cost_path = EXP_DIR / f"{run_id_use}_label_adjudication_{CHAPTER_ID}_v2_cost.csv"
    pd.DataFrame([{**totals_run, **{f"est_{k}": v for k, v in totals_est.items()}}]).assign(**meta_use, experiment="label_adjudication_v2_cost", adjudication_model=MODEL).to_csv(cost_path, index=False)
    print("Saved:", cost_path)


Selected 26 docs for adjudication (MAX_ITEMS=40)
Stage score column: score_hybrid_pool
Selection label mix (existing):


,count
final_label,
exclude,13
maybe,11
include,2


Adjudication totals (this run): {'seconds': 38.0870304107666, 'requests': 26, 'input_tokens': 14536, 'cached_input_tokens': 0, 'output_tokens': 987, 'cost_usd': 0.005608, 'cached_files': 0}
Adjudication totals (estimated incl cached): {'seconds': 38.0870304107666, 'requests': 26, 'input_tokens': 14536, 'cached_input_tokens': 0, 'output_tokens': 987, 'cost_usd': 0.005608}

Label comparison counts:


new_label,exclude,maybe
old_label,,
exclude,4,9
include,0,2
maybe,2,9



Changed labels: 13


,merge_key,title,old_label,new_label,new_confidence,new_rationale,score_hybrid_pool,score_tfidf,facet_best_query
0,doi:10.1287/mnsc.2023.4675,Dog Eat Dog: Balancing Network Effects and Dif...,include,maybe,75,Empirical merger study of pet‑sitting platform...,0.817779,0.340256,case synthesis dominant network effects strate...
1,doi:10.1109/tem.2025.3534433,Subsidy Wars for Market Dominance in Emerging ...,include,maybe,90,Theoretical subsidy-war model; lacks empirical...,0.544730,0.158289,case synthesis dominant network effects strate...
4,doi:https://doi.org/10.2139/ssrn.917785,Coordination and Lock-In: Competition with Swi...,maybe,exclude,92,Title indicates theoretical work; no evidence ...,0.761148,0.324719,multi‑homing prevalence and switching cost evi...
6,doi:10.65150/ep-jmrr/v1e5/2025-03,Network Effects Simulation Model for Multi-Sid...,exclude,maybe,80,Simulation/theoretical focus; lacks clear empi...,0.734518,0.228874,case synthesis dominant network effects strate...
7,doi:10.47134/jobm.v2i3.33,Network Effects and Trust in Malaysia’s Platfo...,exclude,maybe,75,Empirical Grab case study but likely omits pri...,0.714991,0.218792,network effects linked to user and provider gr...
11,doi:10.1177/0149206311429861,Market Entry in the Presence of Network Effects,exclude,maybe,40,Title implies theoretical analysis; no evidenc...,0.692992,0.244992,network effects linked to user and provider gr...
12,doi:https://doi.org/10.2202/1446-9022.1256,Failure to Launch: Critical Mass in Platform B...,exclude,maybe,88,Theoretical essay on critical mass; relevant b...,0.680866,0.200129,network effects linked to user and provider gr...
13,doi:10.1002/9781118783764.wbieme0056,Multistep Flow of Communication: Network Effects,exclude,maybe,80,Appears theoretical on network effects without...,0.676139,0.234314,network effects linked to user and provider gr...
16,ty:managerial delegation nder network effects...,Managerial delegation under network effects,exclude,maybe,85,Theoretical study of delegation and network ef...,0.644954,0.239974,case synthesis dominant network effects strate...
19,doi:10.1504/ijtm.2004.003956,"Network effects, standardisation and competiti...",maybe,exclude,85,Theoretical focus on standards and strategy; n...,0.622229,0.157412,case synthesis dominant network effects strate...



Low-confidence (<70) items: 4


,merge_key,title,old_label,new_label,new_confidence,new_rationale
10,ty:strategic implications of network effects ...,Strategic implications of network effects : an...,maybe,maybe,30,Title implies empirical network-effects study ...
11,doi:10.1177/0149206311429861,Market Entry in the Presence of Network Effects,exclude,maybe,40,Title implies theoretical analysis; no evidenc...
14,doi:10.1007/s12525-025-00817-4,What role do data network effects play for mul...,maybe,maybe,60,Empirical focus but unclear if applies to a sp...
22,doi:https://doi.org/10.1016/j.tranpol.2019.01.004,Exploring network effects of point-to-point ne...,exclude,maybe,55,Empirical airline network analysis likely addr...


Saved: eval_dataset\experiments\20260202_103909_b62a29d1b69b_label_adjudication_platform_empirical_case_v2.csv
Saved: eval_dataset\experiments\20260202_103909_b62a29d1b69b_label_adjudication_platform_empirical_case_v2_cost.csv


In [49]:
import os

if RUN_MODE != "label_adjudication" or int(os.getenv("LABEL_ADJUDICATION_APPLY", "1")) != 1:
    print(f"Skipping st_label_adjudication_apply_v2 (RUN_MODE={RUN_MODE}; LABEL_ADJUDICATION_APPLY={os.getenv('LABEL_ADJUDICATION_APPLY','1')})")
else:
    # Apply the latest adjudication CSV to create a new dataset version.
    #
    # Scientific intent:
    # - Keep original dataset immutable.
    # - Create a new dataset folder under `eval_dataset/datasets/` with updated labels.
    # - Also write a *labels-synced* copy of the Stage C.3 scored CSV so rerank_analysis cells pick it up.

    import json
    import shutil
    import hashlib
    from pathlib import Path
    from datetime import datetime, timezone

    import numpy as np
    import pandas as pd

    CHAPTER_ID = os.getenv("LABEL_ADJUDICATION_CHAPTER_ID", "platform_methodology").strip() or "platform_methodology"
    CONF_MIN = int(os.getenv("LABEL_ADJUDICATION_APPLY_CONFIDENCE_MIN", "0"))

    # Find latest adjudication output
    cand = sorted(
        EXP_DIR.glob(f"*_label_adjudication_{CHAPTER_ID}_v2.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not cand:
        raise FileNotFoundError(f"No adjudication CSV found for {CHAPTER_ID}. Run st_label_adjudication_v2 first.")

    adj_path = cand[0]
    print("Using adjudication CSV:", adj_path)
    adj = pd.read_csv(adj_path)

    # Normalize column names (older files might have label/confidence)
    if "new_label" not in adj.columns and "label" in adj.columns:
        adj = adj.rename(columns={"label": "new_label"})
    if "new_confidence" not in adj.columns and "confidence" in adj.columns:
        adj = adj.rename(columns={"confidence": "new_confidence"})

    required = ["merge_key", "new_label", "new_confidence"]
    missing = [c for c in required if c not in adj.columns]
    if missing:
        raise RuntimeError(f"Adjudication CSV missing required columns: {missing}")

    # Filter failed rows if present
    if "_error" in adj.columns:
        ok = adj[adj["_error"].isna() | (adj["_error"].astype(str).str.len() == 0)].copy()
    else:
        ok = adj.copy()

    ok = ok.dropna(subset=["merge_key", "new_label"]).copy()

    if CONF_MIN > 0:
        ok = ok[pd.to_numeric(ok["new_confidence"], errors="coerce").fillna(0) >= CONF_MIN].copy()
        print(f"Applying only rows with new_confidence >= {CONF_MIN}: {len(ok)}/{len(adj)}")

    # Load base dataset
    base_path = Path(DATASET_PATH)
    base_df = pd.read_csv(base_path)

    if "final_label" not in base_df.columns or "final_confidence" not in base_df.columns:
        raise RuntimeError("Base dataset missing final_label/final_confidence")

    mask = base_df["chapter_id"].astype(str) == CHAPTER_ID
    if mask.sum() == 0:
        raise RuntimeError(f"Base dataset has no rows for chapter_id={CHAPTER_ID}")

    before = base_df.loc[mask, "final_label"].value_counts(dropna=False).to_dict()

    mk_to_label = dict(zip(ok["merge_key"].astype(str), ok["new_label"].astype(str)))
    mk_to_conf = dict(
        zip(
            ok["merge_key"].astype(str),
            pd.to_numeric(ok["new_confidence"], errors="coerce").astype(float),
        )
    )

    new_df = base_df.copy()

    new_labels = new_df.loc[mask, "merge_key"].astype(str).map(mk_to_label)
    new_df.loc[mask, "final_label"] = new_labels.fillna(new_df.loc[mask, "final_label"])

    conf_map = new_df.loc[mask, "merge_key"].astype(str).map(mk_to_conf)
    conf_ok = conf_map.notna()
    new_df.loc[new_df.loc[mask].index[conf_ok], "final_confidence"] = conf_map[conf_ok].astype(float).values

    after = new_df.loc[mask, "final_label"].value_counts(dropna=False).to_dict()

    print("Before label counts (chapter):", before)
    print("After  label counts (chapter):", after)

    # Create dataset tag + write outputs
    base_tag = globals().get("PINNED_DATASET_TAG")
    if not base_tag:
        base_tag = base_path.parent.name

    csv_bytes = new_df.to_csv(index=False).encode("utf-8")
    sha12 = hashlib.sha1(csv_bytes).hexdigest()[:12]
    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

    dataset_tag = f"{base_tag}__{CHAPTER_ID}_labels_v2_{sha12}"  # deterministic by content
    out_dir = Path("eval_dataset/datasets") / dataset_tag
    if out_dir.exists():
        dataset_tag = f"{base_tag}__{CHAPTER_ID}_labels_v2_{sha12}_{ts}"
        out_dir = Path("eval_dataset/datasets") / dataset_tag

    out_dir.mkdir(parents=True, exist_ok=False)

    (out_dir / "labeled_dataset.csv").write_bytes(csv_bytes)

    # Copy adjudication artifacts for traceability
    patch_dir = out_dir / "label_patches"
    patch_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(adj_path, patch_dir / adj_path.name)

    cost_cand = sorted(
        EXP_DIR.glob(f"*_label_adjudication_{CHAPTER_ID}_v2_cost.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if cost_cand:
        shutil.copy2(cost_cand[0], patch_dir / cost_cand[0].name)

    # Write manifest
    manifest = {
        "dataset_tag": dataset_tag,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "base_dataset_path": str(base_path),
        "base_dataset_tag": str(base_tag),
        "chapter_relabel": {
            "chapter_id": CHAPTER_ID,
            "adjudication_csv": str(adj_path),
            "confidence_min_applied": int(CONF_MIN),
        },
        "label_counts": {
            "chapter_before": before,
            "chapter_after": after,
            "all": new_df.groupby(["chapter_id", "final_label"]).size().unstack(fill_value=0).to_dict(),
        },
        "dataset_sha1_12": sha12,
    }
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    print("\\nCreated new dataset tag:", dataset_tag)
    print("Dataset path:", out_dir / "labeled_dataset.csv")
    print("Manifest:", out_dir / "manifest.json")

    # -----------------------------
    # Also write a label-synced scored CSV (for rerank_analysis)
    # -----------------------------
    scored_cands = sorted(
        EXP_DIR.glob("*_stageC3_rerank_v1_scored.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not scored_cands:
        print("\\nNo *_stageC3_rerank_v1_scored.csv found; skipping scored-CSV label sync.")
    else:
        scored_path = scored_cands[0]
        print("\\nSyncing labels into scored CSV:", scored_path)
        scored = pd.read_csv(scored_path)

        lab = new_df[["chapter_id", "merge_key", "final_label", "final_confidence"]].copy()
        lab = lab.rename(columns={"final_label": "final_label_new", "final_confidence": "final_confidence_new"})

        merged = scored.merge(lab, on=["chapter_id", "merge_key"], how="left")
        if "final_label" in merged.columns:
            merged["final_label"] = merged["final_label_new"].fillna(merged["final_label"])
        else:
            merged["final_label"] = merged["final_label_new"]

        if "final_confidence" in merged.columns:
            merged["final_confidence"] = merged["final_confidence_new"].fillna(merged["final_confidence"])
        else:
            merged["final_confidence"] = merged["final_confidence_new"]

        merged = merged.drop(columns=["final_label_new", "final_confidence_new"], errors="ignore")
        merged["labels_dataset_tag"] = dataset_tag
        merged["labels_dataset_sha1_12"] = sha12

        scored_out = EXP_DIR / f"{ts}_{sha12}_stageC3_rerank_v1_scored.csv"
        merged.to_csv(scored_out, index=False)
        print("Saved label-synced scored CSV:", scored_out)

    print("\\nNext steps:")
    print("1) Set PINNED_DATASET_TAG=\"latest\" (or the dataset_tag above) in the first cell.")
    print("2) Switch RUN_MODE=\"rerank_analysis\" and run all to recompute Stage C/C.3/Stage D analyses under the updated labels.")


Skipping st_label_adjudication_apply_v2 (RUN_MODE=label_adjudication; LABEL_ADJUDICATION_APPLY=0)
